### Construction of Graphs and Matrices from Excel Data

In [ ]:
## APELDOORN SATELLITE GRAPH + MATRICES 

import pandas as pd
import networkx as nx
import numpy as np
import pickle
import os


# Global variables
FILE_PATH = "./Datasets/Cl_mainRoads.xlsx"  # Path to the Apeldoorn main roads dataset
SHEET_NAME = "Cl_mainRoads_Proccessed"
OUTPUT_DIR = "./precomputed_matrices"  ## MAKE SURE THE FOLDER WITH THIS NAME IS LOCATED IN THE DIRECTORY OF THIS SCRIPT

FRACTION = 1  # Use a fraction of the data for testing (set to 1.0 for full dataset)
PROGRESS_STEP = 100  # Print progress every 100 rows

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Function to compute haversine distance
def haversine(lat1, lon1, lat2, lon2):
    R = 6371e3  # Earth's radius in meters
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    delta_phi = phi2 - phi1
    delta_lambda = np.radians(lon2 - lon1)
    a = np.sin(delta_phi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

# Compute haversine distance matrix
def compute_haversine_distance_matrix(edge_attributes):
    print("Computing haversine distance matrix...")
    centroids = [
        (
            (attr['Start_Node'][0] + attr['End_Node'][0]) / 2,
            (attr['Start_Node'][1] + attr['End_Node'][1]) / 2
        )
        for attr in edge_attributes
    ]

    n = len(centroids)
    dist_matrix = np.zeros((n, n))
    for i, coord1 in enumerate(centroids):
        for j in range(i + 1, n):  # Compute only upper triangle
            coord2 = centroids[j]
            dist_matrix[i, j] = haversine(coord1[0], coord1[1], coord2[0], coord2[1])
        if i % PROGRESS_STEP == 0:
            print(f"Processed {i}/{n} rows for distance matrix...")
    dist_matrix += dist_matrix.T  # Mirror upper triangle to lower triangle
    print("Haversine distance matrix computation completed.")
    return dist_matrix

# Compute all-pairs shortest paths
def compute_edge_shortest_path_matrix(edge_attributes, graph):
    print("Computing all-pairs shortest paths for the graph nodes...")

    n = len(edge_attributes)
    edge_path_matrix = np.full((n, n), np.inf)  # Initialize with infinity

    for i, edge1 in enumerate(edge_attributes):
        for j, edge2 in enumerate(edge_attributes):
            if i == j:
                edge_path_matrix[i, j] = 0  # Distance to itself is 0
            else:
                try:
                    # Compute shortest path length between edges
                    path_length = nx.shortest_path_length(
                        graph,
                        source=edge1['Start_Node'],
                        target=edge2['End_Node'],
                        weight=None
                    )
                    edge_path_matrix[i, j] = path_length
                except nx.NetworkXNoPath:
                    edge_path_matrix[i, j] = np.inf

        if i % PROGRESS_STEP == 0:
            print(f"Processed {i}/{n} edges for shortest path matrix...")

    print("Edge-to-edge shortest path matrix computation completed.")
    return edge_path_matrix

# Save matrices and metadata
def save_matrices(distance_matrix, shortest_path_matrix, edge_attributes):
    print("Saving matrices...")
    np.save(f"{OUTPUT_DIR}/distance_matrix.npy", distance_matrix)
    np.save(f"{OUTPUT_DIR}/shortest_path_matrix.npy", shortest_path_matrix)
    with open(f"{OUTPUT_DIR}/edge_attributes.pkl", "wb") as f:
        pickle.dump(edge_attributes, f)
    print("Matrices and attributes saved successfully.")

# Extract largest connected component
def extract_largest_connected_component(G):
    print("Extracting the largest connected component...")
    largest_component_nodes = max(nx.connected_components(G), key=len)
    largest_component_subgraph = G.subgraph(largest_component_nodes).copy()
    print(f"Largest connected component has {largest_component_subgraph.number_of_nodes()} nodes and {largest_component_subgraph.number_of_edges()} edges.")
    return largest_component_subgraph

# Main function
def main():
    print("Loading data...")
    data = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
    data = data.sample(frac=FRACTION, random_state=42).reset_index(drop=True)

    print("Constructing graph...")
    G = nx.Graph()
    edge_attributes = []

    # Use a set to track unique edges
    added_edges = set()

    for _, row in data.iterrows():
        start_node = (row["Start_Lat"], row["Start_Lon"])
        end_node = (row["End_Lat"], row["End_Lon"])

        # Skip self-loops
        if start_node == end_node:
            continue

        # Check for duplicate edges
        edge = (start_node, end_node)
        reverse_edge = (end_node, start_node)
        if edge in added_edges or reverse_edge in added_edges:
            continue

        # Add the edge to the graph
        attributes = {
            'Start_Node': start_node,
            'End_Node': end_node,
            'Local_ID': row['Local_ID'],
            'Bld_densit': row['Bld_densit'],
            'Veg_densit': row['Veg_densit'],
            'HW_Ratio': row['HW_Ratio'],
            'Width': row['Width'],
            'Pred_Facad': row['Pred_Facad'],
            'Pred_LandU': row['Pred_LandU'],
            'Mean_Popul': row['Mean_Popul'],
            'Mean_Bld_E': row['Mean_Bld_E'],
            'Max_Bld_He': row['Max_Bld_He'],
            'Delta_T': row['Delta_T'],
            'Water_dens': row['Water_dens'],
            'Bike_pct_a': row['Bike_pct_a'],
            'Vehicle_pc': row['Vehicle_pc']
        }
        G.add_edge(start_node, end_node, **attributes)
        edge_attributes.append(attributes)

        # Mark this edge as added
        added_edges.add(edge)

    print(f"Graph created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

    # Extract largest connected component
    G = extract_largest_connected_component(G)

    # Filter edge attributes for the largest connected component
    largest_component_edges = set(G.edges())
    filtered_edge_attributes = [
        attr for attr in edge_attributes
        if (attr['Start_Node'], attr['End_Node']) in largest_component_edges or
           (attr['End_Node'], attr['Start_Node']) in largest_component_edges
    ]

    # Debugging
    print(f"Number of edge attributes before filtering: {len(edge_attributes)}")
    print(f"Number of edge attributes after filtering: {len(filtered_edge_attributes)}")
    print(f"Number of edges in the graph's largest component: {len(largest_component_edges)}")

    # Compute distance and shortest path matrices
    distance_matrix = compute_haversine_distance_matrix(filtered_edge_attributes)
    shortest_path_matrix = compute_edge_shortest_path_matrix(filtered_edge_attributes, G)

    # Save matrices and attributes
    save_matrices(distance_matrix, shortest_path_matrix, filtered_edge_attributes)

    print("Script completed successfully.")

if __name__ == "__main__":
    main()

In [ ]:
## ROTTERDAM GRAPH + MATRICES
## NOTE: THIS VERSION DOES NOT INCLUDE EXTRACTION OF THE MAIN COMPONENT
## THERE IS ALSO NO NODE-BASED MATRIX

import pandas as pd
import networkx as nx
import numpy as np
import pickle
import os

# Global variables
FILE_PATH = "./Datasets/Rotterdam_coordinates.xlsx"
SHEET_NAME = "Sheet1"
OUTPUT_DIR = "./precomputed_matrices"
FRACTION = 1  # Use a fraction of the data for testing (set to 1.0 for full dataset)
PROGRESS_STEP = 100  # Print progress every 100 rows

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Function to compute haversine distance
def haversine(lat1, lon1, lat2, lon2):
    R = 6371e3  # Earth's radius in meters
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    delta_phi = phi2 - phi1
    delta_lambda = np.radians(lon2 - lon1)
    a = np.sin(delta_phi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

# Compute haversine distance matrix
def compute_haversine_distance_matrix(edge_attributes):
    print("Computing haversine distance matrix...")
    centroids = [
        (
            (attr['Start_Node'][0] + attr['End_Node'][0]) / 2,
            (attr['Start_Node'][1] + attr['End_Node'][1]) / 2
        )
        for attr in edge_attributes
    ]

    n = len(centroids)
    dist_matrix = np.zeros((n, n))
    for i, coord1 in enumerate(centroids):
        for j in range(i + 1, n):  # Compute only upper triangle
            coord2 = centroids[j]
            dist_matrix[i, j] = haversine(coord1[0], coord1[1], coord2[0], coord2[1])
        if i % PROGRESS_STEP == 0:
            print(f"Processed {i}/{n} rows for distance matrix...")
    dist_matrix += dist_matrix.T  # Mirror upper triangle to lower triangle
    print("Haversine distance matrix computation completed.")
    return dist_matrix

# Save matrices and metadata
def save_matrices(distance_matrix, edge_attributes):
    print("Saving matrices...")
    np.save(f"{OUTPUT_DIR}/distance_matrix_Rotterdam.npy", distance_matrix)
    with open(f"{OUTPUT_DIR}/edge_attributes_Rotterdam.pkl", "wb") as f:
        pickle.dump(edge_attributes, f)
    print("Matrices and attributes saved successfully.")

# Main function
def main():
    print("Loading data...")
    data = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
    data = data.sample(frac=FRACTION, random_state=42).reset_index(drop=True)

    print("Constructing graph...")
    G = nx.Graph()
    edge_attributes = []

    # Use a set to track unique edges
    added_edges = set()

    for _, row in data.iterrows():
        start_node = (row["Start_Lat"], row["Start_Lon"])
        end_node = (row["End_Lat"], row["End_Lon"])

        # Skip self-loops
        if start_node == end_node:
            continue

        # Check for duplicate edges
        edge = (start_node, end_node)
        reverse_edge = (end_node, start_node)
        if edge in added_edges or reverse_edge in added_edges:
            continue

        # Add the edge to the graph
        attributes = {
            'Start_Node': start_node,
            'End_Node': end_node,
            'Local_ID': row['Local_ID'],
            'Bld_densit': row['Bld_densit'],
            'Veg_densit': row['Veg_densit'],
            'HW_Ratio': row['HW_Ratio'],
            'Width': row['Width'],
            'Pred_LandU': row['Pred_LandU'],
            'Mean_Popul': row['Mean_Popul'],
            'Mean_Bld_E': row['Mean_Bld_E'],
            'Max_Bld_He': row['Max_Bld_He'],
            'Delta_T': row['Delta_T'],
            'Water_dens': row['Water_dens']
        }
        G.add_edge(start_node, end_node, **attributes)
        edge_attributes.append(attributes)

        # Mark this edge as added
        added_edges.add(edge)

    print(f"Graph created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")
    
    # Note: Skipping extraction of the largest connected component. All edges are used.

    # Compute distance matrix using haversine distances
    distance_matrix = compute_haversine_distance_matrix(edge_attributes)

    # Save matrix and attributes (shortest path matrix omitted)
    save_matrices(distance_matrix, edge_attributes)

    print("Script completed successfully.")

if __name__ == "__main__":
    main()



In [ ]:
## MONTREAL GRAPH + MATRICES
## NOTE: THIS VERSION DOES NOT INCLUDE EXTRACTION OF THE MAIN COMPONENT
## THERE IS ALSO NO NODE-BASED MATRIX

import pandas as pd
import networkx as nx
import numpy as np
import pickle
import os

# Global variables
FILE_PATH = "./Datasets/Montreal_coordinates.xlsx"
SHEET_NAME = "Sheet1"
OUTPUT_DIR = "./precomputed_matrices"
FRACTION = 1  # Use a fraction of the data for testing (set to 1.0 for full dataset)
PROGRESS_STEP = 100  # Print progress every 100 rows

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Function to compute haversine distance
def haversine(lat1, lon1, lat2, lon2):
    R = 6371e3  # Earth's radius in meters
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    delta_phi = phi2 - phi1
    delta_lambda = np.radians(lon2 - lon1)
    a = np.sin(delta_phi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

# Compute haversine distance matrix
def compute_haversine_distance_matrix(edge_attributes):
    print("Computing haversine distance matrix...")
    centroids = [
        (
            (attr['Start_Node'][0] + attr['End_Node'][0]) / 2,
            (attr['Start_Node'][1] + attr['End_Node'][1]) / 2
        )
        for attr in edge_attributes
    ]

    n = len(centroids)
    dist_matrix = np.zeros((n, n))
    for i, coord1 in enumerate(centroids):
        for j in range(i + 1, n):  # Compute only upper triangle
            coord2 = centroids[j]
            dist_matrix[i, j] = haversine(coord1[0], coord1[1], coord2[0], coord2[1])
        if i % PROGRESS_STEP == 0:
            print(f"Processed {i}/{n} rows for distance matrix...")
    dist_matrix += dist_matrix.T  # Mirror upper triangle to lower triangle
    print("Haversine distance matrix computation completed.")
    return dist_matrix

# Save matrices and metadata
def save_matrices(distance_matrix, edge_attributes):
    print("Saving matrices...")
    np.save(f"{OUTPUT_DIR}/distance_matrix_Montreal.npy", distance_matrix)
    with open(f"{OUTPUT_DIR}/edge_attributes_Montreal.pkl", "wb") as f:
        pickle.dump(edge_attributes, f)
    print("Matrices and attributes saved successfully.")

# Main function
def main():
    print("Loading data...")
    data = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
    data = data.sample(frac=FRACTION, random_state=42).reset_index(drop=True)

    print("Constructing graph...")
    G = nx.Graph()
    edge_attributes = []

    # Use a set to track unique edges
    added_edges = set()

    for _, row in data.iterrows():
        start_node = (row["Start_Lat"], row["Start_Lon"])
        end_node = (row["End_Lat"], row["End_Lon"])

        # Skip self-loops
        if start_node == end_node:
            continue

        # Check for duplicate edges
        edge = (start_node, end_node)
        reverse_edge = (end_node, start_node)
        if edge in added_edges or reverse_edge in added_edges:
            continue

        # Add the edge to the graph
        attributes = {
            'Start_Node': start_node,
            'End_Node': end_node,
            'Local_ID': row['Local_ID'],
            'Bld_densit': row['Bld_densit'],
            'Veg_densit': row['Veg_densit'],
            'HW_Ratio': row['HW_Ratio'],
            'Width': row['Width'],
            'Pred_LandU': row['Pred_LandU'],
            'Mean_Popul': row['Mean_Popul'],
            'Mean_Bld_E': row['Mean_Bld_E'],
            'Max_Bld_He': row['Max_Bld_He'],
            'Delta_T': row['Delta_T'],
            'Water_dens': row['Water_dens']
        }
        G.add_edge(start_node, end_node, **attributes)
        edge_attributes.append(attributes)

        # Mark this edge as added
        added_edges.add(edge)

    print(f"Graph created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")
    
    # Note: Skipping extraction of the largest connected component. All edges are used.

    # Compute distance matrix using haversine distances
    distance_matrix = compute_haversine_distance_matrix(edge_attributes)

    # Save matrix and attributes (shortest path matrix omitted)
    save_matrices(distance_matrix, edge_attributes)

    print("Script completed successfully.")

if __name__ == "__main__":
    main()



In [ ]:
## APELDOORN BIKE GRAPH + MATRICES
## NOTE: THIS VERSION DOES NOT INCLUDE EXTRACTION OF THE MAIN COMPONENT
## THERE IS ALSO NO NODE-BASED MATRIX AND THERE IS A TIME COMPONENT ADDED 

import pandas as pd
import networkx as nx
import numpy as np
import pickle
import os

# Global variables
FILE_PATH = "./Datasets/BikeData_Coordinates.xlsx"
SHEET_NAME = "df_bikeCoord"
OUTPUT_DIR = "./precomputed_matrices"

FRACTION = 1.0      # Use a fraction of the data (1.0 for full dataset)
PROGRESS_STEP = 100 # Print progress every 100 rows

os.makedirs(OUTPUT_DIR, exist_ok=True)

# -------------------------------------------------------------------------
# 1) Haversine Distance Function
# -------------------------------------------------------------------------
def haversine(lat1, lon1, lat2, lon2):
    """
    Compute great-circle distance in meters between two lat/lon points.
    """
    R = 6371e3  # Earth's radius in meters
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    delta_phi = phi2 - phi1
    delta_lambda = np.radians(lon2 - lon1)
    a = (np.sin(delta_phi / 2) ** 2 +
         np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda / 2) ** 2)
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

# -------------------------------------------------------------------------
# 2) Main Construction
# -------------------------------------------------------------------------
def main():
    print("Loading data...")
    data = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
    data = data.sample(frac=FRACTION, random_state=42).reset_index(drop=True)

    # This dictionary will map a geometry (start/end lat-lon) to a unique segment_id
    geometry_to_id = {}
    next_segment_id = 0

    # Lists to store final outputs
    segment_attributes = []  # geometry-based attributes (one entry per unique segment)
    timebased_data = []      # time-varying attributes (one entry per row/time-step)

    print("Identifying unique geometry segments and building time-based records...")

    for idx, row in data.iterrows():
        if idx % PROGRESS_STEP == 0:
            print(f"Processed {idx}/{len(data)} rows...")

        # Extract geometry
        start_node = (row["Start_Lat"], row["Start_Lon"])
        end_node = (row["End_Lat"], row["End_Lon"])

        # Skip self-loops
        if start_node == end_node:
            continue

        # Sort the tuple so that (A,B) and (B,A) become the same geometry
        geometry_key = tuple(sorted([start_node, end_node]))

        # If it's a brand new geometry, assign a new segment_id and store geometry-level attributes
        if geometry_key not in geometry_to_id:
            geometry_to_id[geometry_key] = next_segment_id

            # Example geometry-based attributes (constant in time):
            # If you have more columns that are truly constant across time, add them here
            seg_attr = {
                "segment_id": next_segment_id,
                "Start_Node": start_node,
                "End_Node": end_node,
                "Local_ID": row["Local_ID"],
                "Bld_densit": row["Bld_densit"],
                "Veg_densit": row["Veg_densit"],
                "HW_Ratio": row["HW_Ratio"],
                "Road_width": row["Road_width"],
                "Pred_LandUse": row["Pred_LandUse"],
                "Mean_population": row["Mean_population"],
                "Mean_Bld_E": row["Mean_Bld_E"],
                "Max_Bld_He": row["Max_Bld_He"],
                "Water_dens": row["Water_dens"],
                "Pred_facade_material": row["Pred_facade_material"]
                # Add any other truly static features if needed
            }
            segment_attributes.append(seg_attr)

            next_segment_id += 1

        # Identify the segment_id for this row
        seg_id = geometry_to_id[geometry_key]

        # Store time-varying data (date, part_of_day, temperature, humidity, etc.)
        # Adjust these column names to match your dataset
        time_data = {
            "segment_id": seg_id,
            "Date": row["Date"],
            "Year": row["Year"],
            "Month": row["Month"],
            "Part_of_the_day": row["Part_of_the_day"],
            # Time-varying attributes:
            "Mean_canopy_temp": row.get("Mean_canopy_temp", np.nan),
            "Ref_canopy_temp": row.get("Ref_canopy_temp", np.nan),
            "Mean_surface_temp": row.get("Mean_surface_temp", np.nan),
            "Ref_surface_temp": row.get("Ref_surface_temp", np.nan),
            "Mean_RH": row.get("Mean_RH", np.nan),
            "Ref_RH": row.get("Ref_RH", np.nan),
            "Delta_canopy": row.get("Delta_canopy", np.nan),
            "Delta_surface": row.get("Delta_surface", np.nan)
        }
        timebased_data.append(time_data)

    # ---------------------------------------------------------------------
    # Build a graph with unique segments and compute the distance matrix
    # ---------------------------------------------------------------------
    print("Constructing graph for unique segments...")
    G = nx.Graph()

    for seg_attr in segment_attributes:
        sid = seg_attr["segment_id"]
        start_node = seg_attr["Start_Node"]
        end_node = seg_attr["End_Node"]

        # Add an edge to the graph with geometry-level attributes
        G.add_edge(start_node, end_node, **seg_attr)

    print(f"Graph created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")
    print("Computing haversine distance matrix for unique segments...")

    # Compute centroids for each unique segment
    n_segments = len(segment_attributes)
    centroids = []
    for seg_attr in segment_attributes:
        lat_c = (seg_attr["Start_Node"][0] + seg_attr["End_Node"][0]) / 2
        lon_c = (seg_attr["Start_Node"][1] + seg_attr["End_Node"][1]) / 2
        centroids.append((lat_c, lon_c))

    # Calculate NxN distance matrix (N = number of unique segments)
    dist_matrix = np.zeros((n_segments, n_segments), dtype=float)
    for i in range(n_segments):
        for j in range(i + 1, n_segments):
            coord1 = centroids[i]
            coord2 = centroids[j]
            dist_matrix[i, j] = haversine(coord1[0], coord1[1], coord2[0], coord2[1])
        # Mirror the distances
        dist_matrix[i + 1 :, i] = dist_matrix[i, i + 1 :]

    # Or simply mirror after the loop to ensure symmetry:
    dist_matrix = dist_matrix + dist_matrix.T

    print("Distance matrix computation completed.")

    # ---------------------------------------------------------------------
    #  Save outputs
    # ---------------------------------------------------------------------
    print("Saving outputs...")
    np.save(f"{OUTPUT_DIR}/distance_matrix_Bike.npy", dist_matrix)

    # Save geometry-based attributes
    with open(f"{OUTPUT_DIR}/edge_attributes_Bike.pkl", "wb") as f:
        pickle.dump(segment_attributes, f)

    # Save time-based attributes
    with open(f"{OUTPUT_DIR}/timebased_attributes_Bike.pkl", "wb") as f:
        pickle.dump(timebased_data, f)

    print("All files saved. Script completed successfully.")


if __name__ == "__main__":
    main()

### GA Random Forest Apeldoorn Satellite

In [ ]:
## RADIUS-BASED SELECTION, 4 AGGREGATION METHODS, RF TRAINING

import time
import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
from deap import base, creator, tools, algorithms
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import pickle
from sklearn.decomposition import PCA

# ==== USER CONFIGURATION ====
EXPERIMENT = 4  # Choose experiment: 1, 2, 3 (no alpha/beta), 4 (uses alpha/beta)

# ==== LOAD DATA ====
OUTPUT_DIR = "./precomputed_matrices"

FEATURE_COLUMNS = [
    'Bld_densit', 'Veg_densit', 'HW_Ratio', 'Width', 'Pred_Facad',
    'Pred_LandU', 'Mean_Popul', 'Mean_Bld_E', 'Max_Bld_He',
    'Water_dens', 'Bike_pct_a', 'Vehicle_pc'
]
TARGET_COLUMN = 'Delta_T'

print("\nLoading data...")
# Load precomputed matrices; neighbor selection will be based on the distance matrix.
shortest_path_matrix = np.load(f"{OUTPUT_DIR}/shortest_path_matrix.npy")
distance_matrix = np.load(f"{OUTPUT_DIR}/distance_matrix.npy")
with open(f"{OUTPUT_DIR}/edge_attributes.pkl", "rb") as f:
    edge_attributes = pickle.load(f)
print(" Data Loaded.")

# ==== FEATURE AGGREGATION FUNCTION (Radius-based Neighbor Selection) ====
aggregation_cache = {}  # Cache for computed aggregations

def aggregate_features(edge_attributes, distance_matrix, feature_columns, radius, alpha=0.5, beta=0.5):
    cache_key = (EXPERIMENT, radius, alpha, beta)
    if cache_key in aggregation_cache:
        return aggregation_cache[cache_key]
    
    aggregated_features = {f"original_{col}": [] for col in feature_columns}
    if EXPERIMENT in [1, 2, 4]:
        if EXPERIMENT == 4:
            aggregated_features.update({f"weighted_avg_{col}_neighbors": [] for col in feature_columns})
        else:
            aggregated_features.update({f"avg_{col}_neighbors": [] for col in feature_columns})
    
    # Loop over each segment and aggregate features from neighbors within the given radius
    for segment_index, segment_attrs in enumerate(edge_attributes):
        neighbor_indices = np.where(distance_matrix[segment_index] <= radius)[0]
        neighbors = [n for n in neighbor_indices if n != segment_index]
        
        if neighbors:
            neighbor_features = np.array([
                [edge_attributes[n].get(col, np.nan) for col in feature_columns] 
                for n in neighbors
            ])
            # Replace NaNs with the median along each column of neighbor features
            neighbor_features = np.where(np.isnan(neighbor_features),
                                         np.nanmedian(neighbor_features, axis=0),
                                         neighbor_features)
            avg_neighbors = np.nanmean(neighbor_features, axis=0)
            
            if EXPERIMENT == 4:
                # Local normalization among only the selected neighbors
                local_shortest = shortest_path_matrix[segment_index, neighbors]
                local_distance = distance_matrix[segment_index, neighbors]
                local_max_shortest = np.max(local_shortest)
                local_max_distance = np.max(local_distance)
                # Avoid division by zero
                if local_max_shortest > 0:
                    norm_shortest_local = 1 - local_shortest / local_max_shortest
                else:
                    norm_shortest_local = np.ones_like(local_shortest)
                if local_max_distance > 0:
                    norm_distance_local = 1 - local_distance / local_max_distance
                else:
                    norm_distance_local = np.ones_like(local_distance)
                # Combine the two normalized values using alpha and beta
                weights = alpha * norm_shortest_local + beta * norm_distance_local
                weights_sum = np.sum(weights)
                if weights_sum != 0:
                    weights /= weights_sum
                else:
                    weights = np.full_like(weights, 1/len(weights))
                weighted_avg = [np.average(neighbor_features[:, i], weights=weights) 
                                for i in range(len(feature_columns))]
        else:
            avg_neighbors = np.array([segment_attrs.get(col, np.nan) for col in feature_columns])
            if EXPERIMENT == 4:
                weighted_avg = avg_neighbors

        # Append the original and aggregated features
        for i, col in enumerate(feature_columns):
            aggregated_features[f"original_{col}"].append(segment_attrs.get(col, np.nan))
            if EXPERIMENT in [1, 2, 4]:
                if EXPERIMENT == 4:
                    aggregated_features[f"weighted_avg_{col}_neighbors"].append(weighted_avg[i])
                else:
                    aggregated_features[f"avg_{col}_neighbors"].append(avg_neighbors[i])
                    
    aggregated_df = pd.DataFrame(aggregated_features)
    
    # For Experiment 3, perform a global PCA for each feature.
    if EXPERIMENT == 3:
        avg_values = {col: [] for col in feature_columns}
        for segment_index, segment_attrs in enumerate(edge_attributes):
            seg_feats = {col: segment_attrs.get(col, np.nan) for col in feature_columns}
            for col in feature_columns:
                if np.isnan(seg_feats[col]):
                    seg_feats[col] = np.nanmedian([edge.get(col, np.nan) for edge in edge_attributes])
            neighbor_indices = np.where(distance_matrix[segment_index] <= radius)[0]
            neighbors = [n for n in neighbor_indices if n != segment_index]
            if neighbors:
                neighbor_feats = np.array([
                    [edge_attributes[n].get(col, np.nan) for col in feature_columns]
                    for n in neighbors
                ])
                neighbor_feats = np.where(np.isnan(neighbor_feats),
                                          np.nanmedian(neighbor_feats, axis=0),
                                          neighbor_feats)
                avg_neighbors = np.nanmean(neighbor_feats, axis=0)
            else:
                avg_neighbors = np.array([seg_feats[col] for col in feature_columns])
            for i, col in enumerate(feature_columns):
                avg_values[col].append(avg_neighbors[i])
        for col in feature_columns:
            aggregated_df[f"avg_{col}_neighbors"] = avg_values[col]
        for col in feature_columns:
            feature_pair = aggregated_df[[f"original_{col}", f"avg_{col}_neighbors"]].fillna(0)
            pca = PCA(n_components=1)
            aggregated_df[f"pca_{col}"] = pca.fit_transform(feature_pair).flatten()
    
    aggregation_cache[cache_key] = aggregated_df
    return aggregated_df

# Global variable to hold the current radius for the GA run.
current_radius = None

# ==== GENETIC ALGORITHM EVALUATION FUNCTION ====
def evaluate_rf(individual):
    global current_radius
    radius = current_radius
    if EXPERIMENT == 4:
        # Individual: [alpha, n_estimators, max_features, max_depth, min_samples_split, bootstrap]
        alpha = individual[0]
        n_estimators = individual[1]
        max_features = individual[2]
        max_depth = individual[3]
        min_samples_split = individual[4]
        bootstrap = individual[5]
        beta = 1 - alpha
    else:
        # Individual: [n_estimators, max_features, max_depth, min_samples_split, bootstrap]
        n_estimators = individual[0]
        max_features = individual[1]
        max_depth = individual[2]
        min_samples_split = individual[3]
        bootstrap = individual[4]
        alpha, beta = 0.5, 0.5

    aggregated_data = aggregate_features(edge_attributes, distance_matrix, FEATURE_COLUMNS, radius, alpha, beta)
    
    if EXPERIMENT == 3:
        X = aggregated_data[[f"pca_{col}" for col in FEATURE_COLUMNS]]
    elif EXPERIMENT == 2:
        X = pd.DataFrame()
        for col in FEATURE_COLUMNS:
            X[col] = (aggregated_data[f"original_{col}"] + aggregated_data[f"avg_{col}_neighbors"]) / 2
    elif EXPERIMENT == 4:
        X = aggregated_data[[f"original_{col}" for col in FEATURE_COLUMNS] +
                            [f"weighted_avg_{col}_neighbors" for col in FEATURE_COLUMNS]]
    else:
        X = aggregated_data[[f"original_{col}" for col in FEATURE_COLUMNS] +
                            [f"avg_{col}_neighbors" for col in FEATURE_COLUMNS]]
    
    X = X.fillna(X.median())
    y = np.array([edge_attrs[TARGET_COLUMN] for edge_attrs in edge_attributes])
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    rf = RandomForestRegressor(
        n_estimators=n_estimators,
        max_features=max_features,
        max_depth=max_depth,
        min_samples_split=int(min_samples_split),
        bootstrap=bool(bootstrap),
        random_state=42
    )
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    return (r2_score(y_test, y_pred),)

# ==== GENETIC ALGORITHM SETUP ====
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("n_estimators", random.choice, [10, 30, 50, 100, 150, 200])
toolbox.register("max_features", random.choice, ['sqrt', 'log2'])
toolbox.register("max_depth", random.choice, [10, 20, 30, 50, 100])
toolbox.register("min_samples_split", random.choice, [2, 4, 6])
toolbox.register("bootstrap", random.choice, [True, False])
if EXPERIMENT == 4:
    toolbox.register("alpha", random.choice, [0, 0.25, 0.5, 0.75, 1])

def create_individual():
    if EXPERIMENT == 4:
        return [toolbox.alpha(), toolbox.n_estimators(), toolbox.max_features(), toolbox.max_depth(), toolbox.min_samples_split(), toolbox.bootstrap()]
    else:
        return [toolbox.n_estimators(), toolbox.max_features(), toolbox.max_depth(), toolbox.min_samples_split(), toolbox.bootstrap()]

toolbox.register("individual", tools.initIterate, creator.Individual, create_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", evaluate_rf)
toolbox.register("mate", tools.cxTwoPoint)

def mutate_individual(ind):
    if EXPERIMENT == 4:
        ind[:] = [toolbox.alpha(), toolbox.n_estimators(), toolbox.max_features(), toolbox.max_depth(), toolbox.min_samples_split(), toolbox.bootstrap()]
    else:
        ind[:] = [toolbox.n_estimators(), toolbox.max_features(), toolbox.max_depth(), toolbox.min_samples_split(), toolbox.bootstrap()]
    return (ind,)

toolbox.register("mutate", mutate_individual)
toolbox.register("select", tools.selTournament, tournsize=3)

# ==== GA PARAMETERS ====
POP_SIZE = 50
GENS = 20
CXPB, MUTPB = 0.8, 0.5
radii_list = [0, 100, 200, 300, 400, 500, 600, 700, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 8500, 8700]
results_per_radius = {}

print("\nStarting GA Optimization for each radius...")
for radius in radii_list:
    print(f"\nRunning GA for radius = {radius}")
    current_radius = radius
    start_time = time.time()
    
    # Initialize population for this radius
    pop = toolbox.population(n=POP_SIZE)
    
    # Early stopping parameters
    improvement_threshold = 0.01
    max_no_improve = 5
    no_improve_count = 0
    best_overall_r2 = -np.inf
    best_overall_individual = None
    
    for gen in range(GENS):
        print(f"\n🔄 Generation {gen + 1}/{GENS} for radius {radius}")
        fitnesses = list(map(toolbox.evaluate, tqdm(pop, desc="Evaluating individuals", leave=False)))
        r2_scores = []
        for ind, fit in zip(pop, fitnesses):
            ind.fitness.values = fit
            r2_scores.append(fit[0])
        print(f" R² Scores: {r2_scores}")
        
        gen_best_r2 = max(r2_scores)
        gen_best_index = r2_scores.index(gen_best_r2)
        gen_best_individual = toolbox.clone(pop[gen_best_index])
        
        if gen_best_r2 - best_overall_r2 > improvement_threshold:
            best_overall_r2 = gen_best_r2
            best_overall_individual = toolbox.clone(gen_best_individual)
            no_improve_count = 0
        else:
            no_improve_count += 1
        
        print(f"🔥 Best R² in Gen {gen + 1}: {gen_best_r2:.4f}")
        if no_improve_count >= max_no_improve:
            print(f"🚫 No significant improvement in the last {max_no_improve} generations. Early stopping...")
            break
        
        offspring = toolbox.select(pop, len(pop))
        offspring = list(map(toolbox.clone, offspring))
        
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CXPB:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values
        
        for mutant in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(mutant)
                del mutant.fitness.values
        
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = list(map(toolbox.evaluate, tqdm(invalid_ind, desc="Evaluating offspring", leave=False)))
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit
        
        pop[:] = offspring
    
    # Rescan final population to ensure the best overall individual is chosen
    final_best = max(pop, key=lambda ind: ind.fitness.values[0])
    if final_best.fitness.values[0] > best_overall_r2:
        best_overall_r2 = final_best.fitness.values[0]
        best_overall_individual = toolbox.clone(final_best)
    
    elapsed = time.time() - start_time
    print(f"\nBest hyperparameters for radius {radius}: {best_overall_individual}")
    print(f"🔥 Best R² Score for radius {radius}: {best_overall_r2:.4f}")
    print(f"⏱ Elapsed time for radius {radius}: {elapsed:.2f} seconds")
    
    results_per_radius[radius] = {
        "best_hyperparameters": best_overall_individual,
        "best_r2": best_overall_r2,
        "time": elapsed
    }

radii = sorted(results_per_radius.keys())
r2_scores = [results_per_radius[r]["best_r2"] for r in radii]

plt.figure(figsize=(10, 6))
plt.plot(radii, r2_scores, marker='o', linestyle='-', color='b')
plt.xlabel("Radius")
plt.ylabel("Best R² Score")
plt.title("Best R² Score vs. Radius")
plt.grid(True)
plt.show()

print("\n📌 Summary of Best Hyperparameters per Radius:")
for radius in radii:
    best_params = results_per_radius[radius]["best_hyperparameters"]
    best_r2 = results_per_radius[radius]["best_r2"]
    print(f"Radius: {radius} | Best R²: {best_r2:.4f} | Hyperparameters: {best_params}")

In [ ]:
## NODE-BASED SELECTION, 4 AGGREGATION METHODS, RF TRAINING

import time
import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
from deap import base, creator, tools
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import pickle
from sklearn.decomposition import PCA

# ==== USER CONFIGURATION ====
EXPERIMENT = 4  # Choose experiment: 1, 2, 3 (no alpha/beta), 4 (uses alpha/beta)

# ==== LOAD DATA ====
OUTPUT_DIR = "./precomputed_matrices"

FEATURE_COLUMNS = [
    'Bld_densit', 'Veg_densit', 'HW_Ratio', 'Width', 'Pred_Facad',
    'Pred_LandU', 'Mean_Popul', 'Mean_Bld_E', 'Max_Bld_He',
    'Water_dens', 'Bike_pct_a', 'Vehicle_pc'
]
TARGET_COLUMN = 'Delta_T'

print("\nLoading data...")
# Load precomputed matrices; neighbor selection will be based on the shortest path matrix.
shortest_path_matrix = np.load(f"{OUTPUT_DIR}/shortest_path_matrix.npy")
distance_matrix = np.load(f"{OUTPUT_DIR}/distance_matrix.npy")
with open(f"{OUTPUT_DIR}/edge_attributes.pkl", "rb") as f:
    edge_attributes = pickle.load(f)
print(" Data Loaded.")

# ==== FEATURE AGGREGATION FUNCTION (Node-based Neighbor Selection) ====
aggregation_cache = {}  # Cache for computed aggregations

def aggregate_features(edge_attrs, sp_matrix, feat_cols, radius, alpha=0.5, beta=0.5):
    cache_key = (EXPERIMENT, radius, alpha, beta)
    if cache_key in aggregation_cache:
        return aggregation_cache[cache_key]
    
    aggregated_features = {f"original_{c}": [] for c in feat_cols}
    if EXPERIMENT in [1, 2, 4]:
        if EXPERIMENT == 4:
            aggregated_features.update({f"weighted_avg_{c}_neighbors": [] for c in feat_cols})
        else:
            aggregated_features.update({f"avg_{c}_neighbors": [] for c in feat_cols})
    
    # No precomputation of global normalization here; we'll do local normalization.
    for seg_idx, seg_attrs in enumerate(edge_attrs):
        # Determine neighbors using the shortest path matrix.
        neighbor_indices = np.where(sp_matrix[seg_idx] <= radius)[0]
        # Exclude the segment itself.
        neighbors = [n for n in neighbor_indices if n != seg_idx]
        
        if neighbors:
            neighbor_feats = np.array([
                [edge_attrs[n].get(col, np.nan) for col in feat_cols]
                for n in neighbors
            ])
            # Replace NaNs with median along each column.
            neighbor_feats = np.where(np.isnan(neighbor_feats),
                                      np.nanmedian(neighbor_feats, axis=0),
                                      neighbor_feats)
            avg_neighbors = np.nanmean(neighbor_feats, axis=0)
            if EXPERIMENT == 4:
                # Local normalization: compute local maximums only among selected neighbors.
                local_sp = sp_matrix[seg_idx, neighbors]
                local_dist = distance_matrix[seg_idx, neighbors]
                max_sp = np.max(local_sp) if len(local_sp) > 0 else 0
                max_dist = np.max(local_dist) if len(local_dist) > 0 else 0
                if max_sp > 0:
                    norm_sp_local = 1 - (local_sp / max_sp)
                else:
                    norm_sp_local = np.ones_like(local_sp)
                if max_dist > 0:
                    norm_dist_local = 1 - (local_dist / max_dist)
                else:
                    norm_dist_local = np.ones_like(local_dist)
                # Combine normalized values using alpha and beta.
                weights = alpha * norm_sp_local + beta * norm_dist_local
                w_sum = np.sum(weights)
                if w_sum != 0:
                    weights /= w_sum
                else:
                    weights = np.full_like(weights, 1/len(weights))
                weighted_avg = [np.average(neighbor_feats[:, i], weights=weights)
                                for i in range(len(feat_cols))]
        else:
            avg_neighbors = np.array([seg_attrs.get(col, np.nan) for col in feat_cols])
            if EXPERIMENT == 4:
                weighted_avg = avg_neighbors

        for i, col in enumerate(feat_cols):
            aggregated_features[f"original_{col}"].append(seg_attrs.get(col, np.nan))
            if EXPERIMENT in [1, 2, 4]:
                if EXPERIMENT == 4:
                    aggregated_features[f"weighted_avg_{col}_neighbors"].append(weighted_avg[i])
                else:
                    aggregated_features[f"avg_{col}_neighbors"].append(avg_neighbors[i])
    
    aggregated_df = pd.DataFrame(aggregated_features)
    
    # For Experiment 3, perform global PCA for each feature.
    if EXPERIMENT == 3:
        avg_values = {col: [] for col in feat_cols}
        for seg_idx, seg_attrs in enumerate(edge_attrs):
            seg_feats = {col: seg_attrs.get(col, np.nan) for col in feat_cols}
            for col in feat_cols:
                if np.isnan(seg_feats[col]):
                    seg_feats[col] = np.nanmedian([e.get(col, np.nan) for e in edge_attrs])
            neighbor_indices = np.where(sp_matrix[seg_idx] <= radius)[0]
            neighbors = [n for n in neighbor_indices if n != seg_idx]
            if neighbors:
                neighbor_feats = np.array([
                    [edge_attrs[n].get(col, np.nan) for col in feat_cols]
                    for n in neighbors
                ])
                neighbor_feats = np.where(np.isnan(neighbor_feats),
                                          np.nanmedian(neighbor_feats, axis=0),
                                          neighbor_feats)
                avg_nb = np.nanmean(neighbor_feats, axis=0)
            else:
                avg_nb = np.array([seg_feats[c] for c in feat_cols])
            for i, c in enumerate(feat_cols):
                avg_values[c].append(avg_nb[i])
        for col in feat_cols:
            aggregated_df[f"avg_{col}_neighbors"] = avg_values[col]
        for col in feat_cols:
            pair = aggregated_df[[f"original_{col}", f"avg_{col}_neighbors"]].fillna(0)
            pca = PCA(n_components=1)
            aggregated_df[f"pca_{col}"] = pca.fit_transform(pair).flatten()
    
    aggregation_cache[cache_key] = aggregated_df
    return aggregated_df

# Global variable to hold the current nodes threshold (radius) for the GA run.
current_radius = None

def evaluate_rf(ind):
    global current_radius
    radius = current_radius
    # Unpack hyperparameters.
    if EXPERIMENT == 4:
        alpha = ind[0]
        n_estimators = ind[1]
        max_features = ind[2]
        max_depth = ind[3]
        min_samples_split = ind[4]
        bootstrap = ind[5]
        beta = 1 - alpha
    else:
        n_estimators = ind[0]
        max_features = ind[1]
        max_depth = ind[2]
        min_samples_split = ind[3]
        bootstrap = ind[4]
        alpha, beta = 0.5, 0.5

    agg_data = aggregate_features(edge_attributes, shortest_path_matrix, FEATURE_COLUMNS,
                                  radius, alpha, beta)
    
    if EXPERIMENT == 3:
        X = agg_data[[f"pca_{c}" for c in FEATURE_COLUMNS]]
    elif EXPERIMENT == 2:
        X = pd.DataFrame()
        for c in FEATURE_COLUMNS:
            X[c] = (agg_data[f"original_{c}"] + agg_data[f"avg_{c}_neighbors"]) / 2
    elif EXPERIMENT == 4:
        X = agg_data[[f"original_{c}" for c in FEATURE_COLUMNS] +
                     [f"weighted_avg_{c}_neighbors" for c in FEATURE_COLUMNS]]
    else:
        X = agg_data[[f"original_{c}" for c in FEATURE_COLUMNS] +
                     [f"avg_{c}_neighbors" for c in FEATURE_COLUMNS]]
    
    X = X.fillna(X.median())
    y = np.array([e[TARGET_COLUMN] for e in edge_attributes])
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    rf = RandomForestRegressor(
        n_estimators=n_estimators,
        max_features=max_features,
        max_depth=max_depth,
        min_samples_split=int(min_samples_split),
        bootstrap=bool(bootstrap),
        random_state=42
    )
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    return (r2_score(y_test, y_pred),)

# ==== GENETIC ALGORITHM SETUP ====
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("n_estimators", random.choice, [10, 30, 50, 100, 150, 200])
toolbox.register("max_features", random.choice, ['sqrt', 'log2'])
toolbox.register("max_depth", random.choice, [10, 20, 30, 50, 100])
toolbox.register("min_samples_split", random.choice, [2, 4, 6])
toolbox.register("bootstrap", random.choice, [True, False])
if EXPERIMENT == 4:
    toolbox.register("alpha", random.choice, [0, 0.25, 0.5, 0.75, 1])

def create_individual():
    if EXPERIMENT == 4:
        return [toolbox.alpha(), toolbox.n_estimators(), toolbox.max_features(),
                toolbox.max_depth(), toolbox.min_samples_split(), toolbox.bootstrap()]
    else:
        return [toolbox.n_estimators(), toolbox.max_features(), toolbox.max_depth(),
                toolbox.min_samples_split(), toolbox.bootstrap()]

toolbox.register("individual", tools.initIterate, creator.Individual, create_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", evaluate_rf)
toolbox.register("mate", tools.cxTwoPoint)

def mutate_individual(ind):
    if EXPERIMENT == 4:
        ind[:] = [toolbox.alpha(), toolbox.n_estimators(), toolbox.max_features(),
                  toolbox.max_depth(), toolbox.min_samples_split(), toolbox.bootstrap()]
    else:
        ind[:] = [toolbox.n_estimators(), toolbox.max_features(), toolbox.max_depth(),
                  toolbox.min_samples_split(), toolbox.bootstrap()]
    return ind,

toolbox.register("mutate", mutate_individual)
toolbox.register("select", tools.selTournament, tournsize=3)

# ==== GA PARAMETERS ====
POP_SIZE = 50
GENS = 50
CXPB, MUTPB = 0.8, 0.5

# List of node thresholds (radii) to iterate over.
nodes_list = [0, 1, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 36, 52, 68, 84, 100, 125, 150, 175, 200, 205, 210, 215]
results_per_nodes = {}

# Define improvement threshold (e.g., 0.001)
improvement_threshold = 0.01

for nodes in nodes_list:
    print(f"\nRunning GA for {nodes} nodes")
    current_radius = nodes  # Set the current nodes threshold.
    start_time = time.time()

    pop = toolbox.population(n=POP_SIZE)
    max_no_improve = 5
    no_improve_count = 0
    best_overall_r2 = -np.inf
    best_overall_individual = None

    for gen in range(GENS):
        print(f"\n🔄 Generation {gen+1}/{GENS} for {nodes} nodes")
        fitnesses = list(map(toolbox.evaluate, tqdm(pop, desc="Evaluating individuals", leave=False)))
        r2_scores = []
        for ind, fit in zip(pop, fitnesses):
            ind.fitness.values = fit
            r2_scores.append(fit[0])
        
        gen_best_r2 = max(r2_scores)
        gen_best_index = r2_scores.index(gen_best_r2)
        gen_best_individual = toolbox.clone(pop[gen_best_index])
        
        # Update best overall only if improvement exceeds threshold.
        if gen_best_r2 - best_overall_r2 > improvement_threshold:
            best_overall_r2 = gen_best_r2
            best_overall_individual = toolbox.clone(gen_best_individual)
            no_improve_count = 0
        else:
            no_improve_count += 1
        
        print(f"   🏆 Gen {gen+1} best R²: {gen_best_r2:.4f} | Overall best: {best_overall_r2:.4f}")
        if no_improve_count >= max_no_improve:
            print(f"🚫 Early stopping, no significant improvement in {max_no_improve} generations.")
            break
        
        offspring = toolbox.select(pop, len(pop))
        offspring = list(map(toolbox.clone, offspring))
        
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CXPB:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values
        
        for mutant in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(mutant)
                del mutant.fitness.values
        
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = list(map(toolbox.evaluate, tqdm(invalid_ind, desc="Evaluating offspring", leave=False)))
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit
        pop[:] = offspring

    # Final re-scan with threshold check.
    final_best = max(pop, key=lambda x: x.fitness.values[0])
    if final_best.fitness.values[0] - best_overall_r2 > improvement_threshold:
        best_overall_r2 = final_best.fitness.values[0]
        best_overall_individual = toolbox.clone(final_best)
    
    elapsed = time.time() - start_time
    print(f"\nBest hyperparameters for {nodes} nodes: {best_overall_individual}")
    print(f"🔥 Best R² Score for {nodes} nodes: {best_overall_r2:.4f}")
    print(f"⏱ Elapsed time for {nodes} nodes: {elapsed:.2f} seconds")
    
    results_per_nodes[nodes] = {
        "best_r2": best_overall_r2,
        "best_hyperparameters": best_overall_individual,
        "time": elapsed
    }

# ==== PLOTTING RESULTS ====
sorted_nodes = sorted(results_per_nodes.keys())
scores_plot = [results_per_nodes[n]["best_r2"] for n in sorted_nodes]

plt.figure(figsize=(10, 6))
plt.plot(sorted_nodes, scores_plot, marker='o', linestyle='-', color='blue')
plt.xlabel("Nodes threshold")
plt.ylabel("Best R² Score")
plt.title("GA-Optimized RF vs. Nodes threshold")
plt.grid(True)
plt.show()

print("\n📌 Summary of Best Hyperparameters per Nodes threshold:")
for nd in sorted_nodes:
    best_params = results_per_nodes[nd]["best_hyperparameters"]
    best_r2 = results_per_nodes[nd]["best_r2"]
    print(f"  {nd} nodes => R²: {best_r2:.4f} | Hyperparameters: {best_params}")


### GA Neural Network Apeldoorn Satellite

In [ ]:
## RADIUS-BASED SELECTION, 4 AGGREGATION METHODS, NN TRAINING

import time
import copy
import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt

# DEAP for the GA
from deap import base, creator, tools

# PyTorch for the NN
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# Sklearn for splitting and scoring
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.decomposition import PCA

from tqdm import tqdm
import pickle

# ==== USER CONFIGURATION ====
EXPERIMENT = 1  # 1, 2, 3, or 4

# ==== LOAD DATA ====
OUTPUT_DIR = "./precomputed_matrices"

FEATURE_COLUMNS = [
    'Bld_densit', 'Veg_densit', 'HW_Ratio', 'Width', 'Pred_Facad',
    'Pred_LandU', 'Mean_Popul', 'Mean_Bld_E', 'Max_Bld_He',
    'Water_dens', 'Bike_pct_a', 'Vehicle_pc'
]
TARGET_COLUMN = 'Delta_T'

print("\nLoading data...")
shortest_path_matrix = np.load(f"{OUTPUT_DIR}/shortest_path_matrix.npy")
distance_matrix = np.load(f"{OUTPUT_DIR}/distance_matrix.npy")
with open(f"{OUTPUT_DIR}/edge_attributes.pkl", "rb") as f:
    edge_attributes = pickle.load(f)
print(" Data Loaded.")

# If you want GPU acceleration:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ==== FEATURE AGGREGATION FUNCTION (Radius-based Neighbor Selection) ====
aggregation_cache = {}

def aggregate_features(edge_attributes, distance_matrix, feature_columns, radius, alpha=0.5, beta=0.5):
    """
    Same logic as your original script, with or without weighting.
    """
    cache_key = (EXPERIMENT, radius, alpha, beta)
    if cache_key in aggregation_cache:
        return aggregation_cache[cache_key]
    
    aggregated_features = {f"original_{col}": [] for col in feature_columns}
    
    if EXPERIMENT in [1, 2, 4]:
        if EXPERIMENT == 4:
            aggregated_features.update({f"weighted_avg_{col}_neighbors": [] for col in feature_columns})
        else:
            aggregated_features.update({f"avg_{col}_neighbors": [] for col in feature_columns})
    
    for segment_index, segment_attrs in enumerate(edge_attributes):
        neighbor_indices = np.where(distance_matrix[segment_index] <= radius)[0]
        neighbors = [n for n in neighbor_indices if n != segment_index]
        
        if neighbors:
            neighbor_features = np.array([
                [edge_attributes[n].get(col, np.nan) for col in feature_columns] 
                for n in neighbors
            ])
            neighbor_features = np.where(
                np.isnan(neighbor_features),
                np.nanmedian(neighbor_features, axis=0),
                neighbor_features
            )
            avg_neighbors = np.nanmean(neighbor_features, axis=0)
            
            if EXPERIMENT == 4:
                local_shortest = shortest_path_matrix[segment_index, neighbors]
                local_distance = distance_matrix[segment_index, neighbors]
                local_max_shortest = np.max(local_shortest)
                local_max_distance = np.max(local_distance)
                if local_max_shortest > 0:
                    norm_shortest_local = 1 - local_shortest / local_max_shortest
                else:
                    norm_shortest_local = np.ones_like(local_shortest)
                if local_max_distance > 0:
                    norm_distance_local = 1 - local_distance / local_max_distance
                else:
                    norm_distance_local = np.ones_like(local_distance)
                weights = alpha * norm_shortest_local + beta * norm_distance_local
                weights_sum = np.sum(weights)
                if weights_sum != 0:
                    weights /= weights_sum
                else:
                    weights = np.full_like(weights, 1 / len(weights))
                weighted_avg = np.dot(weights, neighbor_features)
        else:
            avg_neighbors = np.array([segment_attrs.get(col, np.nan) for col in feature_columns])
            if EXPERIMENT == 4:
                weighted_avg = avg_neighbors

        for i, col in enumerate(feature_columns):
            aggregated_features[f"original_{col}"].append(segment_attrs.get(col, np.nan))
            if EXPERIMENT in [1, 2, 4]:
                if EXPERIMENT == 4:
                    aggregated_features[f"weighted_avg_{col}_neighbors"].append(weighted_avg[i])
                else:
                    aggregated_features[f"avg_{col}_neighbors"].append(avg_neighbors[i])
    
    aggregated_df = pd.DataFrame(aggregated_features)
    
    if EXPERIMENT == 3:
        # PCA approach
        avg_values = {col: [] for col in feature_columns}
        for segment_index, segment_attrs in enumerate(edge_attributes):
            seg_feats = {col: segment_attrs.get(col, np.nan) for col in feature_columns}
            for col in feature_columns:
                if np.isnan(seg_feats[col]):
                    seg_feats[col] = np.nanmedian([edge.get(col, np.nan) for edge in edge_attributes])
            neighbor_indices = np.where(distance_matrix[segment_index] <= radius)[0]
            neighbors = [n for n in neighbor_indices if n != segment_index]
            if neighbors:
                neighbor_feats = np.array([
                    [edge_attributes[n].get(col, np.nan) for col in feature_columns]
                    for n in neighbors
                ])
                neighbor_feats = np.where(
                    np.isnan(neighbor_feats),
                    np.nanmedian(neighbor_feats, axis=0),
                    neighbor_feats
                )
                avg_neighbors = np.nanmean(neighbor_feats, axis=0)
            else:
                avg_neighbors = np.array([seg_feats[col] for col in feature_columns])
            for i, col in enumerate(feature_columns):
                avg_values[col].append(avg_neighbors[i])
        
        for col in feature_columns:
            aggregated_df[f"avg_{col}_neighbors"] = avg_values[col]
        
        # Then do PCA on each pair
        for col in feature_columns:
            feature_pair = aggregated_df[[f"original_{col}", f"avg_{col}_neighbors"]].fillna(0)
            pca = PCA(n_components=1)
            aggregated_df[f"pca_{col}"] = pca.fit_transform(feature_pair).flatten()
    
    aggregation_cache[cache_key] = aggregated_df
    return aggregated_df

# ==== PyTorch Model Definition ====
class Net(nn.Module):
    def __init__(self, input_dim, num_layers, num_neurons, activation, dropout_rate, kernel_init):
        super(Net, self).__init__()
        self.layers = nn.ModuleList()
        self.dropouts = nn.ModuleList()
        
        # Activation function mapping
        act_map = {
            'relu': nn.ReLU(),
            'elu': nn.ELU(),
            'tanh': nn.Tanh(),
            'sigmoid': nn.Sigmoid()
        }
        self.activation_func = act_map.get(activation, nn.ReLU())  # default to ReLU if not found

        # Create hidden layers
        prev_dim = input_dim
        for _ in range(num_layers):
            linear_layer = nn.Linear(prev_dim, num_neurons)
            self.layers.append(linear_layer)
            self.dropouts.append(nn.Dropout(dropout_rate))
            prev_dim = num_neurons

        # Output layer (1 neuron for regression)
        self.out = nn.Linear(prev_dim, 1)
        
        # Initialize weights
        self.apply(lambda m: self.init_weights(m, kernel_init))

    def init_weights(self, m, kernel_init):
        """
        Initialize the weights with either uniform or normal distribution
        """
        if isinstance(m, nn.Linear):
            if kernel_init == 'uniform':
                nn.init.uniform_(m.weight, a=-0.05, b=0.05)
            elif kernel_init == 'normal':
                nn.init.normal_(m.weight, mean=0.0, std=0.05)
            # bias is often just zero-initialized
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.0)
        
    def forward(self, x):
        for linear, dropout in zip(self.layers, self.dropouts):
            x = linear(x)
            x = self.activation_func(x)
            x = dropout(x)
        x = self.out(x)
        return x

# ==== GENETIC ALGORITHM EVALUATION FUNCTION ====
current_radius = None  # global var to store the radius in each iteration

def evaluate_nn(individual):
    global current_radius
    radius = current_radius

    # --------------------------------------------------
    # 1) Parse hyperparameters
    # --------------------------------------------------
    if EXPERIMENT == 4:
        # [alpha, activation, batch_size, dropout_rate, epochs, kernel_init, num_layers, num_neurons, optimizer]
        alpha = individual[0]
        activation = individual[1]
        batch_size = individual[2]
        dropout_rate = individual[3]
        epochs = individual[4]
        kernel_init = individual[5]
        num_layers = individual[6]
        num_neurons = individual[7]
        optimizer_name = individual[8]
        beta = 1 - alpha
    else:
        # [activation, batch_size, dropout_rate, epochs, kernel_init, num_layers, num_neurons, optimizer]
        alpha, beta = 0.5, 0.5
        activation = individual[0]
        batch_size = individual[1]
        dropout_rate = individual[2]
        epochs = individual[3]
        kernel_init = individual[4]
        num_layers = individual[5]
        num_neurons = individual[6]
        optimizer_name = individual[7]

    # --------------------------------------------------
    # 2) Aggregate features
    # --------------------------------------------------
    aggregated_data = aggregate_features(
        edge_attributes, distance_matrix, FEATURE_COLUMNS, radius, alpha, beta
    )

    # --------------------------------------------------
    # 3) Build X matrix based on experiment
    # --------------------------------------------------
    if EXPERIMENT == 3:
        X = aggregated_data[[f"pca_{col}" for col in FEATURE_COLUMNS]]
    elif EXPERIMENT == 2:
        X = pd.DataFrame()
        for col in FEATURE_COLUMNS:
            X[col] = (
                aggregated_data[f"original_{col}"] + 
                aggregated_data[f"avg_{col}_neighbors"]
            ) / 2
    elif EXPERIMENT == 4:
        X = aggregated_data[
            [f"original_{col}" for col in FEATURE_COLUMNS] +
            [f"weighted_avg_{col}_neighbors" for col in FEATURE_COLUMNS]
        ]
    else:
        X = aggregated_data[
            [f"original_{col}" for col in FEATURE_COLUMNS] +
            [f"avg_{col}_neighbors" for col in FEATURE_COLUMNS]
        ]
    
    # Fill missing
    X = X.fillna(X.median())

    # --------------------------------------------------
    # 4) Prepare train/test data
    # --------------------------------------------------
    y = np.array([edge_attrs[TARGET_COLUMN] for edge_attrs in edge_attributes])
    X_train, X_test, y_train, y_test = train_test_split(
        X.values, y, test_size=0.2, random_state=42
    )

    # Convert to torch tensors
    X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1).to(device)
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1).to(device)

    input_dim = X_train_t.shape[1]

    # --------------------------------------------------
    # 5) Build the PyTorch model
    # --------------------------------------------------
    model = Net(
        input_dim=input_dim,
        num_layers=num_layers,
        num_neurons=num_neurons,
        activation=activation,
        dropout_rate=dropout_rate,
        kernel_init=kernel_init
    ).to(device)

    # Define loss & optimizer
    criterion = nn.MSELoss()
    if optimizer_name == 'adam':
        optimizer = optim.Adam(model.parameters(), lr=0.001)
    elif optimizer_name == 'adagrad':
        optimizer = optim.Adagrad(model.parameters(), lr=0.001)
    elif optimizer_name == 'rmsprop':
        optimizer = optim.RMSprop(model.parameters(), lr=0.001)
    elif optimizer_name == 'sgd':
        optimizer = optim.SGD(model.parameters(), lr=0.001)
    else:
        # default to Adam if unknown
        optimizer = optim.Adam(model.parameters(), lr=0.001)

    # --------------------------------------------------
    # 6) Train the model (mini-batch loop)
    # --------------------------------------------------
    model.train()
    for epoch in range(epochs):
        # Shuffle training data each epoch if desired
        perm = torch.randperm(X_train_t.size(0))
        X_train_t = X_train_t[perm]
        y_train_t = y_train_t[perm]

        for i in range(0, X_train_t.size(0), batch_size):
            x_batch = X_train_t[i:i+batch_size]
            y_batch = y_train_t[i:i+batch_size]

            optimizer.zero_grad()
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

    # --------------------------------------------------
    # 7) Evaluate with R² on the test set
    # --------------------------------------------------
    model.eval()
    with torch.no_grad():
        y_pred_t = model(X_test_t).cpu().numpy().flatten()
    y_test_cpu = y_test_t.cpu().numpy().flatten()
    
    r2 = r2_score(y_test_cpu, y_pred_t)
    return (r2,)

# ==== GA Setup (DEAP) ====
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

# -------------------------------------------------------------
# Register hyperparameter generators
# -------------------------------------------------------------
ACTIVATIONS = ['relu', 'elu', 'tanh', 'sigmoid']
BATCH_SIZES = [10, 50, 100]
DROPOUT_RATES = [0.0, 0.1, 0.2, 0.3]
EPOCHS_LIST = [50, 100, 150, 200, 250]
KERNEL_INITS = ['uniform', 'normal']
NUM_LAYERS = [1, 2, 3, 4]
OPTIMIZERS = ['adam', 'adagrad', 'rmsprop', 'sgd']

def random_neurons():
    return random.randint(50, 500)

ALPHAS = [0, 0.25, 0.5, 0.75, 1]  # For experiment 4

# If you want to adaptively remove bad parameters, you'd create a dynamic approach
# rather than a static random.choice. But let's keep it simple for now.
toolbox.register("activation", random.choice, ACTIVATIONS)
toolbox.register("batch_size", random.choice, BATCH_SIZES)
toolbox.register("dropout_rate", random.choice, DROPOUT_RATES)
toolbox.register("epochs_hp", random.choice, EPOCHS_LIST)
toolbox.register("kernel_init", random.choice, KERNEL_INITS)
toolbox.register("num_layers", random.choice, NUM_LAYERS)
toolbox.register("num_neurons", random_neurons)
toolbox.register("optimizer", random.choice, OPTIMIZERS)
if EXPERIMENT == 4:
    toolbox.register("alpha", random.choice, ALPHAS)

def create_individual():
    """
    Return a list of hyperparams, optionally including alpha if EXPERIMENT=4.
    """
    if EXPERIMENT == 4:
        return [
            toolbox.alpha(),
            toolbox.activation(),
            toolbox.batch_size(),
            toolbox.dropout_rate(),
            toolbox.epochs_hp(),
            toolbox.kernel_init(),
            toolbox.num_layers(),
            toolbox.num_neurons(),
            toolbox.optimizer()
        ]
    else:
        return [
            toolbox.activation(),
            toolbox.batch_size(),
            toolbox.dropout_rate(),
            toolbox.epochs_hp(),
            toolbox.kernel_init(),
            toolbox.num_layers(),
            toolbox.num_neurons(),
            toolbox.optimizer()
        ]

toolbox.register("individual", tools.initIterate, creator.Individual, create_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", evaluate_nn)
toolbox.register("mate", tools.cxTwoPoint)

def mutate_individual(ind):
    """
    A simple re-initialization approach. If you want more fine-grained or
    adaptive mutation, you could do partial changes or remove poor params.
    """
    if EXPERIMENT == 4:
        ind[:] = [
            toolbox.alpha(),
            toolbox.activation(),
            toolbox.batch_size(),
            toolbox.dropout_rate(),
            toolbox.epochs_hp(),
            toolbox.kernel_init(),
            toolbox.num_layers(),
            toolbox.num_neurons(),
            toolbox.optimizer()
        ]
    else:
        ind[:] = [
            toolbox.activation(),
            toolbox.batch_size(),
            toolbox.dropout_rate(),
            toolbox.epochs_hp(),
            toolbox.kernel_init(),
            toolbox.num_layers(),
            toolbox.num_neurons(),
            toolbox.optimizer()
        ]
    return (ind,)

toolbox.register("mutate", mutate_individual)
toolbox.register("select", tools.selTournament, tournsize=3)

# ==== GA PARAMETERS ====
POP_SIZE = 20  # Fewer than typical RF because NNs are expensive to train
GENS = 50     # Also fewer for demonstration
CXPB, MUTPB = 0.8, 0.5

# Radii list
radii_list = [100, 500, 1000, 3000, 4000, 5000, 8000]

results_per_radius = {}

print("\nStarting GA Optimization for each radius...")

for radius in radii_list:
    print(f"\nRunning GA for radius = {radius}")
    current_radius = radius
    start_time = time.time()

    # Initialize population
    pop = toolbox.population(n=POP_SIZE)
    
    # Early stopping parameters
    improvement_threshold = 0.001
    max_no_improve = 5
    no_improve_count = 0

    best_overall_r2 = -np.inf
    best_overall_individual = None

    for gen in range(GENS):
        print(f"\n🔄 Generation {gen + 1}/{GENS} for radius {radius}")
        fitnesses = list(map(toolbox.evaluate, tqdm(pop, desc="Evaluating individuals", leave=False)))
        r2_scores = []
        for ind, fit in zip(pop, fitnesses):
            ind.fitness.values = fit
            r2_scores.append(fit[0])
        print(f" R² Scores: {r2_scores}")

        gen_best_r2 = max(r2_scores)
        gen_best_index = r2_scores.index(gen_best_r2)
        gen_best_individual = pop[gen_best_index]

        if gen_best_r2 - best_overall_r2 > improvement_threshold:
            best_overall_r2 = gen_best_r2
            best_overall_individual = copy.deepcopy(gen_best_individual)
            no_improve_count = 0
        else:
            no_improve_count += 1

        print(f"🔥 Best R² in Gen {gen + 1}: {gen_best_r2:.4f}")
        if no_improve_count >= max_no_improve:
            print(f"🚫 No significant improvement in the last {max_no_improve} generations. Early stopping...")
            break

        # Selection
        offspring = toolbox.select(pop, len(pop))
        offspring = [copy.deepcopy(ind) for ind in offspring]
        
        # Crossover
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CXPB:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values
        
        # Mutation
        for mutant in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(mutant)
                del mutant.fitness.values
        
        # Re-evaluate newly mutated/crossover individuals
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = list(map(toolbox.evaluate, tqdm(invalid_ind, desc="Evaluating offspring", leave=False)))
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit
        
        pop[:] = offspring

    elapsed = time.time() - start_time
    print(f"\nBest hyperparameters for radius {radius}: {best_overall_individual}")
    print(f"🔥 Best R² Score for radius {radius}: {best_overall_r2:.4f}")
    print(f"⏱ Elapsed time for radius {radius}: {elapsed:.2f} seconds")

    results_per_radius[radius] = {
        "best_hyperparameters": best_overall_individual,
        "best_r2": best_overall_r2,
        "time": elapsed
    }

# ==== PLOTTING RESULTS ====
radii = sorted(results_per_radius.keys())
r2_scores = [results_per_radius[r]["best_r2"] for r in radii]

plt.figure(figsize=(10, 6))
plt.plot(radii, r2_scores, marker='o', linestyle='-', color='b')
plt.xlabel("Radius")
plt.ylabel("Best R² Score")
plt.title("Best R² Score vs. Radius (PyTorch NN)")
plt.grid(True)
plt.show()

# ==== DISPLAY BEST HYPERPARAMETERS PER RADIUS ====
print("\n📌 Summary of Best Hyperparameters per Radius:")
for radius in radii:
    best_params = results_per_radius[radius]["best_hyperparameters"]
    best_r2 = results_per_radius[radius]["best_r2"]
    print(f"Radius: {radius} | Best R²: {best_r2:.4f} | Hyperparameters: {best_params}")

### Linear Regression Apeldoorn Satellite

In [ ]:
## SLX EXTENDED TO A LIMITED NEIGHBOUR SELECTION RADIUS

import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import defaultdict
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

# Global Variables
OUTPUT_DIR = "./precomputed_matrices"
FEATURE_COLUMNS = [
    'Bld_densit', 'Veg_densit', 'HW_Ratio', 'Width', 'Pred_Facad',
    'Pred_LandU', 'Mean_Popul', 'Mean_Bld_E', 'Max_Bld_He',
    'Water_dens', 'Bike_pct_a', 'Vehicle_pc'
]
TARGET_COLUMN = 'Delta_T'

# The list of radii (in meters) for sensitivity analysis.
RADIUS_VALUES = [100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600,
                 1100, 1600, 2100, 2600, 3100, 3600, 4100, 4600, 5100,
                 5600, 6100, 6600, 7100, 7500, 7600, 7700, 7800, 7900,
                 8000, 8100, 8200, 8300, 8400, 8500, 8600, 8700]

def compute_slx_features(edge_attributes, distance_matrix, feature_columns, radius):
    """
    Compute spatially lagged features using only neighbors within a given radius.
    For each segment, the spatial lag for each feature is computed as the weighted
    sum of the feature values of neighbors (where weights are 1/distance normalized
    to sum to one).
    
    Returns:
        A DataFrame with columns for the original features (prefixed with "original_")
        and the spatial lag features (prefixed with "lag_").
    """
    results = defaultdict(list)
    n_segments = len(edge_attributes)
    
    # Loop over each segment
    for i in range(n_segments):
        seg_attr = edge_attributes[i]
        # Save original features
        for col in feature_columns:
            results[f"original_{col}"].append(seg_attr.get(col, np.nan))
        
        # Find neighbor indices with distance <= radius (excluding self)
        neighbor_indices = np.where(distance_matrix[i] <= radius)[0]
        neighbor_indices = neighbor_indices[neighbor_indices != i]
        
        # Compute weighted average for each feature from the neighbors
        lag_features = {}
        if len(neighbor_indices) > 0:
            distances = distance_matrix[i, neighbor_indices]
            # Compute inverse-distance weights
            inv_distances = 1.0 / distances
            normalized_weights = inv_distances / inv_distances.sum()
            for col in feature_columns:
                neighbor_values = np.array([edge_attributes[j].get(col, np.nan) for j in neighbor_indices])
                lag_features[col] = np.nansum(normalized_weights * neighbor_values)
        else:
            # No neighbors: set lagged value to 0 (or consider np.nan if preferred)
            for col in feature_columns:
                lag_features[col] = 0.0
        
        # Save the lag features
        for col in feature_columns:
            results[f"lag_{col}"].append(lag_features[col])
    
    return pd.DataFrame(results)

def train_slx_model(X, y):
    """
    Splits the data, fits a linear regression (SLX) model, and returns the R² score.
    """
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return r2_score(y_test, y_pred)

if __name__ == "__main__":
    # Load precomputed data
    print("Loading distance matrix and edge attributes...")
    distance_matrix = np.load(f"{OUTPUT_DIR}/distance_matrix.npy")
    with open(f"{OUTPUT_DIR}/edge_attributes.pkl", "rb") as f:
        edge_attributes = pickle.load(f)
    
    # Prepare the target variable (assumes one value per segment)
    y = np.array([edge.get(TARGET_COLUMN, np.nan) for edge in edge_attributes])
    
    r2_scores = []

    print("Starting sensitivity analysis for different radii...")
    # Loop over each radius and compute the corresponding R² score
    for radius in tqdm(RADIUS_VALUES, desc="Radius Sensitivity"):
        # Compute spatial lag features with the current radius
        features_df = compute_slx_features(edge_attributes, distance_matrix, FEATURE_COLUMNS, radius)
        
        # Create the design matrix (using both original and lagged features)
        cols_original = [f"original_{col}" for col in FEATURE_COLUMNS]
        cols_lag = [f"lag_{col}" for col in FEATURE_COLUMNS]
        X = features_df[cols_original + cols_lag]
        
        # Ensure we use only rows with valid (non-missing) data
        valid_idx = X.notnull().all(axis=1) & ~pd.isnull(y)
        X_valid = X[valid_idx]
        y_valid = y[valid_idx]
        
        # If no valid data, set R² to NaN; otherwise, train and compute R²
        if X_valid.shape[0] == 0:
            r2 = np.nan
        else:
            r2 = train_slx_model(X_valid, y_valid)
        r2_scores.append(r2)
    
    # Create a DataFrame to summarize the sensitivity results
    sensitivity_df = pd.DataFrame({
        "Radius": RADIUS_VALUES,
        "R2": r2_scores
    })
    
    # Plot R² against Radius
    plt.figure(figsize=(10, 6))
    plt.plot(sensitivity_df["Radius"], sensitivity_df["R2"], marker="o", linestyle="-")
    plt.xlabel("Radius (meters)")
    plt.ylabel("R²")
    plt.title("SLX Model Sensitivity Analysis: R² vs Radius")
    plt.grid(True)
    plt.show()
    
    # Print the sensitivity analysis table
    print(sensitivity_df)

In [ ]:
## SIMPLE LINEAR REGRESSION (BASELINE)

import pandas as pd
import numpy as np
import pickle
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Global Variables
OUTPUT_DIR = "./precomputed_matrices"
FEATURE_COLUMNS = [
    'Bld_densit', 'Veg_densit', 'HW_Ratio', 'Width', 'Pred_Facad',
    'Pred_LandU', 'Mean_Popul', 'Mean_Bld_E', 'Max_Bld_He',
    'Water_dens', 'Bike_pct_a', 'Vehicle_pc'
]
TARGET_COLUMN = 'Delta_T'

def load_edge_attributes(output_dir):
    """Load edge attributes from a pickle file."""
    with open(f"{output_dir}/edge_attributes.pkl", "rb") as f:
        edge_attributes = pickle.load(f)
    return edge_attributes

def prepare_data(edge_attributes, feature_columns, target_column):
    """
    Create a DataFrame using only the original features (no spatial lag)
    and the target variable.
    """
    data = []
    for edge in edge_attributes:
        row = {col: edge.get(col, np.nan) for col in feature_columns}
        row[target_column] = edge.get(target_column, np.nan)
        data.append(row)
    
    df = pd.DataFrame(data)
    # Remove any rows with missing values
    df = df.dropna()
    X = df[feature_columns]
    y = df[target_column]
    return X, y

def train_and_evaluate_lr(X, y):
    """Train and evaluate a simple linear regression model using only original features."""
    print("Splitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print("Training Linear Regression model on original features...")
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    
    print("Predicting on test data...")
    y_pred = lr.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    
    print(f"\nSimple Linear Regression Results:\nR²: {r2:.4f}\nMSE: {mse:.4f}\n")
    
    # Display the model coefficients
    coef_df = pd.DataFrame({
        'Feature': X.columns,
        'Coefficient': lr.coef_
    }).sort_values(by='Coefficient', key=lambda x: np.abs(x), ascending=False)
    
    print("Model Coefficients:")
    print(coef_df.to_string(index=False))
    
    # Plot coefficients for visualization
    coef_df.plot(kind='bar', x='Feature', y='Coefficient', legend=False,
                 figsize=(10, 6), title="Linear Regression Coefficients")
    plt.ylabel("Coefficient Value")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    return r2

if __name__ == "__main__":
    print("Loading edge attributes...")
    edge_attributes = load_edge_attributes(OUTPUT_DIR)
    
    print("Preparing data (original features only)...")
    X, y = prepare_data(edge_attributes, FEATURE_COLUMNS, TARGET_COLUMN)
    
    print("Training and evaluating the simple linear regression model...")
    train_and_evaluate_lr(X, y)

### Transferability Tests

In [ ]:
## ROTTERDAM RADIUS-BASED SELECTION, SEPARATE MEANS AGGREGATION

import time
import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
from deap import base, creator, tools, algorithms
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import pickle
from collections import defaultdict

# ---------------- USER CONFIGURATION -----------------
# Choose experiment: 1 for simple (unweighted) neighbor aggregation,
EXPERIMENT = 1

# Rotterdam-specific settings:
OUTPUT_DIR = "./precomputed_matrices"
FEATURE_COLUMNS = [
    'Bld_densit', 'Veg_densit', 'HW_Ratio', 'Width',
    'Pred_LandU', 'Mean_Popul', 'Mean_Bld_E', 'Max_Bld_He',
    'Water_dens'
]
TARGET_COLUMN = 'Delta_T'

# GA parameters for the environment (same as your first script)
POP_SIZE = 20
GENS = 10
CXPB, MUTPB = 0.8, 0.5
improvement_threshold = 0.01
max_no_improve = 3

# Rotterdam-specific radii (in meters)
radii_list = [100, 200, 300, 400, 500, 750, 1000, 1250, 1500, 1750, 2000, 2500, 3000, 3500, 4000, 5000, 7000, 9000, 11000, 13000, 15000, 20000, 30000, 43000]


# ---------------- HELPER FUNCTIONS -----------------

def normalize_matrix(matrix):
    """Normalize a matrix to the range [0,1]."""
    return (matrix - np.min(matrix)) / (np.max(matrix) - np.min(matrix))

def compute_adjacency_index(distance_matrix):
    """
    Compute the adjacency index using only the distance matrix.
    (Equivalent to setting alpha=0, i.e., using only the distance component.)
    """
    distance_normalized = normalize_matrix(distance_matrix)
    adjacency_index = 1 - distance_normalized
    return adjacency_index

def aggregate_features_with_adjacency(edge_attributes, adjacency_index, distance_matrix, feature_columns, radius):
    """
    For experiment 4:
    Aggregate features using a weighted average based on the adjacency index.
    Neighbors are selected if their distance is <= radius.
    """
    aggregated_features = defaultdict(list)
    for segment_index, segment_attrs in enumerate(edge_attributes):
        segment_features = {col: segment_attrs.get(col, np.nan) for col in feature_columns}
        neighbor_indices = np.where(distance_matrix[segment_index] <= radius)[0]
        neighbors = [n for n in neighbor_indices if n != segment_index]

        if neighbors:
            neighbor_features = np.array([
                [edge_attributes[n].get(col, np.nan) for col in feature_columns]
                for n in neighbors
            ])
            weights = adjacency_index[segment_index, neighbors]
            weight_sum = np.sum(weights)
            if weight_sum != 0:
                weights = weights / weight_sum
            else:
                weights = np.ones_like(weights) / len(weights)
            weighted_avg = {
                f"weighted_avg_{col}_neighbors": np.average(neighbor_features[:, i], weights=weights)
                for i, col in enumerate(feature_columns)
            }
        else:
            weighted_avg = {f"weighted_avg_{col}_neighbors": np.nan for col in feature_columns}

        for col in feature_columns:
            aggregated_features[f"original_{col}"].append(segment_features[col])
            aggregated_features[f"weighted_avg_{col}_neighbors"].append(weighted_avg[f"weighted_avg_{col}_neighbors"])

    return pd.DataFrame(aggregated_features)

def aggregate_features_simple(edge_attributes, distance_matrix, feature_columns, radius):
    """
    For experiment 1:
    Aggregate features by computing a simple (unweighted) average of neighbor values.
    """
    aggregated_features = defaultdict(list)
    for segment_index, segment_attrs in enumerate(edge_attributes):
        segment_features = {col: segment_attrs.get(col, np.nan) for col in feature_columns}
        neighbor_indices = np.where(distance_matrix[segment_index] <= radius)[0]
        neighbors = [n for n in neighbor_indices if n != segment_index]

        if neighbors:
            neighbor_features = np.array([
                [edge_attributes[n].get(col, np.nan) for col in feature_columns]
                for n in neighbors
            ])
            avg_neighbors = np.nanmean(neighbor_features, axis=0)
        else:
            avg_neighbors = np.array([segment_features[col] for col in feature_columns])

        for i, col in enumerate(feature_columns):
            aggregated_features[f"original_{col}"].append(segment_features[col])
            aggregated_features[f"avg_{col}_neighbors"].append(avg_neighbors[i])
    return pd.DataFrame(aggregated_features)

def train_rf(X, y, rf_params):
    """Train the Random Forest and return the trained model along with performance metrics."""
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    rf = RandomForestRegressor(**rf_params)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    print(f"Random Forest Performance: R² = {r2:.4f}, MSE = {mse:.4f}")
    return rf

# ---------------- GA EVALUATION FUNCTION -----------------
# Global variable to hold the current radius for the GA run.
current_radius = None

def evaluate_rf(individual):
    global current_radius
    radius = current_radius
    # Unpack hyperparameters from individual.
    n_estimators = individual[0]
    max_features = individual[1]
    max_depth = individual[2]
    min_samples_split = individual[3]
    bootstrap = individual[4]

    # Aggregate features based on chosen experiment.
    if EXPERIMENT == 4:
        aggregated_data = aggregate_features_with_adjacency(
            edge_attributes, adjacency_index, distance_matrix, FEATURE_COLUMNS, radius
        )
        feature_set = [f"original_{col}" for col in FEATURE_COLUMNS] + \
                      [f"weighted_avg_{col}_neighbors" for col in FEATURE_COLUMNS]
    else:  # EXPERIMENT == 1
        aggregated_data = aggregate_features_simple(
            edge_attributes, distance_matrix, FEATURE_COLUMNS, radius
        )
        feature_set = [f"original_{col}" for col in FEATURE_COLUMNS] + \
                      [f"avg_{col}_neighbors" for col in FEATURE_COLUMNS]

    X = aggregated_data[feature_set].fillna(aggregated_data[feature_set].median())
    y = np.array([edge_attrs[TARGET_COLUMN] for edge_attrs in edge_attributes])
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    rf = RandomForestRegressor(
        n_estimators=n_estimators,
        max_features=max_features,
        max_depth=max_depth,
        min_samples_split=int(min_samples_split),
        bootstrap=bool(bootstrap),
        random_state=42
    )
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    return (r2_score(y_test, y_pred),)

# ---------------- GA SETUP -----------------
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("n_estimators", random.choice, [10, 30, 50, 100, 150, 200])
toolbox.register("max_features", random.choice, ['sqrt', 'log2'])
toolbox.register("max_depth", random.choice, [10, 20, 30, 50, 100])
toolbox.register("min_samples_split", random.choice, [2, 4, 6])
toolbox.register("bootstrap", random.choice, [True, False])

def create_individual():
    return [
        toolbox.n_estimators(),
        toolbox.max_features(),
        toolbox.max_depth(),
        toolbox.min_samples_split(),
        toolbox.bootstrap()
    ]

toolbox.register("individual", tools.initIterate, creator.Individual, create_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", evaluate_rf)
toolbox.register("mate", tools.cxTwoPoint)

def mutate_individual(ind):
    ind[:] = create_individual()
    return (ind,)

toolbox.register("mutate", mutate_individual)
toolbox.register("select", tools.selTournament, tournsize=3)

# ---------------- MAIN EXECUTION -----------------
if __name__ == "__main__":
    print("Loading matrices and dataset...")

    # Load the precomputed distance matrix and edge attributes for Rotterdam.
    distance_matrix = np.load(f"{OUTPUT_DIR}/distance_matrix_Rotterdam.npy")
    with open(f"{OUTPUT_DIR}/edge_attributes_Rotterdam.pkl", "rb") as f:
        edge_attributes = pickle.load(f)

    # Compute the adjacency index from the distance matrix.
    adjacency_index = compute_adjacency_index(distance_matrix)
    print("Adjacency index computed.")

    # ---------------- GA OPTIMIZATION OVER RADII -----------------
    results_per_radius = {}

    print("\nStarting GA Optimization for Rotterdam data...")
    for radius in radii_list:
        print(f"\nRunning GA for radius = {radius} meters")
        current_radius = radius  # update global radius
        start_time = time.time()

        pop = toolbox.population(n=POP_SIZE)
        no_improve_count = 0
        best_overall_r2 = -np.inf
        best_overall_individual = None

        for gen in range(GENS):
            print(f"\n🔄 Generation {gen+1}/{GENS} for radius {radius}")
            fitnesses = list(map(toolbox.evaluate, tqdm(pop, desc="Evaluating individuals", leave=False)))
            for ind, fit in zip(pop, fitnesses):
                ind.fitness.values = fit
            r2_scores = [ind.fitness.values[0] for ind in pop]
            gen_best_r2 = max(r2_scores)
            gen_best_individual = pop[r2_scores.index(gen_best_r2)]

            if gen_best_r2 - best_overall_r2 > improvement_threshold:
                best_overall_r2 = gen_best_r2
                best_overall_individual = toolbox.clone(gen_best_individual)
                no_improve_count = 0
            else:
                no_improve_count += 1

            print(f"🔥 Best R² in generation {gen+1}: {gen_best_r2:.4f}")
            if no_improve_count >= max_no_improve:
                print(f"🚫 No significant improvement in the last {max_no_improve} generations. Early stopping...")
                break

            offspring = toolbox.select(pop, len(pop))
            offspring = list(map(toolbox.clone, offspring))
            for child1, child2 in zip(offspring[::2], offspring[1::2]):
                if random.random() < CXPB:
                    toolbox.mate(child1, child2)
                    del child1.fitness.values, child2.fitness.values
            for mutant in offspring:
                if random.random() < MUTPB:
                    toolbox.mutate(mutant)
                    del mutant.fitness.values
            invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
            fitnesses = list(map(toolbox.evaluate, tqdm(invalid_ind, desc="Evaluating offspring", leave=False)))
            for ind, fit in zip(invalid_ind, fitnesses):
                ind.fitness.values = fit
            pop[:] = offspring

        elapsed = time.time() - start_time
        print(f"\nBest hyperparameters for radius {radius}: {best_overall_individual}")
        print(f"🔥 Best R² Score for radius {radius}: {best_overall_r2:.4f}")
        print(f"⏱ Elapsed time for radius {radius}: {elapsed:.2f} seconds")
        results_per_radius[radius] = {
            "best_hyperparameters": best_overall_individual,
            "best_r2": best_overall_r2,
            "time": elapsed
        }

    # ---------------- PLOTTING AND SUMMARY -----------------
    radii_sorted = sorted(results_per_radius.keys())
    r2_scores = [results_per_radius[r]["best_r2"] for r in radii_sorted]

    plt.figure(figsize=(10, 6))
    plt.plot(radii_sorted, r2_scores, marker='o', linestyle='-')
    plt.xlabel("Radius (meters)")
    plt.ylabel("Best R² Score")
    plt.title("Best R² Score vs. Radius for Rotterdam Data")
    plt.grid(True)
    plt.show()

    print("\n📌 Summary of Best Hyperparameters per Radius:")
    for radius in radii_sorted:
        best_params = results_per_radius[radius]["best_hyperparameters"]
        best_r2 = results_per_radius[radius]["best_r2"]
        print(f"Radius: {radius} | Best R²: {best_r2:.4f} | Hyperparameters: {best_params}")


In [ ]:
## ROTTERDAM RADIUS-BASED SELECTION, WEIGHTED MEANSS AGGREGATION

import time
import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
from deap import base, creator, tools, algorithms
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import pickle
from collections import defaultdict

# ---------------- USER CONFIGURATION -----------------
# We are using experiment 4 (weighted neighbor aggregation using local distance normalization).
EXPERIMENT = 4

# Rotterdam-specific settings:
OUTPUT_DIR = "./precomputed_matrices"
FEATURE_COLUMNS = [
    'Bld_densit', 'Veg_densit', 'HW_Ratio', 'Width',
    'Pred_LandU', 'Mean_Popul', 'Mean_Bld_E', 'Max_Bld_He',
    'Water_dens'
]
TARGET_COLUMN = 'Delta_T'

# GA parameters
POP_SIZE = 50
GENS = 20
CXPB, MUTPB = 0.8, 0.5
improvement_threshold = 0.01
max_no_improve = 3

# Rotterdam-specific radii (in meters)
radii_list = [0, 100, 200, 300, 400, 500, 750, 1000, 1250, 1500, 1750, 2000, 2500, 3000, 3500, 4000, 5000, 7000, 9000, 11000, 13000, 15000, 20000, 30000, 43000]

# ---------------- HELPER FUNCTIONS -----------------

def normalize_matrix(matrix):
    """Normalize a matrix to the range [0,1]."""
    return (matrix - np.min(matrix)) / (np.max(matrix) - np.min(matrix))

def aggregate_features_with_local_weighting(edge_attributes, distance_matrix, feature_columns, radius):
    """
    For experiment 4:
    Aggregate features using a locally weighted average based solely on distance.
    Neighbors are selected if their distance is <= radius.
    The weight for each neighbor is computed by local normalization of the distances.
    """
    aggregated_features = defaultdict(list)
    for segment_index, segment_attrs in enumerate(edge_attributes):
        # Get the feature values for the current segment.
        segment_features = {col: segment_attrs.get(col, np.nan) for col in feature_columns}
        # Select neighbor indices within the given radius (excluding self).
        neighbor_indices = np.where(distance_matrix[segment_index] <= radius)[0]
        neighbors = [n for n in neighbor_indices if n != segment_index]

        if neighbors:
            neighbor_features = np.array([
                [edge_attributes[n].get(col, np.nan) for col in feature_columns]
                for n in neighbors
            ])
            # Use local distance normalization:
            local_distance = distance_matrix[segment_index, neighbors]
            local_max_distance = np.max(local_distance)
            # Avoid division by zero: if local_max_distance is zero, assign equal weights.
            if local_max_distance > 0:
                norm_distance_local = 1 - local_distance / local_max_distance
            else:
                norm_distance_local = np.ones_like(local_distance)
            # Since we have no shortest-path component, weights are solely based on distance.
            weights = norm_distance_local
            weights_sum = np.sum(weights)
            if weights_sum != 0:
                weights = weights / weights_sum
            else:
                weights = np.ones_like(weights) / len(weights)
            weighted_avg = [np.average(neighbor_features[:, i], weights=weights)
                            for i in range(len(feature_columns))]
        else:
            weighted_avg = [np.nan for _ in feature_columns]

        # Save original and aggregated features.
        for col in feature_columns:
            aggregated_features[f"original_{col}"].append(segment_features[col])
        for i, col in enumerate(feature_columns):
            aggregated_features[f"weighted_avg_{col}_neighbors"].append(weighted_avg[i])
    return pd.DataFrame(aggregated_features)

# ---------------- GA EVALUATION FUNCTION -----------------
# Global variable to hold the current radius for the GA run.
current_radius = None

def evaluate_rf(individual):
    global current_radius
    radius = current_radius
    # For experiment 4, the individual contains:
    # [n_estimators, max_features, max_depth, min_samples_split, bootstrap]
    n_estimators = individual[0]
    max_features = individual[1]
    max_depth = individual[2]
    min_samples_split = individual[3]
    bootstrap = individual[4]
    
    # Aggregate features using the locally weighted aggregation.
    aggregated_data = aggregate_features_with_local_weighting(
        edge_attributes, distance_matrix, FEATURE_COLUMNS, radius
    )
    # For experiment 4, the feature set includes the original and weighted neighbor features.
    feature_set = [f"original_{col}" for col in FEATURE_COLUMNS] + \
                  [f"weighted_avg_{col}_neighbors" for col in FEATURE_COLUMNS]
    X = aggregated_data[feature_set].fillna(aggregated_data[feature_set].median())
    y = np.array([edge_attrs[TARGET_COLUMN] for edge_attrs in edge_attributes])
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    rf = RandomForestRegressor(
        n_estimators=n_estimators,
        max_features=max_features,
        max_depth=max_depth,
        min_samples_split=int(min_samples_split),
        bootstrap=bool(bootstrap),
        random_state=42
    )
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    return (r2_score(y_test, y_pred),)

# ---------------- GA SETUP -----------------
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
# We use the following hyperparameter choices:
toolbox.register("n_estimators", random.choice, [10, 30, 50, 100, 150, 200])
toolbox.register("max_features", random.choice, ['sqrt', 'log2'])
toolbox.register("max_depth", random.choice, [10, 20, 30, 50, 100])
toolbox.register("min_samples_split", random.choice, [2, 4, 6])
toolbox.register("bootstrap", random.choice, [True, False])

def create_individual():
    return [
        toolbox.n_estimators(),
        toolbox.max_features(),
        toolbox.max_depth(),
        toolbox.min_samples_split(),
        toolbox.bootstrap()
    ]

toolbox.register("individual", tools.initIterate, creator.Individual, create_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", evaluate_rf)
toolbox.register("mate", tools.cxTwoPoint)

def mutate_individual(ind):
    ind[:] = create_individual()
    return (ind,)

toolbox.register("mutate", mutate_individual)
toolbox.register("select", tools.selTournament, tournsize=3)

# ---------------- MAIN EXECUTION -----------------
if __name__ == "__main__":
    print("Loading matrices and dataset for Rotterdam...")

    # Load the precomputed distance matrix and edge attributes for Rotterdam.
    distance_matrix = np.load(f"{OUTPUT_DIR}/distance_matrix_Rotterdam.npy")
    with open(f"{OUTPUT_DIR}/edge_attributes_Rotterdam.pkl", "rb") as f:
        edge_attributes = pickle.load(f)

    print("Data loaded. Starting GA optimization for Rotterdam (experiment 4, local weighting)...")

    results_per_radius = {}

    for radius in radii_list:
        print(f"\nRunning GA for radius = {radius} meters")
        current_radius = radius
        start_time = time.time()

        pop = toolbox.population(n=POP_SIZE)
        no_improve_count = 0
        best_overall_r2 = -np.inf
        best_overall_individual = None

        for gen in range(GENS):
            print(f"\n🔄 Generation {gen+1}/{GENS} for radius {radius}")
            fitnesses = list(map(toolbox.evaluate, tqdm(pop, desc="Evaluating individuals", leave=False)))
            for ind, fit in zip(pop, fitnesses):
                ind.fitness.values = fit
            r2_scores = [ind.fitness.values[0] for ind in pop]
            gen_best_r2 = max(r2_scores)
            gen_best_individual = pop[r2_scores.index(gen_best_r2)]

            if gen_best_r2 - best_overall_r2 > improvement_threshold:
                best_overall_r2 = gen_best_r2
                best_overall_individual = toolbox.clone(gen_best_individual)
                no_improve_count = 0
            else:
                no_improve_count += 1

            print(f"🔥 Best R² in generation {gen+1}: {gen_best_r2:.4f}")
            if no_improve_count >= max_no_improve:
                print(f"🚫 No significant improvement in the last {max_no_improve} generations. Early stopping...")
                break

            offspring = toolbox.select(pop, len(pop))
            offspring = list(map(toolbox.clone, offspring))
            for child1, child2 in zip(offspring[::2], offspring[1::2]):
                if random.random() < CXPB:
                    toolbox.mate(child1, child2)
                    del child1.fitness.values, child2.fitness.values
            for mutant in offspring:
                if random.random() < MUTPB:
                    toolbox.mutate(mutant)
                    del mutant.fitness.values
            invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
            fitnesses = list(map(toolbox.evaluate, tqdm(invalid_ind, desc="Evaluating offspring", leave=False)))
            for ind, fit in zip(invalid_ind, fitnesses):
                ind.fitness.values = fit
            pop[:] = offspring

        elapsed = time.time() - start_time
        print(f"\nBest hyperparameters for radius {radius}: {best_overall_individual}")
        print(f"🔥 Best R² Score for radius {radius}: {best_overall_r2:.4f}")
        print(f"⏱ Elapsed time for radius {radius}: {elapsed:.2f} seconds")
        results_per_radius[radius] = {
            "best_hyperparameters": best_overall_individual,
            "best_r2": best_overall_r2,
            "time": elapsed
        }

    # ---------------- PLOTTING AND SUMMARY -----------------
    radii_sorted = sorted(results_per_radius.keys())
    r2_scores = [results_per_radius[r]["best_r2"] for r in radii_sorted]

    plt.figure(figsize=(10, 6))
    plt.plot(radii_sorted, r2_scores, marker='o', linestyle='-')
    plt.xlabel("Radius (meters)")
    plt.ylabel("Best R² Score")
    plt.title("Best R² Score vs. Radius for Rotterdam Data (Local Weighting)")
    plt.grid(True)
    plt.show()

    print("\n📌 Summary of Best Hyperparameters per Radius:")
    for radius in radii_sorted:
        best_params = results_per_radius[radius]["best_hyperparameters"]
        best_r2 = results_per_radius[radius]["best_r2"]
        print(f"Radius: {radius} | Best R²: {best_r2:.4f} | Hyperparameters: {best_params}")

In [ ]:
## MONTREAL RADIUS-BASED SELECTION, SEPARATE MEANS AGGREGATION

import time
import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
from deap import base, creator, tools, algorithms
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import pickle
from collections import defaultdict

# ---------------- USER CONFIGURATION -----------------
# Choose experiment: 1 for simple (unweighted) neighbor aggregation,

EXPERIMENT = 1

# Montreal-specific settings:
OUTPUT_DIR = "./precomputed_matrices"
FEATURE_COLUMNS = [
    'Bld_densit', 'Veg_densit', 'HW_Ratio', 'Width',
    'Pred_LandU', 'Mean_Popul', 'Mean_Bld_E', 'Max_Bld_He',
    'Water_dens'
]
TARGET_COLUMN = 'Delta_T'

# GA parameters for the environment (same as your first script)
POP_SIZE = 20
GENS = 10
CXPB, MUTPB = 0.8, 0.5
improvement_threshold = 0.01
max_no_improve = 3

# Montreal-specific radii (in meters)
radii_list = [0, 100, 200, 300, 400, 500, 750, 1000, 1500, 2000,  3000, 4000, 5000, 7000, 10000, 13000, 14500]


# ---------------- HELPER FUNCTIONS -----------------

def normalize_matrix(matrix):
    """Normalize a matrix to the range [0,1]."""
    return (matrix - np.min(matrix)) / (np.max(matrix) - np.min(matrix))

def compute_adjacency_index(distance_matrix):
    """
    Compute the adjacency index using only the distance matrix.
    (Equivalent to setting alpha=0, i.e., using only the distance component.)
    """
    distance_normalized = normalize_matrix(distance_matrix)
    adjacency_index = 1 - distance_normalized
    return adjacency_index

def aggregate_features_with_adjacency(edge_attributes, adjacency_index, distance_matrix, feature_columns, radius):
    """
    For experiment 4:
    Aggregate features using a weighted average based on the adjacency index.
    Neighbors are selected if their distance is <= radius.
    """
    aggregated_features = defaultdict(list)
    for segment_index, segment_attrs in enumerate(edge_attributes):
        segment_features = {col: segment_attrs.get(col, np.nan) for col in feature_columns}
        neighbor_indices = np.where(distance_matrix[segment_index] <= radius)[0]
        neighbors = [n for n in neighbor_indices if n != segment_index]

        if neighbors:
            neighbor_features = np.array([
                [edge_attributes[n].get(col, np.nan) for col in feature_columns]
                for n in neighbors
            ])
            weights = adjacency_index[segment_index, neighbors]
            weight_sum = np.sum(weights)
            if weight_sum != 0:
                weights = weights / weight_sum
            else:
                weights = np.ones_like(weights) / len(weights)
            weighted_avg = {
                f"weighted_avg_{col}_neighbors": np.average(neighbor_features[:, i], weights=weights)
                for i, col in enumerate(feature_columns)
            }
        else:
            weighted_avg = {f"weighted_avg_{col}_neighbors": np.nan for col in feature_columns}

        for col in feature_columns:
            aggregated_features[f"original_{col}"].append(segment_features[col])
            aggregated_features[f"weighted_avg_{col}_neighbors"].append(weighted_avg[f"weighted_avg_{col}_neighbors"])

    return pd.DataFrame(aggregated_features)

def aggregate_features_simple(edge_attributes, distance_matrix, feature_columns, radius):
    """
    For experiment 1:
    Aggregate features by computing a simple (unweighted) average of neighbor values.
    """
    aggregated_features = defaultdict(list)
    for segment_index, segment_attrs in enumerate(edge_attributes):
        segment_features = {col: segment_attrs.get(col, np.nan) for col in feature_columns}
        neighbor_indices = np.where(distance_matrix[segment_index] <= radius)[0]
        neighbors = [n for n in neighbor_indices if n != segment_index]

        if neighbors:
            neighbor_features = np.array([
                [edge_attributes[n].get(col, np.nan) for col in feature_columns]
                for n in neighbors
            ])
            avg_neighbors = np.nanmean(neighbor_features, axis=0)
        else:
            avg_neighbors = np.array([segment_features[col] for col in feature_columns])

        for i, col in enumerate(feature_columns):
            aggregated_features[f"original_{col}"].append(segment_features[col])
            aggregated_features[f"avg_{col}_neighbors"].append(avg_neighbors[i])
    return pd.DataFrame(aggregated_features)

def train_rf(X, y, rf_params):
    """Train the Random Forest and return the trained model along with performance metrics."""
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    rf = RandomForestRegressor(**rf_params)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    print(f"Random Forest Performance: R² = {r2:.4f}, MSE = {mse:.4f}")
    return rf

# ---------------- GA EVALUATION FUNCTION -----------------
# Global variable to hold the current radius for the GA run.
current_radius = None

def evaluate_rf(individual):
    global current_radius
    radius = current_radius
    # Unpack hyperparameters from individual.
    n_estimators = individual[0]
    max_features = individual[1]
    max_depth = individual[2]
    min_samples_split = individual[3]
    bootstrap = individual[4]

    # Aggregate features based on chosen experiment.
    if EXPERIMENT == 4:
        aggregated_data = aggregate_features_with_adjacency(
            edge_attributes, adjacency_index, distance_matrix, FEATURE_COLUMNS, radius
        )
        feature_set = [f"original_{col}" for col in FEATURE_COLUMNS] + \
                      [f"weighted_avg_{col}_neighbors" for col in FEATURE_COLUMNS]
    else:  # EXPERIMENT == 1
        aggregated_data = aggregate_features_simple(
            edge_attributes, distance_matrix, FEATURE_COLUMNS, radius
        )
        feature_set = [f"original_{col}" for col in FEATURE_COLUMNS] + \
                      [f"avg_{col}_neighbors" for col in FEATURE_COLUMNS]

    X = aggregated_data[feature_set].fillna(aggregated_data[feature_set].median())
    y = np.array([edge_attrs[TARGET_COLUMN] for edge_attrs in edge_attributes])
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    rf = RandomForestRegressor(
        n_estimators=n_estimators,
        max_features=max_features,
        max_depth=max_depth,
        min_samples_split=int(min_samples_split),
        bootstrap=bool(bootstrap),
        random_state=42
    )
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    return (r2_score(y_test, y_pred),)

# ---------------- GA SETUP -----------------
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("n_estimators", random.choice, [10, 30, 50, 100, 150, 200])
toolbox.register("max_features", random.choice, ['sqrt', 'log2'])
toolbox.register("max_depth", random.choice, [10, 20, 30, 50, 100])
toolbox.register("min_samples_split", random.choice, [2, 4, 6])
toolbox.register("bootstrap", random.choice, [True, False])

def create_individual():
    return [
        toolbox.n_estimators(),
        toolbox.max_features(),
        toolbox.max_depth(),
        toolbox.min_samples_split(),
        toolbox.bootstrap()
    ]

toolbox.register("individual", tools.initIterate, creator.Individual, create_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", evaluate_rf)
toolbox.register("mate", tools.cxTwoPoint)

def mutate_individual(ind):
    ind[:] = create_individual()
    return (ind,)

toolbox.register("mutate", mutate_individual)
toolbox.register("select", tools.selTournament, tournsize=3)

# ---------------- MAIN EXECUTION -----------------
if __name__ == "__main__":
    print("Loading matrices and dataset...")

    # Load the precomputed distance matrix and edge attributes for Montreal.
    distance_matrix = np.load(f"{OUTPUT_DIR}/distance_matrix_Montreal.npy")
    with open(f"{OUTPUT_DIR}/edge_attributes_Montreal.pkl", "rb") as f:
        edge_attributes = pickle.load(f)

    # Compute the adjacency index from the distance matrix.
    adjacency_index = compute_adjacency_index(distance_matrix)
    print("Adjacency index computed.")

    # ---------------- GA OPTIMIZATION OVER RADII -----------------
    results_per_radius = {}

    print("\nStarting GA Optimization for Montreal data...")
    for radius in radii_list:
        print(f"\nRunning GA for radius = {radius} meters")
        current_radius = radius  # update global radius
        start_time = time.time()

        pop = toolbox.population(n=POP_SIZE)
        no_improve_count = 0
        best_overall_r2 = -np.inf
        best_overall_individual = None

        for gen in range(GENS):
            print(f"\n🔄 Generation {gen+1}/{GENS} for radius {radius}")
            fitnesses = list(map(toolbox.evaluate, tqdm(pop, desc="Evaluating individuals", leave=False)))
            for ind, fit in zip(pop, fitnesses):
                ind.fitness.values = fit
            r2_scores = [ind.fitness.values[0] for ind in pop]
            gen_best_r2 = max(r2_scores)
            gen_best_individual = pop[r2_scores.index(gen_best_r2)]

            if gen_best_r2 - best_overall_r2 > improvement_threshold:
                best_overall_r2 = gen_best_r2
                best_overall_individual = toolbox.clone(gen_best_individual)
                no_improve_count = 0
            else:
                no_improve_count += 1

            print(f"🔥 Best R² in generation {gen+1}: {gen_best_r2:.4f}")
            if no_improve_count >= max_no_improve:
                print(f"🚫 No significant improvement in the last {max_no_improve} generations. Early stopping...")
                break

            offspring = toolbox.select(pop, len(pop))
            offspring = list(map(toolbox.clone, offspring))
            for child1, child2 in zip(offspring[::2], offspring[1::2]):
                if random.random() < CXPB:
                    toolbox.mate(child1, child2)
                    del child1.fitness.values, child2.fitness.values
            for mutant in offspring:
                if random.random() < MUTPB:
                    toolbox.mutate(mutant)
                    del mutant.fitness.values
            invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
            fitnesses = list(map(toolbox.evaluate, tqdm(invalid_ind, desc="Evaluating offspring", leave=False)))
            for ind, fit in zip(invalid_ind, fitnesses):
                ind.fitness.values = fit
            pop[:] = offspring

        elapsed = time.time() - start_time
        print(f"\nBest hyperparameters for radius {radius}: {best_overall_individual}")
        print(f"🔥 Best R² Score for radius {radius}: {best_overall_r2:.4f}")
        print(f"⏱ Elapsed time for radius {radius}: {elapsed:.2f} seconds")
        results_per_radius[radius] = {
            "best_hyperparameters": best_overall_individual,
            "best_r2": best_overall_r2,
            "time": elapsed
        }

    # ---------------- PLOTTING AND SUMMARY -----------------
    radii_sorted = sorted(results_per_radius.keys())
    r2_scores = [results_per_radius[r]["best_r2"] for r in radii_sorted]

    plt.figure(figsize=(10, 6))
    plt.plot(radii_sorted, r2_scores, marker='o', linestyle='-')
    plt.xlabel("Radius (meters)")
    plt.ylabel("Best R² Score")
    plt.title("Best R² Score vs. Radius for Montreal Data")
    plt.grid(True)
    plt.show()

    print("\n📌 Summary of Best Hyperparameters per Radius:")
    for radius in radii_sorted:
        best_params = results_per_radius[radius]["best_hyperparameters"]
        best_r2 = results_per_radius[radius]["best_r2"]
        print(f"Radius: {radius} | Best R²: {best_r2:.4f} | Hyperparameters: {best_params}")

In [ ]:
## MONTREAL RADIUS-BASED SELECTION, WEIGHTED MEANS AGGREGATION

import time
import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
from deap import base, creator, tools, algorithms
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import pickle
from collections import defaultdict

# ---------------- USER CONFIGURATION -----------------
# We are using experiment 4 (weighted neighbor aggregation using local distance normalization).
EXPERIMENT = 4

# Montreal-specific settings:
OUTPUT_DIR = "./precomputed_matrices"
FEATURE_COLUMNS = [
    'Bld_densit', 'Veg_densit', 'HW_Ratio', 'Width',
    'Pred_LandU', 'Mean_Popul', 'Mean_Bld_E', 'Max_Bld_He',
    'Water_dens'
]
TARGET_COLUMN = 'Delta_T'

# GA parameters for the environment
POP_SIZE = 50
GENS = 10
CXPB, MUTPB = 0.8, 0.5
improvement_threshold = 0.01
max_no_improve = 3

# Montreal-specific radii (in meters)
radii_list = [0, 100, 200, 300, 400, 500, 750, 1000, 1500, 2000, 3000, 4000, 5000, 7000, 10000, 13000, 14500]

# ---------------- HELPER FUNCTIONS -----------------

def normalize_matrix(matrix):
    """Normalize a matrix to the range [0,1]."""
    return (matrix - np.min(matrix)) / (np.max(matrix) - np.min(matrix))

def aggregate_features_with_local_weighting(edge_attributes, distance_matrix, feature_columns, radius):
    """
    For experiment 4 (local weighting):
    Aggregate features using a locally weighted average based solely on distance.
    Neighbors are selected if their distance is <= radius.
    For each segment, the distances of its neighbors are normalized locally 
    (using the maximum distance among the selected neighbors) to compute weights,
    so that closer neighbors receive higher weights.
    """
    aggregated_features = defaultdict(list)
    for segment_index, segment_attrs in enumerate(edge_attributes):
        # Get original feature values for the current segment.
        segment_features = {col: segment_attrs.get(col, np.nan) for col in feature_columns}
        # Identify neighbor indices within the given radius (exclude self).
        neighbor_indices = np.where(distance_matrix[segment_index] <= radius)[0]
        neighbors = [n for n in neighbor_indices if n != segment_index]
        
        if neighbors:
            neighbor_features = np.array([
                [edge_attributes[n].get(col, np.nan) for col in feature_columns]
                for n in neighbors
            ])
            # Local weighting: use only the distances of selected neighbors.
            local_distance = distance_matrix[segment_index, neighbors]
            local_max_distance = np.max(local_distance)
            # Avoid division by zero: if local_max_distance is zero, assign equal weights.
            if local_max_distance > 0:
                norm_distance_local = 1 - local_distance / local_max_distance
            else:
                norm_distance_local = np.ones_like(local_distance)
            weights = norm_distance_local
            weights_sum = np.sum(weights)
            if weights_sum != 0:
                weights = weights / weights_sum
            else:
                weights = np.ones_like(weights) / len(weights)
            weighted_avg = [np.average(neighbor_features[:, i], weights=weights)
                            for i in range(len(feature_columns))]
        else:
            weighted_avg = [np.nan for _ in feature_columns]
        
        # Save original features and weighted averages.
        for col in feature_columns:
            aggregated_features[f"original_{col}"].append(segment_features[col])
        for i, col in enumerate(feature_columns):
            aggregated_features[f"weighted_avg_{col}_neighbors"].append(weighted_avg[i])
    return pd.DataFrame(aggregated_features)

# ---------------- GA EVALUATION FUNCTION -----------------
# Global variable to hold the current radius for the GA run.
current_radius = None

def evaluate_rf(individual):
    global current_radius
    radius = current_radius
    # For experiment 4, the individual contains:
    # [n_estimators, max_features, max_depth, min_samples_split, bootstrap]
    n_estimators = individual[0]
    max_features = individual[1]
    max_depth = individual[2]
    min_samples_split = individual[3]
    bootstrap = individual[4]
    
    # Aggregate features using the locally weighted aggregation function.
    aggregated_data = aggregate_features_with_local_weighting(
        edge_attributes, distance_matrix, FEATURE_COLUMNS, radius
    )
    # Construct the feature set: original values and weighted neighbor averages.
    feature_set = [f"original_{col}" for col in FEATURE_COLUMNS] + \
                  [f"weighted_avg_{col}_neighbors" for col in FEATURE_COLUMNS]
    X = aggregated_data[feature_set].fillna(aggregated_data[feature_set].median())
    y = np.array([edge_attrs[TARGET_COLUMN] for edge_attrs in edge_attributes])
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    rf = RandomForestRegressor(
        n_estimators=n_estimators,
        max_features=max_features,
        max_depth=max_depth,
        min_samples_split=int(min_samples_split),
        bootstrap=bool(bootstrap),
        random_state=42
    )
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    return (r2_score(y_test, y_pred),)

# ---------------- GA SETUP -----------------
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
# Register hyperparameter generators.
toolbox.register("n_estimators", random.choice, [10, 30, 50, 100, 150, 200])
toolbox.register("max_features", random.choice, ['sqrt', 'log2'])
toolbox.register("max_depth", random.choice, [10, 20, 30, 50, 100])
toolbox.register("min_samples_split", random.choice, [2, 4, 6])
toolbox.register("bootstrap", random.choice, [True, False])

def create_individual():
    return [
        toolbox.n_estimators(),
        toolbox.max_features(),
        toolbox.max_depth(),
        toolbox.min_samples_split(),
        toolbox.bootstrap()
    ]

toolbox.register("individual", tools.initIterate, creator.Individual, create_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", evaluate_rf)
toolbox.register("mate", tools.cxTwoPoint)

def mutate_individual(ind):
    ind[:] = create_individual()
    return (ind,)

toolbox.register("mutate", mutate_individual)
toolbox.register("select", tools.selTournament, tournsize=3)

# ---------------- MAIN EXECUTION -----------------
if __name__ == "__main__":
    print("Loading matrices and dataset for Montreal...")

    # Load the precomputed distance matrix and edge attributes for Montreal.
    distance_matrix = np.load(f"{OUTPUT_DIR}/distance_matrix_Montreal.npy")
    with open(f"{OUTPUT_DIR}/edge_attributes_Montreal.pkl", "rb") as f:
        edge_attributes = pickle.load(f)

    print("Data loaded. Starting GA optimization for Montreal (experiment 4, local weighting)...")

    results_per_radius = {}

    for radius in radii_list:
        print(f"\nRunning GA for radius = {radius} meters")
        current_radius = radius
        start_time = time.time()

        pop = toolbox.population(n=POP_SIZE)
        no_improve_count = 0
        best_overall_r2 = -np.inf
        best_overall_individual = None

        for gen in range(GENS):
            print(f"\n🔄 Generation {gen+1}/{GENS} for radius {radius}")
            fitnesses = list(map(toolbox.evaluate, tqdm(pop, desc="Evaluating individuals", leave=False)))
            for ind, fit in zip(pop, fitnesses):
                ind.fitness.values = fit
            r2_scores = [ind.fitness.values[0] for ind in pop]
            gen_best_r2 = max(r2_scores)
            gen_best_individual = pop[r2_scores.index(gen_best_r2)]

            if gen_best_r2 - best_overall_r2 > improvement_threshold:
                best_overall_r2 = gen_best_r2
                best_overall_individual = toolbox.clone(gen_best_individual)
                no_improve_count = 0
            else:
                no_improve_count += 1

            print(f"🔥 Best R² in generation {gen+1}: {gen_best_r2:.4f}")
            if no_improve_count >= max_no_improve:
                print(f"🚫 No significant improvement in the last {max_no_improve} generations. Early stopping...")
                break

            offspring = toolbox.select(pop, len(pop))
            offspring = list(map(toolbox.clone, offspring))
            for child1, child2 in zip(offspring[::2], offspring[1::2]):
                if random.random() < CXPB:
                    toolbox.mate(child1, child2)
                    del child1.fitness.values, child2.fitness.values
            for mutant in offspring:
                if random.random() < MUTPB:
                    toolbox.mutate(mutant)
                    del mutant.fitness.values
            invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
            fitnesses = list(map(toolbox.evaluate, tqdm(invalid_ind, desc="Evaluating offspring", leave=False)))
            for ind, fit in zip(invalid_ind, fitnesses):
                ind.fitness.values = fit
            pop[:] = offspring

        elapsed = time.time() - start_time
        print(f"\nBest hyperparameters for radius {radius}: {best_overall_individual}")
        print(f"🔥 Best R² Score for radius {radius}: {best_overall_r2:.4f}")
        print(f"⏱ Elapsed time for radius {radius}: {elapsed:.2f} seconds")
        results_per_radius[radius] = {
            "best_hyperparameters": best_overall_individual,
            "best_r2": best_overall_r2,
            "time": elapsed
        }

    # ---------------- PLOTTING AND SUMMARY -----------------
    radii_sorted = sorted(results_per_radius.keys())
    r2_scores = [results_per_radius[r]["best_r2"] for r in radii_sorted]

    plt.figure(figsize=(10, 6))
    plt.plot(radii_sorted, r2_scores, marker='o', linestyle='-')
    plt.xlabel("Radius (meters)")
    plt.ylabel("Best R² Score")
    plt.title("Best R² Score vs. Radius for Montreal Data (Local Weighting)")
    plt.grid(True)
    plt.show()

    print("\n📌 Summary of Best Hyperparameters per Radius:")
    for radius in radii_sorted:
        best_params = results_per_radius[radius]["best_hyperparameters"]
        best_r2 = results_per_radius[radius]["best_r2"]
        print(f"Radius: {radius} | Best R²: {best_r2:.4f} | Hyperparameters: {best_params}")


In [ ]:
## APELDOORN BIKE SURFACE RADIUS-BASED SELECTION, SEPARATE MEANS AGGREGATION

import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from collections import defaultdict
from deap import base, creator, tools
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# =============================================================================
# Global Configuration
# =============================================================================

OUTPUT_DIR = "./precomputed_matrices"

# Geometry-based (time-invariant) columns
GEOMETRY_COLUMNS = [
    "Bld_densit", "Veg_densit", "HW_Ratio", "Road_width",
    "Pred_LandUse", "Mean_population", "Mean_Bld_E",
    "Max_Bld_He", "Water_dens", "Pred_facade_material"
]

# Experiment definitions (each with its own target and time-dependent features)
EXPERIMENTS = {
    "surface": {
        "target_column": "Delta_surface",
        "time_feature_columns": ["Ref_surface_temp", "Mean_RH", "Ref_RH", "Mean_canopy_temp", "Ref_canopy_temp"]
    },
    "canopy": {
        "target_column": "Delta_canopy",
        "time_feature_columns": ["Ref_canopy_temp", "Mean_RH", "Ref_RH", "Mean_surface_temp", "Ref_surface_temp"]
    }
}

# Set experiment mode ("surface" or "canopy")
EXPERIMENT_MODE = "surface"  # Change to "canopy" for the canopy experiment

# GA settings for Random Forest hyperparameter optimization
POP_SIZE = 20
GENS = 10
CXPB = 0.8
MUTPB = 0.5
improvement_threshold = 0.01
max_no_improve = 3

# List of radii (in meters) to test
radii_list = [0, 100, 250, 500, 750, 1000, 1500, 2000, 3000, 4000, 4500, 5140]

# Aggregation types: 1 for simple (unweighted) and 4 for weighted (using adjacency index)
AGGREGATION_TYPES = [1, 4]

# Global variables to be updated by the GA evaluation function
current_radius = None       # Current radius threshold
current_exp_type = None     # Current aggregation type (1 or 4)

# =============================================================================
# Helper Functions: Normalization & Adjacency Index
# =============================================================================
def normalize_matrix(matrix):
    """Normalize a matrix to the range [0,1]."""
    min_val, max_val = np.min(matrix), np.max(matrix)
    if min_val == max_val:
        return np.zeros_like(matrix)
    return (matrix - min_val) / (max_val - min_val)

def compute_adjacency_index(distance_matrix):
    """Compute the adjacency index as (1 - normalized distance matrix)."""
    distance_normalized = normalize_matrix(distance_matrix)
    return 1 - distance_normalized

# =============================================================================
# Data Loading and DataFrame Building
# =============================================================================
def load_data():
    print("Loading matrices and attributes...")
    # Load distance matrix
    distance_matrix = np.load(f"{OUTPUT_DIR}/distance_matrix_Bike.npy")
    # Load geometry-based attributes (list of dicts)
    with open(f"{OUTPUT_DIR}/edge_attributes_Bike.pkl", "rb") as f:
        edge_attributes = pickle.load(f)
    # Load time-based attributes (list of dicts, one per time step)
    with open(f"{OUTPUT_DIR}/timebased_attributes_Bike.pkl", "rb") as f:
        timebased_data = pickle.load(f)
    print("Data loaded successfully.")
    return distance_matrix, edge_attributes, timebased_data

def build_geometry_df(edge_attributes):
    """Convert geometry-based attributes into a DataFrame indexed by segment_id."""
    df_list = []
    for seg_attr in edge_attributes:
        sid = seg_attr["segment_id"]
        row_dict = {"segment_id": sid}
        for col in GEOMETRY_COLUMNS:
            row_dict[col] = seg_attr[col]
        df_list.append(row_dict)
    return pd.DataFrame(df_list).set_index("segment_id")

def build_time_df(timebased_data):
    """Convert time-based attributes into a DataFrame."""
    return pd.DataFrame(timebased_data)

# =============================================================================
# Aggregation Function with Dual Modes
# =============================================================================
def aggregate_time_step_features(time_df, geometry_df, adjacency_index, distance_matrix,
                                 radius_threshold, time_feature_columns, target_column, exp_type):
    """
    For each row in time_df, aggregate neighbor features from segments within radius_threshold.
    If exp_type==4, compute a weighted average using the adjacency index.
    If exp_type==1, compute a simple (unweighted) average.
    Returns a DataFrame with:
      - segment_id, Date, Part_of_the_day,
      - geometry-based features,
      - original_{time_feature_columns},
      - aggregated neighbor features (prefixed with "weighted_avg_" or "avg_"),
      - and original target (prefixed with "original_").
    """
    # Build lookup for time-based rows: (segment_id, Date, Part_of_the_day) -> row
    time_lookup = {}
    for i, row in time_df.iterrows():
        key = (row["segment_id"], row["Date"], row["Part_of_the_day"])
        time_lookup[key] = row

    results = []
    for i, row in time_df.iterrows():
        seg_id = row["segment_id"]
        date_val = row["Date"]
        part_day = row["Part_of_the_day"]

        # Get geometry features
        geom_feats = geometry_df.loc[seg_id].to_dict()

        # Get original time-based features
        time_feats = {}
        for col in time_feature_columns:
            time_feats[f"original_{col}"] = row[col]
        time_feats[f"original_{target_column}"] = row.get(target_column, np.nan)

        # Identify neighbor segments using distance_matrix (assumes segment_id is an integer index)
        seg_index = seg_id
        neighbor_ids = np.where(distance_matrix[seg_index] <= radius_threshold)[0]
        neighbor_ids = [nid for nid in neighbor_ids if nid != seg_index]

        aggregated = {}
        if len(neighbor_ids) > 0:
            neighbor_matrix = []
            for nid in neighbor_ids:
                key_neighbor = (nid, date_val, part_day)
                if key_neighbor not in time_lookup:
                    neighbor_matrix.append([np.nan] * len(time_feature_columns))
                else:
                    neighbor_row = time_lookup[key_neighbor]
                    neighbor_matrix.append([neighbor_row[col] for col in time_feature_columns])
            neighbor_matrix = np.array(neighbor_matrix, dtype=float)
            for col_idx, col_name in enumerate(time_feature_columns):
                col_values = neighbor_matrix[:, col_idx]
                if np.isnan(col_values).all():
                    agg_val = np.nan
                else:
                    if exp_type == 4:
                        weights = adjacency_index[seg_index, neighbor_ids]
                        valid_mask = ~np.isnan(col_values)
                        if valid_mask.any():
                            valid_weights = weights[valid_mask]
                            w_sum = valid_weights.sum()
                            if w_sum > 0:
                                valid_weights = valid_weights / w_sum
                            else:
                                valid_weights = np.ones_like(valid_weights) / len(valid_weights)
                            agg_val = np.average(col_values[valid_mask], weights=valid_weights)
                        else:
                            agg_val = np.nan
                        aggregated[f"weighted_avg_{col_name}_neighbors"] = agg_val
                    elif exp_type == 1:
                        agg_val = np.nanmean(col_values)
                        aggregated[f"avg_{col_name}_neighbors"] = agg_val
        else:
            for col_name in time_feature_columns:
                if exp_type == 4:
                    aggregated[f"weighted_avg_{col_name}_neighbors"] = np.nan
                elif exp_type == 1:
                    aggregated[f"avg_{col_name}_neighbors"] = np.nan

        out_row = {"segment_id": seg_id, "Date": date_val, "Part_of_the_day": part_day}
        out_row.update(geom_feats)
        out_row.update(time_feats)
        out_row.update(aggregated)
        results.append(out_row)
    return pd.DataFrame(results)

# =============================================================================
# GA Evaluation Function for Random Forest
# =============================================================================
def evaluate_rf(individual, geometry_df, time_df, adjacency_index, distance_matrix,
                time_feature_columns, target_column):
    global current_radius, current_exp_type
    n_estimators = individual[0]
    max_features = individual[1]
    max_depth = individual[2]
    min_samples_split = individual[3]
    bootstrap = bool(individual[4])
    
    aggregated_df = aggregate_time_step_features(
        time_df=time_df,
        geometry_df=geometry_df,
        adjacency_index=adjacency_index,
        distance_matrix=distance_matrix,
        radius_threshold=current_radius,
        time_feature_columns=time_feature_columns,
        target_column=target_column,
        exp_type=current_exp_type
    )
    aggregated_df = aggregated_df.dropna(subset=[f"original_{target_column}"])
    
    geometry_feats = GEOMETRY_COLUMNS
    if current_exp_type == 4:
        time_feats_orig = [f"original_{col}" for col in time_feature_columns]
        time_feats_agg = [f"weighted_avg_{col}_neighbors" for col in time_feature_columns]
    else:
        time_feats_orig = [f"original_{col}" for col in time_feature_columns]
        time_feats_agg = [f"avg_{col}_neighbors" for col in time_feature_columns]
    all_features = geometry_feats + time_feats_orig + time_feats_agg
    
    X = aggregated_df[all_features].fillna(aggregated_df[all_features].median())
    y = aggregated_df[f"original_{target_column}"]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    rf = RandomForestRegressor(
        n_estimators=n_estimators,
        max_features=max_features,
        max_depth=max_depth,
        min_samples_split=int(min_samples_split),
        bootstrap=bootstrap,
        random_state=42
    )
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    return (r2,)

# =============================================================================
# GA Setup using DEAP
# =============================================================================
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("n_estimators", random.choice, [10, 30, 50, 100, 150, 200])
toolbox.register("max_features", random.choice, ['sqrt', 'log2'])
toolbox.register("max_depth", random.choice, [10, 20, 30, 50, 100])
toolbox.register("min_samples_split", random.choice, [2, 4, 6])
toolbox.register("bootstrap", random.choice, [True, False])

def create_individual():
    return [toolbox.n_estimators(), toolbox.max_features(), toolbox.max_depth(),
            toolbox.min_samples_split(), toolbox.bootstrap()]
toolbox.register("individual", tools.initIterate, creator.Individual, create_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", evaluate_rf, geometry_df=None, time_df=None,
                 adjacency_index=None, distance_matrix=None, time_feature_columns=None,
                 target_column=None)
toolbox.register("mate", tools.cxTwoPoint)
def mutate_individual(ind):
    ind[:] = create_individual()
    return (ind,)
toolbox.register("mutate", mutate_individual)
toolbox.register("select", tools.selTournament, tournsize=3)

# =============================================================================
# GA Optimization per Radius
# =============================================================================
def run_GA_for_radius(geometry_df, time_df, adjacency_index, distance_matrix,
                      time_feature_columns, target_column):
    global current_radius
    pop = toolbox.population(n=POP_SIZE)
    no_improve_count = 0
    best_overall_r2 = -np.inf
    best_overall_individual = None
    
    for gen in range(GENS):
        fitnesses = []
        for ind in tqdm(pop, desc=f"Generation {gen+1} (radius {current_radius})", leave=False):
            fit = toolbox.evaluate(ind, geometry_df=geometry_df, time_df=time_df,
                                     adjacency_index=adjacency_index, distance_matrix=distance_matrix,
                                     time_feature_columns=time_feature_columns, target_column=target_column)
            fitnesses.append(fit)
            ind.fitness.values = fit
        r2_scores = [ind.fitness.values[0] for ind in pop]
        gen_best_r2 = max(r2_scores)
        gen_best_individual = pop[r2_scores.index(gen_best_r2)]
        
        if gen_best_r2 - best_overall_r2 > improvement_threshold:
            best_overall_r2 = gen_best_r2
            best_overall_individual = toolbox.clone(gen_best_individual)
            no_improve_count = 0
        else:
            no_improve_count += 1
        
        if no_improve_count >= max_no_improve:
            print(f"Early stopping at generation {gen+1} for radius {current_radius}")
            break
        
        offspring = toolbox.select(pop, len(pop))
        offspring = list(map(toolbox.clone, offspring))
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CXPB:
                toolbox.mate(child1, child2)
                del child1.fitness.values, child2.fitness.values
        for mutant in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(mutant)
                del mutant.fitness.values
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        for ind in invalid_ind:
            fit = toolbox.evaluate(ind, geometry_df=geometry_df, time_df=time_df,
                                     adjacency_index=adjacency_index, distance_matrix=distance_matrix,
                                     time_feature_columns=time_feature_columns, target_column=target_column)
            ind.fitness.values = fit
        pop[:] = offspring
    return best_overall_individual, best_overall_r2

# =============================================================================
# Main GA Experiment Loop
# =============================================================================
def main():
    global current_radius, current_exp_type
    # Ensure globals are updated
    current_radius = None
    current_exp_type = None
    
    # Load data
    distance_matrix, edge_attributes, timebased_data = load_data()
    adjacency_index = compute_adjacency_index(distance_matrix)
    print("Adjacency index computed.")
    geometry_df = build_geometry_df(edge_attributes)
    time_df = build_time_df(timebased_data)
    
    # Get experiment-specific parameters
    exp_params = EXPERIMENTS[EXPERIMENT_MODE]
    target_column = exp_params["target_column"]
    time_feature_columns = exp_params["time_feature_columns"]
    
    # Dictionary to store GA results by aggregation type and radius
    results = {agg_type: {} for agg_type in AGGREGATION_TYPES}
    
    # Loop over aggregation types (1: simple, 4: weighted)
    for agg_type in AGGREGATION_TYPES:
        current_exp_type = agg_type  # update global current_exp_type
        print(f"\nRunning GA for aggregation type {agg_type} ({'Simple' if agg_type==1 else 'Weighted'}) - {EXPERIMENT_MODE} experiment")
        for radius in radii_list:
            current_radius = radius  # update global current_radius
            print(f"\nOptimizing for radius = {radius} meters")
            best_individual, best_r2 = run_GA_for_radius(
                geometry_df=geometry_df,
                time_df=time_df,
                adjacency_index=adjacency_index,
                distance_matrix=distance_matrix,
                time_feature_columns=time_feature_columns,
                target_column=target_column
            )
            print(f"Radius: {radius}, Best R²: {best_r2:.4f}, Best Hyperparameters: {best_individual}")
            results[agg_type][radius] = {
                "best_individual": best_individual,
                "best_r2": best_r2
            }
    
    # Plotting: R² vs. Radius for both aggregation types
    plt.figure(figsize=(10, 6))
    for agg_type in AGGREGATION_TYPES:
        radii_sorted = sorted(results[agg_type].keys())
        r2_scores = [results[agg_type][r]["best_r2"] for r in radii_sorted]
        label = "Simple Avg" if agg_type == 1 else "Weighted Avg"
        plt.plot(radii_sorted, r2_scores, marker='o', label=label)
    plt.xlabel("Radius (meters)")
    plt.ylabel("Best R² Score")
    plt.title(f"Best R² Score vs. Radius ({EXPERIMENT_MODE.capitalize()} Experiment)")
    plt.legend()
    plt.grid(True)
    plt.show()
    
    # Print summary table
    summary_rows = []
    for agg_type in AGGREGATION_TYPES:
        for radius in sorted(results[agg_type].keys()):
            best_ind = results[agg_type][radius]["best_individual"]
            r2_val = results[agg_type][radius]["best_r2"]
            summary_rows.append({
                "Aggregation Type": "Simple" if agg_type == 1 else "Weighted",
                "Radius (m)": radius,
                "Best R²": f"{r2_val:.4f}",
                "Hyperparameters": best_ind
            })
    summary_df = pd.DataFrame(summary_rows)
    print("\nSummary of Best Hyperparameters per Radius and Aggregation Type:")
    print(summary_df.to_string(index=False))
    
if __name__ == "__main__":
    main()


In [ ]:
## APELDOORN BIKE SURFACE RADIUS-BASED SELECTION, WEIGHTED MEANS AGGREGATION

import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from collections import defaultdict
from deap import base, creator, tools
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# =============================================================================
# Global Configuration
# =============================================================================

OUTPUT_DIR = "./precomputed_matrices"

# Geometry-based (time-invariant) columns
GEOMETRY_COLUMNS = [
    "Bld_densit", "Veg_densit", "HW_Ratio", "Road_width",
    "Pred_LandUse", "Mean_population", "Mean_Bld_E",
    "Max_Bld_He", "Water_dens", "Pred_facade_material"
]

# Experiment definitions (each with its own target and time-dependent features)
EXPERIMENTS = {
    "surface": {
        "target_column": "Delta_surface",
        "time_feature_columns": ["Ref_surface_temp", "Mean_RH", "Ref_RH", "Mean_canopy_temp", "Ref_canopy_temp"]
    },
    "canopy": {
        "target_column": "Delta_canopy",
        "time_feature_columns": ["Ref_canopy_temp", "Mean_RH", "Ref_RH", "Mean_surface_temp", "Ref_surface_temp"]
    }
}

# Set experiment mode ("surface" or "canopy")
EXPERIMENT_MODE = "surface"  # Change to "canopy" for the canopy experiment

# GA settings for Random Forest hyperparameter optimization
POP_SIZE = 50
GENS = 10
CXPB = 0.8
MUTPB = 0.5
improvement_threshold = 0.01
max_no_improve = 5

# List of radii (in meters) to test
radii_list = [0, 100, 250, 500, 750, 1000, 1500, 2000, 3000, 4000, 4500, 5140]

# We only keep the weighted average experiment (experiment type 4)
current_exp_type = 4  # Always weighted average

# Global variables to be updated by the GA evaluation function
current_radius = None  # Current radius threshold

# =============================================================================
# Helper Functions: Normalization & Adjacency Index
# =============================================================================
def normalize_matrix(matrix):
    """Normalize a matrix to the range [0,1]."""
    min_val, max_val = np.min(matrix), np.max(matrix)
    if min_val == max_val:
        return np.zeros_like(matrix)
    return (matrix - min_val) / (max_val - min_val)

def compute_adjacency_index(distance_matrix):
    """Compute the adjacency index as (1 - normalized distance matrix)."""
    distance_normalized = normalize_matrix(distance_matrix)
    return 1 - distance_normalized

# =============================================================================
# Data Loading and DataFrame Building
# =============================================================================
def load_data():
    print("Loading matrices and attributes...")
    # Load distance matrix
    distance_matrix = np.load(f"{OUTPUT_DIR}/distance_matrix_Bike.npy")
    # Load geometry-based attributes (list of dicts)
    with open(f"{OUTPUT_DIR}/edge_attributes_Bike.pkl", "rb") as f:
        edge_attributes = pickle.load(f)
    # Load time-based attributes (list of dicts, one per time step)
    with open(f"{OUTPUT_DIR}/timebased_attributes_Bike.pkl", "rb") as f:
        timebased_data = pickle.load(f)
    print("Data loaded successfully.")
    return distance_matrix, edge_attributes, timebased_data

def build_geometry_df(edge_attributes):
    """Convert geometry-based attributes into a DataFrame indexed by segment_id."""
    df_list = []
    for seg_attr in edge_attributes:
        sid = seg_attr["segment_id"]
        row_dict = {"segment_id": sid}
        for col in GEOMETRY_COLUMNS:
            row_dict[col] = seg_attr[col]
        df_list.append(row_dict)
    return pd.DataFrame(df_list).set_index("segment_id")

def build_time_df(timebased_data):
    """Convert time-based attributes into a DataFrame."""
    return pd.DataFrame(timebased_data)

# =============================================================================
# Aggregation Function: Weighted Average with Local Weighting Only
# =============================================================================
def aggregate_time_step_features(time_df, geometry_df, distance_matrix,
                                 radius_threshold, time_feature_columns, target_column):
    """
    For each row in time_df, aggregate neighbor features from segments within radius_threshold
    using a locally weighted average based solely on distance.
    The weight for each neighbor is computed by normalizing the distances locally (using the maximum
    distance among the selected neighbors) so that closer neighbors receive higher weight.
    Returns a DataFrame with:
      - segment_id, Date, Part_of_the_day,
      - geometry-based features,
      - original_{time_feature_columns},
      - weighted_avg_{time_feature_columns}_neighbors,
      - and original target (prefixed with "original_").
    """
    # Build lookup for time-based rows: (segment_id, Date, Part_of_the_day) -> row
    time_lookup = {}
    for i, row in time_df.iterrows():
        key = (row["segment_id"], row["Date"], row["Part_of_the_day"])
        time_lookup[key] = row

    results = []
    for i, row in time_df.iterrows():
        seg_id = row["segment_id"]
        date_val = row["Date"]
        part_day = row["Part_of_the_day"]

        # Get geometry features
        geom_feats = geometry_df.loc[seg_id].to_dict()

        # Get original time-based features
        time_feats = {}
        for col in time_feature_columns:
            time_feats[f"original_{col}"] = row[col]
        time_feats[f"original_{target_column}"] = row.get(target_column, np.nan)

        # Identify neighbor segments using distance_matrix (assumes segment_id is an integer index)
        seg_index = seg_id
        neighbor_ids = np.where(distance_matrix[seg_index] <= radius_threshold)[0]
        neighbor_ids = [nid for nid in neighbor_ids if nid != seg_index]

        aggregated = {}
        if len(neighbor_ids) > 0:
            neighbor_matrix = []
            for nid in neighbor_ids:
                key_neighbor = (nid, date_val, part_day)
                if key_neighbor not in time_lookup:
                    neighbor_matrix.append([np.nan] * len(time_feature_columns))
                else:
                    neighbor_row = time_lookup[key_neighbor]
                    neighbor_matrix.append([neighbor_row[col] for col in time_feature_columns])
            neighbor_matrix = np.array(neighbor_matrix, dtype=float)
            for col_idx, col_name in enumerate(time_feature_columns):
                col_values = neighbor_matrix[:, col_idx]
                if np.isnan(col_values).all():
                    agg_val = np.nan
                else:
                    # Local weighting: compute weights based on distances of only the selected neighbors
                    local_distance = distance_matrix[seg_index, neighbor_ids]
                    local_max_distance = np.max(local_distance)
                    if local_max_distance > 0:
                        norm_distance_local = 1 - local_distance / local_max_distance
                    else:
                        norm_distance_local = np.ones_like(local_distance)
                    weights = norm_distance_local
                    valid_mask = ~np.isnan(col_values)
                    if valid_mask.any():
                        valid_weights = weights[valid_mask]
                        w_sum = valid_weights.sum()
                        if w_sum > 0:
                            valid_weights = valid_weights / w_sum
                        else:
                            valid_weights = np.ones_like(valid_weights) / len(valid_weights)
                        agg_val = np.average(col_values[valid_mask], weights=valid_weights)
                    else:
                        agg_val = np.nan
                    aggregated[f"weighted_avg_{col_name}_neighbors"] = agg_val
        else:
            for col_name in time_feature_columns:
                aggregated[f"weighted_avg_{col_name}_neighbors"] = np.nan

        out_row = {"segment_id": seg_id, "Date": date_val, "Part_of_the_day": part_day}
        out_row.update(geom_feats)
        out_row.update(time_feats)
        out_row.update(aggregated)
        results.append(out_row)
    return pd.DataFrame(results)

# =============================================================================
# GA Evaluation Function for Random Forest (Weighted Average Only)
# =============================================================================
def evaluate_rf(individual, geometry_df, time_df, distance_matrix, time_feature_columns, target_column):
    global current_radius
    n_estimators = individual[0]
    max_features = individual[1]
    max_depth = individual[2]
    min_samples_split = individual[3]
    bootstrap = bool(individual[4])
    
    aggregated_df = aggregate_time_step_features(
        time_df=time_df,
        geometry_df=geometry_df,
        distance_matrix=distance_matrix,
        radius_threshold=current_radius,
        time_feature_columns=time_feature_columns,
        target_column=target_column
    )
    aggregated_df = aggregated_df.dropna(subset=[f"original_{target_column}"])
    
    geometry_feats = GEOMETRY_COLUMNS
    time_feats_orig = [f"original_{col}" for col in time_feature_columns]
    time_feats_agg = [f"weighted_avg_{col}_neighbors" for col in time_feature_columns]
    all_features = geometry_feats + time_feats_orig + time_feats_agg
    
    X = aggregated_df[all_features].fillna(aggregated_df[all_features].median())
    y = aggregated_df[f"original_{target_column}"]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    rf = RandomForestRegressor(
        n_estimators=n_estimators,
        max_features=max_features,
        max_depth=max_depth,
        min_samples_split=int(min_samples_split),
        bootstrap=bootstrap,
        random_state=42
    )
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    return (r2,)

# =============================================================================
# GA Setup using DEAP
# =============================================================================
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("n_estimators", random.choice, [10, 30, 50, 100, 150, 200])
toolbox.register("max_features", random.choice, ['sqrt', 'log2'])
toolbox.register("max_depth", random.choice, [10, 20, 30, 50, 100])
toolbox.register("min_samples_split", random.choice, [2, 4, 6])
toolbox.register("bootstrap", random.choice, [True, False])

def create_individual():
    return [
        toolbox.n_estimators(),
        toolbox.max_features(),
        toolbox.max_depth(),
        toolbox.min_samples_split(),
        toolbox.bootstrap()
    ]
toolbox.register("individual", tools.initIterate, creator.Individual, create_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", evaluate_rf, geometry_df=None, time_df=None,
                 distance_matrix=None, time_feature_columns=None, target_column=None)
toolbox.register("mate", tools.cxTwoPoint)
def mutate_individual(ind):
    ind[:] = create_individual()
    return (ind,)
toolbox.register("mutate", mutate_individual)
toolbox.register("select", tools.selTournament, tournsize=3)

# =============================================================================
# GA Optimization per Radius
# =============================================================================
def run_GA_for_radius(geometry_df, time_df, distance_matrix, time_feature_columns, target_column):
    global current_radius
    pop = toolbox.population(n=POP_SIZE)
    no_improve_count = 0
    best_overall_r2 = -np.inf
    best_overall_individual = None
    
    for gen in range(GENS):
        fitnesses = []
        for ind in tqdm(pop, desc=f"Generation {gen+1} (radius {current_radius})", leave=False):
            fit = toolbox.evaluate(ind, geometry_df=geometry_df, time_df=time_df,
                                     distance_matrix=distance_matrix,
                                     time_feature_columns=time_feature_columns,
                                     target_column=target_column)
            fitnesses.append(fit)
            ind.fitness.values = fit
        r2_scores = [ind.fitness.values[0] for ind in pop]
        gen_best_r2 = max(r2_scores)
        gen_best_individual = pop[r2_scores.index(gen_best_r2)]
        
        if gen_best_r2 - best_overall_r2 > improvement_threshold:
            best_overall_r2 = gen_best_r2
            best_overall_individual = toolbox.clone(gen_best_individual)
            no_improve_count = 0
        else:
            no_improve_count += 1
        
        if no_improve_count >= max_no_improve:
            print(f"Early stopping at generation {gen+1} for radius {current_radius}")
            break
        
        offspring = toolbox.select(pop, len(pop))
        offspring = list(map(toolbox.clone, offspring))
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CXPB:
                toolbox.mate(child1, child2)
                del child1.fitness.values, child2.fitness.values
        for mutant in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(mutant)
                del mutant.fitness.values
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        for ind in invalid_ind:
            fit = toolbox.evaluate(ind, geometry_df=geometry_df, time_df=time_df,
                                     distance_matrix=distance_matrix,
                                     time_feature_columns=time_feature_columns,
                                     target_column=target_column)
            ind.fitness.values = fit
        pop[:] = offspring
    return best_overall_individual, best_overall_r2

# =============================================================================
# Main GA Experiment Loop
# =============================================================================
def main():
    global current_radius
    # Load data
    distance_matrix, edge_attributes, timebased_data = load_data()
    # Use local weighting: do not use global adjacency index computed from entire matrix.
    # (We compute local weights directly in the aggregation function.)
    geometry_df = build_geometry_df(edge_attributes)
    time_df = build_time_df(timebased_data)
    
    # Get experiment-specific parameters for the chosen experiment mode.
    exp_params = EXPERIMENTS[EXPERIMENT_MODE]
    target_column = exp_params["target_column"]
    time_feature_columns = exp_params["time_feature_columns"]
    
    # Dictionary to store GA results by radius.
    results = {}
    
    print(f"\nStarting GA Optimization for Montreal ({EXPERIMENT_MODE.capitalize()} Experiment, Weighted Average with Local Weighting)")
    for radius in radii_list:
        current_radius = radius  # update global current_radius
        print(f"\nOptimizing for radius = {radius} meters")
        best_individual, best_r2 = run_GA_for_radius(
            geometry_df=geometry_df,
            time_df=time_df,
            distance_matrix=distance_matrix,
            time_feature_columns=time_feature_columns,
            target_column=target_column
        )
        print(f"Radius: {radius}, Best R²: {best_r2:.4f}, Best Hyperparameters: {best_individual}")
        results[radius] = {
            "best_individual": best_individual,
            "best_r2": best_r2
        }
    
    # Plotting: R² vs. Radius
    plt.figure(figsize=(10, 6))
    radii_sorted = sorted(results.keys())
    r2_scores = [results[r]["best_r2"] for r in radii_sorted]
    plt.plot(radii_sorted, r2_scores, marker='o', linestyle='-')
    plt.xlabel("Radius (meters)")
    plt.ylabel("Best R² Score")
    plt.title(f"Best R² Score vs. Radius ({EXPERIMENT_MODE.capitalize()} Experiment)")
    plt.grid(True)
    plt.show()
    
    # Print summary table
    summary_rows = []
    for radius in sorted(results.keys()):
        best_ind = results[radius]["best_individual"]
        r2_val = results[radius]["best_r2"]
        summary_rows.append({
            "Radius (m)": radius,
            "Best R²": f"{r2_val:.4f}",
            "Hyperparameters": best_ind
        })
    summary_df = pd.DataFrame(summary_rows)
    print("\nSummary of Best Hyperparameters per Radius:")
    print(summary_df.to_string(index=False))
    
if __name__ == "__main__":
    main()

In [ ]:
## APELDOORN BIKE CANOPY RADIUS-BASED SELECTION, SEPARATE MEANS AGGREGATION

import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from collections import defaultdict
from deap import base, creator, tools
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# =============================================================================
# Global Configuration
# =============================================================================

OUTPUT_DIR = "./precomputed_matrices"

# Geometry-based (time-invariant) columns
GEOMETRY_COLUMNS = [
    "Bld_densit", "Veg_densit", "HW_Ratio", "Road_width",
    "Pred_LandUse", "Mean_population", "Mean_Bld_E",
    "Max_Bld_He", "Water_dens", "Pred_facade_material"
]

# Experiment definitions (each with its own target and time-dependent features)
EXPERIMENTS = {
    "surface": {
        "target_column": "Delta_surface",
        "time_feature_columns": ["Ref_surface_temp", "Mean_RH", "Ref_RH", "Mean_canopy_temp", "Ref_canopy_temp"]
    },
    "canopy": {
        "target_column": "Delta_canopy",
        "time_feature_columns": ["Ref_canopy_temp", "Mean_RH", "Ref_RH", "Mean_surface_temp", "Ref_surface_temp"]
    }
}

# Set experiment mode ("surface" or "canopy")
EXPERIMENT_MODE = "canopy"  # Change to "canopy" for the canopy experiment

# GA settings for Random Forest hyperparameter optimization
POP_SIZE = 20
GENS = 10
CXPB = 0.8
MUTPB = 0.5
improvement_threshold = 0.01
max_no_improve = 3

# List of radii (in meters) to test
radii_list = [0, 100, 250, 500, 750, 1000, 1500, 2000, 3000, 4000, 4500, 5140]

# Aggregation types: 1 for simple (unweighted) and 4 for weighted (using adjacency index)
AGGREGATION_TYPES = [1, 4]

# Global variables to be updated by the GA evaluation function
current_radius = None       # Current radius threshold
current_exp_type = None     # Current aggregation type (1 or 4)

# =============================================================================
# Helper Functions: Normalization & Adjacency Index
# =============================================================================
def normalize_matrix(matrix):
    """Normalize a matrix to the range [0,1]."""
    min_val, max_val = np.min(matrix), np.max(matrix)
    if min_val == max_val:
        return np.zeros_like(matrix)
    return (matrix - min_val) / (max_val - min_val)

def compute_adjacency_index(distance_matrix):
    """Compute the adjacency index as (1 - normalized distance matrix)."""
    distance_normalized = normalize_matrix(distance_matrix)
    return 1 - distance_normalized

# =============================================================================
# Data Loading and DataFrame Building
# =============================================================================
def load_data():
    print("Loading matrices and attributes...")
    # Load distance matrix
    distance_matrix = np.load(f"{OUTPUT_DIR}/distance_matrix_Bike.npy")
    # Load geometry-based attributes (list of dicts)
    with open(f"{OUTPUT_DIR}/edge_attributes_Bike.pkl", "rb") as f:
        edge_attributes = pickle.load(f)
    # Load time-based attributes (list of dicts, one per time step)
    with open(f"{OUTPUT_DIR}/timebased_attributes_Bike.pkl", "rb") as f:
        timebased_data = pickle.load(f)
    print("Data loaded successfully.")
    return distance_matrix, edge_attributes, timebased_data

def build_geometry_df(edge_attributes):
    """Convert geometry-based attributes into a DataFrame indexed by segment_id."""
    df_list = []
    for seg_attr in edge_attributes:
        sid = seg_attr["segment_id"]
        row_dict = {"segment_id": sid}
        for col in GEOMETRY_COLUMNS:
            row_dict[col] = seg_attr[col]
        df_list.append(row_dict)
    return pd.DataFrame(df_list).set_index("segment_id")

def build_time_df(timebased_data):
    """Convert time-based attributes into a DataFrame."""
    return pd.DataFrame(timebased_data)

# =============================================================================
# Aggregation Function with Dual Modes
# =============================================================================
def aggregate_time_step_features(time_df, geometry_df, adjacency_index, distance_matrix,
                                 radius_threshold, time_feature_columns, target_column, exp_type):
    """
    For each row in time_df, aggregate neighbor features from segments within radius_threshold.
    If exp_type==4, compute a weighted average using the adjacency index.
    If exp_type==1, compute a simple (unweighted) average.
    Returns a DataFrame with:
      - segment_id, Date, Part_of_the_day,
      - geometry-based features,
      - original_{time_feature_columns},
      - aggregated neighbor features (prefixed with "weighted_avg_" or "avg_"),
      - and original target (prefixed with "original_").
    """
    # Build lookup for time-based rows: (segment_id, Date, Part_of_the_day) -> row
    time_lookup = {}
    for i, row in time_df.iterrows():
        key = (row["segment_id"], row["Date"], row["Part_of_the_day"])
        time_lookup[key] = row

    results = []
    for i, row in time_df.iterrows():
        seg_id = row["segment_id"]
        date_val = row["Date"]
        part_day = row["Part_of_the_day"]

        # Get geometry features
        geom_feats = geometry_df.loc[seg_id].to_dict()

        # Get original time-based features
        time_feats = {}
        for col in time_feature_columns:
            time_feats[f"original_{col}"] = row[col]
        time_feats[f"original_{target_column}"] = row.get(target_column, np.nan)

        # Identify neighbor segments using distance_matrix (assumes segment_id is an integer index)
        seg_index = seg_id
        neighbor_ids = np.where(distance_matrix[seg_index] <= radius_threshold)[0]
        neighbor_ids = [nid for nid in neighbor_ids if nid != seg_index]

        aggregated = {}
        if len(neighbor_ids) > 0:
            neighbor_matrix = []
            for nid in neighbor_ids:
                key_neighbor = (nid, date_val, part_day)
                if key_neighbor not in time_lookup:
                    neighbor_matrix.append([np.nan] * len(time_feature_columns))
                else:
                    neighbor_row = time_lookup[key_neighbor]
                    neighbor_matrix.append([neighbor_row[col] for col in time_feature_columns])
            neighbor_matrix = np.array(neighbor_matrix, dtype=float)
            for col_idx, col_name in enumerate(time_feature_columns):
                col_values = neighbor_matrix[:, col_idx]
                if np.isnan(col_values).all():
                    agg_val = np.nan
                else:
                    if exp_type == 4:
                        weights = adjacency_index[seg_index, neighbor_ids]
                        valid_mask = ~np.isnan(col_values)
                        if valid_mask.any():
                            valid_weights = weights[valid_mask]
                            w_sum = valid_weights.sum()
                            if w_sum > 0:
                                valid_weights = valid_weights / w_sum
                            else:
                                valid_weights = np.ones_like(valid_weights) / len(valid_weights)
                            agg_val = np.average(col_values[valid_mask], weights=valid_weights)
                        else:
                            agg_val = np.nan
                        aggregated[f"weighted_avg_{col_name}_neighbors"] = agg_val
                    elif exp_type == 1:
                        agg_val = np.nanmean(col_values)
                        aggregated[f"avg_{col_name}_neighbors"] = agg_val
        else:
            for col_name in time_feature_columns:
                if exp_type == 4:
                    aggregated[f"weighted_avg_{col_name}_neighbors"] = np.nan
                elif exp_type == 1:
                    aggregated[f"avg_{col_name}_neighbors"] = np.nan

        out_row = {"segment_id": seg_id, "Date": date_val, "Part_of_the_day": part_day}
        out_row.update(geom_feats)
        out_row.update(time_feats)
        out_row.update(aggregated)
        results.append(out_row)
    return pd.DataFrame(results)

# =============================================================================
# GA Evaluation Function for Random Forest
# =============================================================================
def evaluate_rf(individual, geometry_df, time_df, adjacency_index, distance_matrix,
                time_feature_columns, target_column):
    global current_radius, current_exp_type
    n_estimators = individual[0]
    max_features = individual[1]
    max_depth = individual[2]
    min_samples_split = individual[3]
    bootstrap = bool(individual[4])
    
    aggregated_df = aggregate_time_step_features(
        time_df=time_df,
        geometry_df=geometry_df,
        adjacency_index=adjacency_index,
        distance_matrix=distance_matrix,
        radius_threshold=current_radius,
        time_feature_columns=time_feature_columns,
        target_column=target_column,
        exp_type=current_exp_type
    )
    aggregated_df = aggregated_df.dropna(subset=[f"original_{target_column}"])
    
    geometry_feats = GEOMETRY_COLUMNS
    if current_exp_type == 4:
        time_feats_orig = [f"original_{col}" for col in time_feature_columns]
        time_feats_agg = [f"weighted_avg_{col}_neighbors" for col in time_feature_columns]
    else:
        time_feats_orig = [f"original_{col}" for col in time_feature_columns]
        time_feats_agg = [f"avg_{col}_neighbors" for col in time_feature_columns]
    all_features = geometry_feats + time_feats_orig + time_feats_agg
    
    X = aggregated_df[all_features].fillna(aggregated_df[all_features].median())
    y = aggregated_df[f"original_{target_column}"]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    rf = RandomForestRegressor(
        n_estimators=n_estimators,
        max_features=max_features,
        max_depth=max_depth,
        min_samples_split=int(min_samples_split),
        bootstrap=bootstrap,
        random_state=42
    )
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    return (r2,)

# =============================================================================
# GA Setup using DEAP
# =============================================================================
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("n_estimators", random.choice, [10, 30, 50, 100, 150, 200])
toolbox.register("max_features", random.choice, ['sqrt', 'log2'])
toolbox.register("max_depth", random.choice, [10, 20, 30, 50, 100])
toolbox.register("min_samples_split", random.choice, [2, 4, 6])
toolbox.register("bootstrap", random.choice, [True, False])

def create_individual():
    return [toolbox.n_estimators(), toolbox.max_features(), toolbox.max_depth(),
            toolbox.min_samples_split(), toolbox.bootstrap()]
toolbox.register("individual", tools.initIterate, creator.Individual, create_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", evaluate_rf, geometry_df=None, time_df=None,
                 adjacency_index=None, distance_matrix=None, time_feature_columns=None,
                 target_column=None)
toolbox.register("mate", tools.cxTwoPoint)
def mutate_individual(ind):
    ind[:] = create_individual()
    return (ind,)
toolbox.register("mutate", mutate_individual)
toolbox.register("select", tools.selTournament, tournsize=3)

# =============================================================================
# GA Optimization per Radius
# =============================================================================
def run_GA_for_radius(geometry_df, time_df, adjacency_index, distance_matrix,
                      time_feature_columns, target_column):
    global current_radius
    pop = toolbox.population(n=POP_SIZE)
    no_improve_count = 0
    best_overall_r2 = -np.inf
    best_overall_individual = None
    
    for gen in range(GENS):
        fitnesses = []
        for ind in tqdm(pop, desc=f"Generation {gen+1} (radius {current_radius})", leave=False):
            fit = toolbox.evaluate(ind, geometry_df=geometry_df, time_df=time_df,
                                     adjacency_index=adjacency_index, distance_matrix=distance_matrix,
                                     time_feature_columns=time_feature_columns, target_column=target_column)
            fitnesses.append(fit)
            ind.fitness.values = fit
        r2_scores = [ind.fitness.values[0] for ind in pop]
        gen_best_r2 = max(r2_scores)
        gen_best_individual = pop[r2_scores.index(gen_best_r2)]
        
        if gen_best_r2 - best_overall_r2 > improvement_threshold:
            best_overall_r2 = gen_best_r2
            best_overall_individual = toolbox.clone(gen_best_individual)
            no_improve_count = 0
        else:
            no_improve_count += 1
        
        if no_improve_count >= max_no_improve:
            print(f"Early stopping at generation {gen+1} for radius {current_radius}")
            break
        
        offspring = toolbox.select(pop, len(pop))
        offspring = list(map(toolbox.clone, offspring))
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CXPB:
                toolbox.mate(child1, child2)
                del child1.fitness.values, child2.fitness.values
        for mutant in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(mutant)
                del mutant.fitness.values
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        for ind in invalid_ind:
            fit = toolbox.evaluate(ind, geometry_df=geometry_df, time_df=time_df,
                                     adjacency_index=adjacency_index, distance_matrix=distance_matrix,
                                     time_feature_columns=time_feature_columns, target_column=target_column)
            ind.fitness.values = fit
        pop[:] = offspring
    return best_overall_individual, best_overall_r2

# =============================================================================
# Main GA Experiment Loop
# =============================================================================
def main():
    global current_radius, current_exp_type
    # Ensure globals are updated
    current_radius = None
    current_exp_type = None
    
    # Load data
    distance_matrix, edge_attributes, timebased_data = load_data()
    adjacency_index = compute_adjacency_index(distance_matrix)
    print("Adjacency index computed.")
    geometry_df = build_geometry_df(edge_attributes)
    time_df = build_time_df(timebased_data)
    
    # Get experiment-specific parameters
    exp_params = EXPERIMENTS[EXPERIMENT_MODE]
    target_column = exp_params["target_column"]
    time_feature_columns = exp_params["time_feature_columns"]
    
    # Dictionary to store GA results by aggregation type and radius
    results = {agg_type: {} for agg_type in AGGREGATION_TYPES}
    
    # Loop over aggregation types (1: simple, 4: weighted)
    for agg_type in AGGREGATION_TYPES:
        current_exp_type = agg_type  # update global current_exp_type
        print(f"\nRunning GA for aggregation type {agg_type} ({'Simple' if agg_type==1 else 'Weighted'}) - {EXPERIMENT_MODE} experiment")
        for radius in radii_list:
            current_radius = radius  # update global current_radius
            print(f"\nOptimizing for radius = {radius} meters")
            best_individual, best_r2 = run_GA_for_radius(
                geometry_df=geometry_df,
                time_df=time_df,
                adjacency_index=adjacency_index,
                distance_matrix=distance_matrix,
                time_feature_columns=time_feature_columns,
                target_column=target_column
            )
            print(f"Radius: {radius}, Best R²: {best_r2:.4f}, Best Hyperparameters: {best_individual}")
            results[agg_type][radius] = {
                "best_individual": best_individual,
                "best_r2": best_r2
            }
    
    # Plotting: R² vs. Radius for both aggregation types
    plt.figure(figsize=(10, 6))
    for agg_type in AGGREGATION_TYPES:
        radii_sorted = sorted(results[agg_type].keys())
        r2_scores = [results[agg_type][r]["best_r2"] for r in radii_sorted]
        label = "Simple Avg" if agg_type == 1 else "Weighted Avg"
        plt.plot(radii_sorted, r2_scores, marker='o', label=label)
    plt.xlabel("Radius (meters)")
    plt.ylabel("Best R² Score")
    plt.title(f"Best R² Score vs. Radius ({EXPERIMENT_MODE.capitalize()} Experiment)")
    plt.legend()
    plt.grid(True)
    plt.show()
    
    # Print summary table
    summary_rows = []
    for agg_type in AGGREGATION_TYPES:
        for radius in sorted(results[agg_type].keys()):
            best_ind = results[agg_type][radius]["best_individual"]
            r2_val = results[agg_type][radius]["best_r2"]
            summary_rows.append({
                "Aggregation Type": "Simple" if agg_type == 1 else "Weighted",
                "Radius (m)": radius,
                "Best R²": f"{r2_val:.4f}",
                "Hyperparameters": best_ind
            })
    summary_df = pd.DataFrame(summary_rows)
    print("\nSummary of Best Hyperparameters per Radius and Aggregation Type:")
    print(summary_df.to_string(index=False))
    
if __name__ == "__main__":
    main()


In [ ]:
## APELDOORN BIKE CANOPY RADIUS-BASED SELECTION, WEIGHTED MEANS AGGREGATION


import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from collections import defaultdict
from deap import base, creator, tools
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# =============================================================================
# Global Configuration
# =============================================================================

OUTPUT_DIR = "./precomputed_matrices"

# Geometry-based (time-invariant) columns
GEOMETRY_COLUMNS = [
    "Bld_densit", "Veg_densit", "HW_Ratio", "Road_width",
    "Pred_LandUse", "Mean_population", "Mean_Bld_E",
    "Max_Bld_He", "Water_dens", "Pred_facade_material"
]

# Experiment definitions (each with its own target and time-dependent features)
EXPERIMENTS = {
    "surface": {
        "target_column": "Delta_surface",
        "time_feature_columns": ["Ref_surface_temp", "Mean_RH", "Ref_RH", "Mean_canopy_temp", "Ref_canopy_temp"]
    },
    "canopy": {
        "target_column": "Delta_canopy",
        "time_feature_columns": ["Ref_canopy_temp", "Mean_RH", "Ref_RH", "Mean_surface_temp", "Ref_surface_temp"]
    }
}

# Set experiment mode ("surface" or "canopy")
EXPERIMENT_MODE = "canopy"  # Change to "canopy" for the canopy experiment

# GA settings for Random Forest hyperparameter optimization
POP_SIZE = 50
GENS = 10
CXPB = 0.8
MUTPB = 0.5
improvement_threshold = 0.01
max_no_improve = 5

# List of radii (in meters) to test
radii_list = [0, 100, 250, 500, 750, 1000, 1500, 2000, 3000, 4000, 4500, 5140]

# We only keep the weighted average experiment (experiment type 4)
current_exp_type = 4  # Always weighted average

# Global variables to be updated by the GA evaluation function
current_radius = None  # Current radius threshold

# =============================================================================
# Helper Functions: Normalization & Adjacency Index
# =============================================================================
def normalize_matrix(matrix):
    """Normalize a matrix to the range [0,1]."""
    min_val, max_val = np.min(matrix), np.max(matrix)
    if min_val == max_val:
        return np.zeros_like(matrix)
    return (matrix - min_val) / (max_val - min_val)

def compute_adjacency_index(distance_matrix):
    """Compute the adjacency index as (1 - normalized distance matrix)."""
    distance_normalized = normalize_matrix(distance_matrix)
    return 1 - distance_normalized

# =============================================================================
# Data Loading and DataFrame Building
# =============================================================================
def load_data():
    print("Loading matrices and attributes...")
    # Load distance matrix
    distance_matrix = np.load(f"{OUTPUT_DIR}/distance_matrix_Bike.npy")
    # Load geometry-based attributes (list of dicts)
    with open(f"{OUTPUT_DIR}/edge_attributes_Bike.pkl", "rb") as f:
        edge_attributes = pickle.load(f)
    # Load time-based attributes (list of dicts, one per time step)
    with open(f"{OUTPUT_DIR}/timebased_attributes_Bike.pkl", "rb") as f:
        timebased_data = pickle.load(f)
    print("Data loaded successfully.")
    return distance_matrix, edge_attributes, timebased_data

def build_geometry_df(edge_attributes):
    """Convert geometry-based attributes into a DataFrame indexed by segment_id."""
    df_list = []
    for seg_attr in edge_attributes:
        sid = seg_attr["segment_id"]
        row_dict = {"segment_id": sid}
        for col in GEOMETRY_COLUMNS:
            row_dict[col] = seg_attr[col]
        df_list.append(row_dict)
    return pd.DataFrame(df_list).set_index("segment_id")

def build_time_df(timebased_data):
    """Convert time-based attributes into a DataFrame."""
    return pd.DataFrame(timebased_data)

# =============================================================================
# Aggregation Function: Weighted Average with Local Weighting Only
# =============================================================================
def aggregate_time_step_features(time_df, geometry_df, distance_matrix,
                                 radius_threshold, time_feature_columns, target_column):
    """
    For each row in time_df, aggregate neighbor features from segments within radius_threshold
    using a locally weighted average based solely on distance.
    The weight for each neighbor is computed by normalizing the distances locally (using the maximum
    distance among the selected neighbors) so that closer neighbors receive higher weight.
    Returns a DataFrame with:
      - segment_id, Date, Part_of_the_day,
      - geometry-based features,
      - original_{time_feature_columns},
      - weighted_avg_{time_feature_columns}_neighbors,
      - and original target (prefixed with "original_").
    """
    # Build lookup for time-based rows: (segment_id, Date, Part_of_the_day) -> row
    time_lookup = {}
    for i, row in time_df.iterrows():
        key = (row["segment_id"], row["Date"], row["Part_of_the_day"])
        time_lookup[key] = row

    results = []
    for i, row in time_df.iterrows():
        seg_id = row["segment_id"]
        date_val = row["Date"]
        part_day = row["Part_of_the_day"]

        # Get geometry features
        geom_feats = geometry_df.loc[seg_id].to_dict()

        # Get original time-based features
        time_feats = {}
        for col in time_feature_columns:
            time_feats[f"original_{col}"] = row[col]
        time_feats[f"original_{target_column}"] = row.get(target_column, np.nan)

        # Identify neighbor segments using distance_matrix (assumes segment_id is an integer index)
        seg_index = seg_id
        neighbor_ids = np.where(distance_matrix[seg_index] <= radius_threshold)[0]
        neighbor_ids = [nid for nid in neighbor_ids if nid != seg_index]

        aggregated = {}
        if len(neighbor_ids) > 0:
            neighbor_matrix = []
            for nid in neighbor_ids:
                key_neighbor = (nid, date_val, part_day)
                if key_neighbor not in time_lookup:
                    neighbor_matrix.append([np.nan] * len(time_feature_columns))
                else:
                    neighbor_row = time_lookup[key_neighbor]
                    neighbor_matrix.append([neighbor_row[col] for col in time_feature_columns])
            neighbor_matrix = np.array(neighbor_matrix, dtype=float)
            for col_idx, col_name in enumerate(time_feature_columns):
                col_values = neighbor_matrix[:, col_idx]
                if np.isnan(col_values).all():
                    agg_val = np.nan
                else:
                    # Local weighting: compute weights based on distances of only the selected neighbors
                    local_distance = distance_matrix[seg_index, neighbor_ids]
                    local_max_distance = np.max(local_distance)
                    if local_max_distance > 0:
                        norm_distance_local = 1 - local_distance / local_max_distance
                    else:
                        norm_distance_local = np.ones_like(local_distance)
                    weights = norm_distance_local
                    valid_mask = ~np.isnan(col_values)
                    if valid_mask.any():
                        valid_weights = weights[valid_mask]
                        w_sum = valid_weights.sum()
                        if w_sum > 0:
                            valid_weights = valid_weights / w_sum
                        else:
                            valid_weights = np.ones_like(valid_weights) / len(valid_weights)
                        agg_val = np.average(col_values[valid_mask], weights=valid_weights)
                    else:
                        agg_val = np.nan
                    aggregated[f"weighted_avg_{col_name}_neighbors"] = agg_val
        else:
            for col_name in time_feature_columns:
                aggregated[f"weighted_avg_{col_name}_neighbors"] = np.nan

        out_row = {"segment_id": seg_id, "Date": date_val, "Part_of_the_day": part_day}
        out_row.update(geom_feats)
        out_row.update(time_feats)
        out_row.update(aggregated)
        results.append(out_row)
    return pd.DataFrame(results)

# =============================================================================
# GA Evaluation Function for Random Forest (Weighted Average Only)
# =============================================================================
def evaluate_rf(individual, geometry_df, time_df, distance_matrix, time_feature_columns, target_column):
    global current_radius
    n_estimators = individual[0]
    max_features = individual[1]
    max_depth = individual[2]
    min_samples_split = individual[3]
    bootstrap = bool(individual[4])
    
    aggregated_df = aggregate_time_step_features(
        time_df=time_df,
        geometry_df=geometry_df,
        distance_matrix=distance_matrix,
        radius_threshold=current_radius,
        time_feature_columns=time_feature_columns,
        target_column=target_column
    )
    aggregated_df = aggregated_df.dropna(subset=[f"original_{target_column}"])
    
    geometry_feats = GEOMETRY_COLUMNS
    time_feats_orig = [f"original_{col}" for col in time_feature_columns]
    time_feats_agg = [f"weighted_avg_{col}_neighbors" for col in time_feature_columns]
    all_features = geometry_feats + time_feats_orig + time_feats_agg
    
    X = aggregated_df[all_features].fillna(aggregated_df[all_features].median())
    y = aggregated_df[f"original_{target_column}"]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    rf = RandomForestRegressor(
        n_estimators=n_estimators,
        max_features=max_features,
        max_depth=max_depth,
        min_samples_split=int(min_samples_split),
        bootstrap=bootstrap,
        random_state=42
    )
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    return (r2,)

# =============================================================================
# GA Setup using DEAP
# =============================================================================
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("n_estimators", random.choice, [10, 30, 50, 100, 150, 200])
toolbox.register("max_features", random.choice, ['sqrt', 'log2'])
toolbox.register("max_depth", random.choice, [10, 20, 30, 50, 100])
toolbox.register("min_samples_split", random.choice, [2, 4, 6])
toolbox.register("bootstrap", random.choice, [True, False])

def create_individual():
    return [
        toolbox.n_estimators(),
        toolbox.max_features(),
        toolbox.max_depth(),
        toolbox.min_samples_split(),
        toolbox.bootstrap()
    ]
toolbox.register("individual", tools.initIterate, creator.Individual, create_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", evaluate_rf, geometry_df=None, time_df=None,
                 distance_matrix=None, time_feature_columns=None, target_column=None)
toolbox.register("mate", tools.cxTwoPoint)
def mutate_individual(ind):
    ind[:] = create_individual()
    return (ind,)
toolbox.register("mutate", mutate_individual)
toolbox.register("select", tools.selTournament, tournsize=3)

# =============================================================================
# GA Optimization per Radius
# =============================================================================
def run_GA_for_radius(geometry_df, time_df, distance_matrix, time_feature_columns, target_column):
    global current_radius
    pop = toolbox.population(n=POP_SIZE)
    no_improve_count = 0
    best_overall_r2 = -np.inf
    best_overall_individual = None
    
    for gen in range(GENS):
        fitnesses = []
        for ind in tqdm(pop, desc=f"Generation {gen+1} (radius {current_radius})", leave=False):
            fit = toolbox.evaluate(ind, geometry_df=geometry_df, time_df=time_df,
                                     distance_matrix=distance_matrix,
                                     time_feature_columns=time_feature_columns,
                                     target_column=target_column)
            fitnesses.append(fit)
            ind.fitness.values = fit
        r2_scores = [ind.fitness.values[0] for ind in pop]
        gen_best_r2 = max(r2_scores)
        gen_best_individual = pop[r2_scores.index(gen_best_r2)]
        
        if gen_best_r2 - best_overall_r2 > improvement_threshold:
            best_overall_r2 = gen_best_r2
            best_overall_individual = toolbox.clone(gen_best_individual)
            no_improve_count = 0
        else:
            no_improve_count += 1
        
        if no_improve_count >= max_no_improve:
            print(f"Early stopping at generation {gen+1} for radius {current_radius}")
            break
        
        offspring = toolbox.select(pop, len(pop))
        offspring = list(map(toolbox.clone, offspring))
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CXPB:
                toolbox.mate(child1, child2)
                del child1.fitness.values, child2.fitness.values
        for mutant in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(mutant)
                del mutant.fitness.values
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        for ind in invalid_ind:
            fit = toolbox.evaluate(ind, geometry_df=geometry_df, time_df=time_df,
                                     distance_matrix=distance_matrix,
                                     time_feature_columns=time_feature_columns,
                                     target_column=target_column)
            ind.fitness.values = fit
        pop[:] = offspring
    return best_overall_individual, best_overall_r2

# =============================================================================
# Main GA Experiment Loop
# =============================================================================
def main():
    global current_radius
    # Load data
    distance_matrix, edge_attributes, timebased_data = load_data()
    # Use local weighting: do not use global adjacency index computed from entire matrix.
    # (We compute local weights directly in the aggregation function.)
    geometry_df = build_geometry_df(edge_attributes)
    time_df = build_time_df(timebased_data)
    
    # Get experiment-specific parameters for the chosen experiment mode.
    exp_params = EXPERIMENTS[EXPERIMENT_MODE]
    target_column = exp_params["target_column"]
    time_feature_columns = exp_params["time_feature_columns"]
    
    # Dictionary to store GA results by radius.
    results = {}
    
    print(f"\nStarting GA Optimization for Montreal ({EXPERIMENT_MODE.capitalize()} Experiment, Weighted Average with Local Weighting)")
    for radius in radii_list:
        current_radius = radius  # update global current_radius
        print(f"\nOptimizing for radius = {radius} meters")
        best_individual, best_r2 = run_GA_for_radius(
            geometry_df=geometry_df,
            time_df=time_df,
            distance_matrix=distance_matrix,
            time_feature_columns=time_feature_columns,
            target_column=target_column
        )
        print(f"Radius: {radius}, Best R²: {best_r2:.4f}, Best Hyperparameters: {best_individual}")
        results[radius] = {
            "best_individual": best_individual,
            "best_r2": best_r2
        }
    
    # Plotting: R² vs. Radius
    plt.figure(figsize=(10, 6))
    radii_sorted = sorted(results.keys())
    r2_scores = [results[r]["best_r2"] for r in radii_sorted]
    plt.plot(radii_sorted, r2_scores, marker='o', linestyle='-')
    plt.xlabel("Radius (meters)")
    plt.ylabel("Best R² Score")
    plt.title(f"Best R² Score vs. Radius ({EXPERIMENT_MODE.capitalize()} Experiment)")
    plt.grid(True)
    plt.show()
    
    # Print summary table
    summary_rows = []
    for radius in sorted(results.keys()):
        best_ind = results[radius]["best_individual"]
        r2_val = results[radius]["best_r2"]
        summary_rows.append({
            "Radius (m)": radius,
            "Best R²": f"{r2_val:.4f}",
            "Hyperparameters": best_ind
        })
    summary_df = pd.DataFrame(summary_rows)
    print("\nSummary of Best Hyperparameters per Radius:")
    print(summary_df.to_string(index=False))
    
if __name__ == "__main__":
    main()

### Feature importance

In [ ]:
## FEATURE IMPORTANCE SENSITIVITY WEIGHTED MEANS

"""
Compute and plot how feature importances vary with the radius threshold.
For each radius, we train a Random Forest (Experiment 4) using weighted features
and then extract the feature importances.
"""

import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split

# --- Global Variables ---
OUTPUT_DIR = "./precomputed_matrices"
FEATURE_COLUMNS = [
    'Bld_densit', 'Veg_densit', 'HW_Ratio', 'Width', 'Pred_Facad',
    'Pred_LandU', 'Mean_Popul', 'Mean_Bld_E', 'Max_Bld_He',
    'Water_dens', 'Bike_pct_a', 'Vehicle_pc'
]
TARGET_COLUMN = 'Delta_T'

# Experiment 4 parameters: using a weighted average of neighbor features
exp4_alpha = 0
exp4_beta = 1 - exp4_alpha
rf_params_exp4 = {
    "n_estimators": 200,
    "max_features": "log2",
    "max_depth": 100,
    "min_samples_split": 2,
    "bootstrap": False,
    "random_state": 42
}

# --- Helper Functions ---
def normalize_matrix(matrix):
    """Normalize a matrix to the range [0,1]."""
    return (matrix - np.min(matrix)) / (np.max(matrix) - np.min(matrix))

def compute_adjacency_index(shortest_path_matrix, distance_matrix, alpha, beta):
    """
    Compute the adjacency index using a weighted combination of the normalized 
    (inverted) shortest path and distance matrices.
    """
    shortest_path_normalized = normalize_matrix(shortest_path_matrix)
    distance_normalized = normalize_matrix(distance_matrix)
    adjacency_index = alpha * (1 - shortest_path_normalized) + beta * (1 - distance_normalized)
    return adjacency_index

def aggregate_features_with_adjacency(edge_attributes, adjacency_index, distance_matrix, feature_columns, radius):
    """
    For each segment, aggregates neighbor features using a weighted average based on the adjacency index.
    Neighbors are selected using the provided distance matrix and radius.
    Returns a DataFrame with both the original features and the weighted neighbor features.
    """
    aggregated_features = defaultdict(list)
    for segment_index, segment_attrs in enumerate(edge_attributes):
        # Get original features for the segment.
        segment_features = {col: segment_attrs.get(col, np.nan) for col in feature_columns}
        # Identify neighbor indices within the given radius (excluding the segment itself).
        neighbor_indices = np.where(distance_matrix[segment_index] <= radius)[0]
        neighbors = [n for n in neighbor_indices if n != segment_index]

        if neighbors:
            neighbor_features = np.array([
                [edge_attributes[n].get(col, np.nan) for col in feature_columns]
                for n in neighbors
            ])
            weights = adjacency_index[segment_index, neighbors]
            weight_sum = np.sum(weights)
            if weight_sum != 0:
                weights = weights / weight_sum
            else:
                weights = np.ones_like(weights) / len(weights)
            aggregated = {
                f"weighted_avg_{col}_neighbors": np.average(neighbor_features[:, i], weights=weights)
                for i, col in enumerate(feature_columns)
            }
        else:
            aggregated = {f"weighted_avg_{col}_neighbors": np.nan for col in feature_columns}

        # Store both the original and the weighted features.
        for col in feature_columns:
            aggregated_features[f"original_{col}"].append(segment_features[col])
            aggregated_features[f"weighted_avg_{col}_neighbors"].append(aggregated[f"weighted_avg_{col}_neighbors"])

    return pd.DataFrame(aggregated_features)

def train_rf(X, y, rf_params):
    """Train a Random Forest Regressor and return the fitted model along with performance metrics."""
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    rf = RandomForestRegressor(**rf_params)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    print(f"RF Performance: R² = {r2:.4f}, MSE = {mse:.4f}")
    return rf

# --- Main Execution ---
if __name__ == "__main__":
    print("Loading precomputed matrices and dataset...")

    # Load the precomputed matrices
    distance_matrix = np.load(f"{OUTPUT_DIR}/distance_matrix.npy")
    shortest_path_matrix = np.load(f"{OUTPUT_DIR}/shortest_path_matrix.npy")
    
    # Load the edge attributes (list of dictionaries with features and target)
    with open(f"{OUTPUT_DIR}/edge_attributes.pkl", "rb") as f:
        edge_attributes = pickle.load(f)

    # Compute the adjacency index (used for weighted feature aggregation)
    adjacency_index = compute_adjacency_index(shortest_path_matrix, distance_matrix, exp4_alpha, exp4_beta)
    print("Adjacency index computed.")

    # Prepare the target variable (Delta_T)
    y = [attrs[TARGET_COLUMN] for attrs in edge_attributes]

    # Define the radii: from 10 to 500 (step 50) and 500 to 2000 (step 250)
    radii_first = np.arange(10, 500, 50)      # e.g., 10, 60, 110, ... 460
    radii_second = np.arange(500, 3000 + 1, 250)  # e.g., 500, 750, 1000, ... 3000
    radii_third = np.arange(3500, 8800, 350)
    radii = np.unique(np.concatenate((radii_first, radii_second,radii_third)))
    print("Radii to be evaluated:", radii)

    # Define the feature names used in Experiment 4.
    feature_names = [f"original_{col}" for col in FEATURE_COLUMNS] + \
                    [f"weighted_avg_{col}_neighbors" for col in FEATURE_COLUMNS]

    # Dictionary to store feature importances for each radius.
    importance_results = {}

    # Loop over each radius, train the model, and record the feature importances.
    for radius in radii:
        print(f"\nProcessing radius = {radius} meters")
        aggregated_data = aggregate_features_with_adjacency(edge_attributes, adjacency_index, distance_matrix, FEATURE_COLUMNS, radius)
        X = aggregated_data[feature_names]
        model = train_rf(X, y, rf_params_exp4)
        importance_results[radius] = model.feature_importances_

    # Convert the collected importances into a DataFrame:
    #   Rows: radii, Columns: feature names.
    importance_df = pd.DataFrame(importance_results, index=feature_names).T
    print("\nFeature Importances Across Radii:")
    print(importance_df)

    # --- Plotting the Results ---
    # We split the plot into two panels: one for the original features and one for the weighted neighbor features.
    original_features = [f"original_{col}" for col in FEATURE_COLUMNS]
    weighted_features = [f"weighted_avg_{col}_neighbors" for col in FEATURE_COLUMNS]


fig, axs = plt.subplots(2, 1, figsize=(12, 12), sharex=True)

# Plot original feature importances.
for feat in original_features:
    axs[0].plot(importance_df.index, importance_df[feat], marker='o', label=feat)
axs[0].set_title("Original Feature Importances")
axs[0].set_ylabel("Importance")
axs[0].legend(loc='upper right', fontsize='small')  # Keep legend inside but positioned neatly

# Plot weighted neighbor feature importances.
for feat in weighted_features:
    axs[1].plot(importance_df.index, importance_df[feat], marker='o', label=feat)
axs[1].set_title("Weighted Neighbor Feature Importances")
axs[1].set_xlabel("Radius (meters)")
axs[1].set_ylabel("Importance")

# Move only the second legend outside of plot
axs[1].legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize='small')

# Force x-axis ticks to be visible in the first plot
axs[0].tick_params(axis='x', which='both', labelbottom=True)

plt.tight_layout()
plt.show()


In [ ]:
## FEATURE IMPORTANCE SENSITIVITY SEPARATE MEANS

"""
FEATURE IMPORTANCE SENSITIVITY EX 1

Compute and plot how feature importances vary with the radius threshold.
For each radius, we train a Random Forest (Experiment 1) using aggregated features,
where the neighbor features are computed as the simple mean of neighbors within the radius.
"""

import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split

# --- Global Variables ---
OUTPUT_DIR = "./precomputed_matrices"
FEATURE_COLUMNS = [
    'Bld_densit', 'Veg_densit', 'HW_Ratio', 'Width', 'Pred_Facad',
    'Pred_LandU', 'Mean_Popul', 'Mean_Bld_E', 'Max_Bld_He',
    'Water_dens', 'Bike_pct_a', 'Vehicle_pc'
]
TARGET_COLUMN = 'Delta_T'

# Experiment 1 hyperparameters: using segment + neighbor features (simple average)
rf_params_exp1 = {
    "n_estimators": 100,
    "max_features": "sqrt",
    "max_depth": 50,
    "min_samples_split": 4,
    "bootstrap": False,
    "random_state": 42
}

# --- Helper Functions ---

def aggregate_features_with_distance(edge_attributes, distance_matrix, feature_columns, radius):
    """
    Aggregate features of a segment and its neighbors (selected using the distance threshold)
    by computing the mean of the neighbor features.
    
    For each segment:
      - The original feature values are stored.
      - The neighbor features (for all segments within the given radius, excluding itself)
        are averaged (using a simple mean).
      
    Returns a DataFrame containing:
      - original_<feature>: the original feature value.
      - avg_<feature>_neighbors: the average of neighbor feature values.
    """
    aggregated_features = defaultdict(list)
    for segment_index, segment_attrs in enumerate(edge_attributes):
        # Get original features for this segment.
        segment_features = {col: segment_attrs.get(col, np.nan) for col in feature_columns}
        # Identify neighbors within the given radius (excluding the segment itself).
        neighbor_indices = np.where(distance_matrix[segment_index] <= radius)[0]
        neighbors = [n for n in neighbor_indices if n != segment_index]

        if neighbors:
            neighbor_features = np.array([
                [edge_attributes[n].get(col, np.nan) for col in feature_columns]
                for n in neighbors
            ])
            # Compute the simple (unweighted) mean of the neighbors for each feature.
            aggregated = {
                f"avg_{col}_neighbors": np.nanmean(neighbor_features[:, i])
                for i, col in enumerate(feature_columns)
            }
        else:
            aggregated = {f"avg_{col}_neighbors": np.nan for col in feature_columns}

        # Save both the original and aggregated neighbor features.
        for col in feature_columns:
            aggregated_features[f"original_{col}"].append(segment_features[col])
            aggregated_features[f"avg_{col}_neighbors"].append(aggregated[f"avg_{col}_neighbors"])

    return pd.DataFrame(aggregated_features)

def train_rf(X, y, rf_params):
    """
    Train a Random Forest Regressor and return the fitted model along with performance metrics.
    """
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    rf = RandomForestRegressor(**rf_params)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    print(f"RF Performance: R² = {r2:.4f}, MSE = {mse:.4f}")
    return rf

# --- Main Execution ---
if __name__ == "__main__":
    print("Loading precomputed matrices and dataset...")

    # Load the precomputed distance matrix
    distance_matrix = np.load(f"{OUTPUT_DIR}/distance_matrix.npy")
    
    # Load the edge attributes (list of dictionaries with features and target)
    with open(f"{OUTPUT_DIR}/edge_attributes.pkl", "rb") as f:
        edge_attributes = pickle.load(f)

    # Prepare the target variable (Delta_T)
    y = [attrs[TARGET_COLUMN] for attrs in edge_attributes]

    # Define the radii to be evaluated.
    # Example: 10 to 500 with a step of 50, 500 to 3000 with a step of 250, and 3500 to 8800 with a step of 350.
    radii_first = np.arange(10, 500, 50)       # 10, 60, 110, ... 460
    radii_second = np.arange(500, 3000 + 1, 250) # 500, 750, 1000, ... 3000
    radii_third = np.arange(3500, 8800, 350)     # 3500, 3850, ... etc.
    radii = np.unique(np.concatenate((radii_first, radii_second, radii_third)))
    print("Radii to be evaluated:", radii)

    # Define the feature names for Experiment 1.
    # For each feature, we have the original value and the averaged neighbor value.
    feature_names = [f"original_{col}" for col in FEATURE_COLUMNS] + \
                    [f"avg_{col}_neighbors" for col in FEATURE_COLUMNS]

    # Dictionary to store feature importances for each radius.
    importance_results = {}

    # Loop over each radius, train the model, and record the feature importances.
    for radius in radii:
        print(f"\nProcessing radius = {radius} meters")
        aggregated_data = aggregate_features_with_distance(edge_attributes, distance_matrix, FEATURE_COLUMNS, radius)
        X = aggregated_data[feature_names]
        model = train_rf(X, y, rf_params_exp1)
        importance_results[radius] = model.feature_importances_

    # Convert the collected importances into a DataFrame:
    # Rows: radii, Columns: feature names.
    importance_df = pd.DataFrame(importance_results, index=feature_names).T
    print("\nFeature Importances Across Radii:")
    print(importance_df)

    # --- Plotting the Results ---
    # Split the plot into two panels: one for the original features and one for the averaged neighbor features.
    original_features = [f"original_{col}" for col in FEATURE_COLUMNS]
    avg_features = [f"avg_{col}_neighbors" for col in FEATURE_COLUMNS]

    fig, axs = plt.subplots(2, 1, figsize=(12, 12), sharex=True)

    # Plot original feature importances.
    for feat in original_features:
        axs[0].plot(importance_df.index, importance_df[feat], marker='o', label=feat)
    axs[0].set_title("Original Feature Importances")
    axs[0].set_ylabel("Importance")
    axs[0].legend(loc='upper right', fontsize='small')

    # Plot averaged neighbor feature importances.
    for feat in avg_features:
        axs[1].plot(importance_df.index, importance_df[feat], marker='o', label=feat)
    axs[1].set_title("Averaged Neighbor Feature Importances")
    axs[1].set_xlabel("Radius (meters)")
    axs[1].set_ylabel("Importance")
    axs[1].legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize='small')

    plt.tight_layout()
    plt.show()


### Visualisations

In [ ]:
## RADIUS BASED NEIGHBOUR SELECTION APELDOORN SATELLITE

import matplotlib.pyplot as plt
import numpy as np

# Updated data from experiments
experiments = {
    "Separate Means": {
        0: 0.6786, 100: 0.7762, 200: 0.8483, 300: 0.8881, 400: 0.9100, 500: 0.9224, 600: 0.9278, 700: 0.9297, 1000: 0.9373, 
        2000: 0.9474, 3000: 0.9480, 4000: 0.9414, 5000: 0.9264, 6000: 0.8731, 7000: 0.7245, 8000: 0.6853, 
        8500: 0.6794, 8700: 0.6782
    },
    "United Means": {
        0: 0.6786, 100: 0.7537, 200: 0.7971, 300: 0.8333, 400: 0.8544, 500: 0.8558, 600: 0.8526, 700: 0.8650, 1000: 0.8708, 
        2000: 0.8847, 3000: 0.8989, 4000: 0.8860, 5000: 0.8811, 6000: 0.8414, 7000: 0.7199, 8000: 0.6813, 
        8500: 0.6734, 8700: 0.6744
    },
    "PCA Means": {
        0: 0.6786, 100: 0.7415, 200: 0.7898, 300: 0.8213, 400: 0.8445, 500: 0.8454, 600: 0.8506, 700: 0.8622, 1000: 0.8682, 
        2000: 0.8838, 3000: 0.9001, 4000: 0.8881, 5000: 0.8816, 6000: 0.8430, 7000: 0.7232, 8000: 0.6801, 
        8500: 0.6755, 8700: 0.6753
    },
    "Weighted Means": {
        0: 0.6786, 100: 0.7568, 200: 0.8460, 300: 0.8869, 400: 0.9095, 500: 0.9271, 600: 0.9352, 700: 0.9405, 1000: 0.9500, 
        2000: 0.9585, 3000: 0.9582, 4000: 0.9585, 5000: 0.9578, 6000: 0.9586, 7000: 0.9569, 8000: 0.9560, 
        8500: 0.9562, 8700: 0.9552
    }
}

# Define line styles and markers for differentiation
line_styles = ['--', '-.', '-', ':']
markers = ['o', 's', 'd', '^']  # Circle, Square, Diamond, Triangle

# Create the plot
plt.figure(figsize=(10, 6))

# Plot each experiment with different line styles and markers
for (label, data), line_style, marker in zip(experiments.items(), line_styles, markers):
    nodes = list(data.keys())
    r2_values = list(data.values())
    plt.plot(nodes, r2_values, linestyle=line_style, marker=marker, markersize=5, color='black', label=label)

# Labels and legend
plt.xlabel("Radius")
plt.ylabel("R²")
plt.title("Sensitivity of R² to the spatial scale: Radius-based neighbour selection")
plt.legend()
plt.grid(True)

# Show the plot
plt.show()

In [ ]:
## NODE-BASED NEIGHBOUR SELECTION APELDOORN SATELLITE

import matplotlib.pyplot as plt

# Updated data from experiments
experiments = {
    "Separate Means": {
        0: 0.7307, 1: 0.7826, 2: 0.8221, 4: 0.8692, 6: 0.8905, 8: 0.9003, 10: 0.9082, 12: 0.9136, 14: 0.9135, 
        16: 0.9163, 18: 0.9152, 20: 0.9151, 36: 0.9108, 52: 0.9050, 68: 0.8912, 84: 0.8842, 100: 0.8772, 
        125: 0.8567, 150: 0.7370, 175: 0.7038, 200: 0.6832, 205: 0.6831, 210: 0.6779, 215: 0.6793
    },
    "United Means": {
        0: 0.7211, 1: 0.7626, 2: 0.7829, 4: 0.8146, 6: 0.8344, 8: 0.8457, 10: 0.8473, 12: 0.8569, 
        14: 0.8626, 16: 0.8597, 18: 0.8595, 20: 0.8566, 36: 0.8592, 52: 0.8645, 68: 0.8477, 84: 0.8368, 
        100: 0.8371, 125: 0.8199, 150: 0.7388, 175: 0.7002, 200: 0.6801, 205: 0.6796, 210: 0.6770, 215: 0.6718
    },
    "PCA Means": {
        0: 0.7171, 1: 0.7428, 2: 0.7684, 4: 0.8007, 6: 0.8154, 8: 0.8308, 10: 0.8425, 12: 0.8533, 
        14: 0.8577, 16: 0.8553, 18: 0.8599, 20: 0.8557, 36: 0.8579, 52: 0.8659, 68: 0.8489, 84: 0.8321, 
        100: 0.8298, 125: 0.8102, 150: 0.7321, 175: 0.6950, 200: 0.6786, 205: 0.6771, 210: 0.6774, 215: 0.6726
    },
    "Weighted Means": {
        0: 0.7280, 1: 0.7689, 2: 0.7991, 4: 0.8649, 6: 0.8972, 8: 0.9093, 10: 0.9166, 12: 0.9271,
        14: 0.9297, 16: 0.9307, 18: 0.9287, 20: 0.9327, 36: 0.9360, 52: 0.9404, 68: 0.9453, 84: 0.9441,
        100: 0.9430, 125: 0.9485, 150: 0.9533, 175: 0.9540, 200: 0.9535, 205: 0.9546, 210: 0.9543, 215: 0.9551
    }
}

# Define line styles and markers for differentiation
line_styles = ['--', '-.', '-', ':']
markers = ['o', 's', 'd', '^']  # Circle, Square, Diamond, Triangle

# Create the plot
plt.figure(figsize=(10, 6))

# Plot each experiment with different line styles and markers
for (label, data), line_style, marker in zip(experiments.items(), line_styles, markers):
    nodes = list(data.keys())
    r2_values = list(data.values())
    plt.plot(nodes, r2_values, linestyle=line_style, marker=marker, markersize=5, color='black', label=label)

# Labels and legend
plt.xlabel("Number of Nodes")
plt.ylabel("R²")
plt.title("Sensitivity of R² to the spatial scale: Nodes-based neighbour selection")
plt.legend()
plt.grid(True)

# Show the plot
plt.show()


In [ ]:
## NEURAL NETWORK FAILURE GRAPH APELDOORN SATELLITE

import matplotlib.pyplot as plt
import numpy as np

# Data from experiments
experiments = {
    "Random Forest: Separate Means, Radius-based aggregation": {
        100: 0.7762, 500: 0.9224, 1000: 0.9373, 3000: 0.9480, 4000: 0.9414, 5000: 0.9264, 8000: 0.6853
    },
    "neural Network: Separate Means, Radius-based aggregation": {
        100: 0.7290, 500: 0.8796, 1000: 0.8855, 3000: 0.8475, 4000: 0.7845, 5000: 0.7114, 8000:  0.5957
    }
}

# Define line styles and markers for differentiation
line_styles = ['-', '--']
markers = ['o', '^']  # Circle, Square, Diamond, Triangle

# Create the plot
plt.figure(figsize=(10, 6))

# Plot each experiment with different line styles and markers
for (label, data), line_style, marker in zip(experiments.items(), line_styles, markers):
    nodes = list(data.keys())
    r2_values = list(data.values())
    plt.plot(nodes, r2_values, linestyle=line_style, marker=marker, markersize=5, color='black', label=label)

# Labels and legend
plt.xlabel("Radius")
plt.ylabel("R²")
plt.title("Comparison of RF and NN: Radius-based neighbour selection")
plt.legend()
plt.grid(True)

# Show the plot
plt.show()

In [ ]:
## ROTTERDAM TRANSFERABILITY TEST

import matplotlib.pyplot as plt

# Updated data for the experiments
experiments = {
    "Separate Means": {
        0: 0.3459,
        100: 0.4611,
        200: 0.6302,
        300: 0.7151,
        400: 0.7486,
        500: 0.7677,
        750: 0.8276,
        1000: 0.8501,
        1250: 0.8609,
        1500: 0.8682,
        1750: 0.8690,
        2000: 0.8781,
        2500: 0.8748,
        3000: 0.8726,
        3500: 0.8737,
        4000: 0.8551,
        5000: 0.8719,
        7000: 0.8812,
        9000: 0.8484,
        11000: 0.8357,
        13000: 0.7961,
        15000: 0.7792,
        20000: 0.7681,
        30000: 0.7264,
        43000: 0.3411
    },
    "Weighted Means": {
        0: 0.3460,
        100: 0.4819,
        200: 0.5891,
        300: 0.6963,
        400: 0.7483,
        500: 0.7710,
        750: 0.8320,
        1000: 0.8596,
        1250: 0.8768,
        1500: 0.8863,
        1750: 0.8931,
        2000: 0.8945,
        2500: 0.8989,
        3000: 0.8985,
        3500: 0.9026,
        4000: 0.8948,
        5000: 0.8994,
        7000: 0.9013,
        9000: 0.9098,
        11000: 0.9081,
        13000: 0.9075,
        15000: 0.9071,
        20000: 0.8984,
        30000: 0.8842,
        43000: 0.8910
    }
}

# Define grayscale colors, line styles, and markers for each experiment
colors = {
    "Separate Means": "#000000",  # black
    "Weighted Means": "#666666"   # dark gray
}
line_styles = ['--', '-']
markers = ['o', '^']

plt.figure(figsize=(10, 6))

for (label, data), ls, marker in zip(experiments.items(), line_styles, markers):
    # Ensure radii are sorted for proper plotting
    radii = sorted(data.keys())
    r2_values = [data[r] for r in radii]
    plt.plot(radii, r2_values, linestyle=ls, marker=marker, markersize=5,
             color=colors[label], label=label)

plt.xlabel("Radius")
plt.ylabel("Best R²")
plt.title("Sensitivity of R² to the Spatial Scale: Radius-based RF, Rotterdam")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
## MONTREAL TRANSFERABILITY TEST

import matplotlib.pyplot as plt

# Correct data for the Montreal experiments
experiments = {
    "Separate Means": {
        0: 0.5408,
        100: 0.6647,
        200: 0.7550,
        300: 0.7932,
        400: 0.8145,
        500: 0.8353,
        750: 0.8375,
        1000: 0.8307,
        1500: 0.8331,
        2000: 0.8313,
        3000: 0.8188,
        4000: 0.8137,
        5000: 0.8219,
        7000: 0.8319,
        10000: 0.6865,
        13000: 0.5737,
        14500: 0.5405
    },
    "Weighted Means": {
        0: 0.5250,
        100: 0.6306,
        200: 0.7355,
        300: 0.7851,
        400: 0.8180,
        500: 0.8338,
        750: 0.8579,
        1000: 0.8614,
        1500: 0.8613,
        2000: 0.8534,
        3000: 0.8652,
        4000: 0.8536,
        5000: 0.8447,
        7000: 0.8441,
        10000: 0.8516,
        13000: 0.8408,
        14500: 0.8417
    }
}

# Define grayscale colors, line styles, and markers for each experiment
colors = {
    "Separate Means": "#000000",  # black
    "Weighted Means": "#666666"   # dark gray
}
line_styles = ['--', '-']
markers = ['o', '^']

plt.figure(figsize=(10, 6))

for (label, data), ls, marker in zip(experiments.items(), line_styles, markers):
    radii = sorted(data.keys())
    r2_values = [data[r] for r in radii]
    plt.plot(radii, r2_values, linestyle=ls, marker=marker, markersize=5,
             color=colors[label], label=label)

plt.xlabel("Radius")
plt.ylabel("Best R²")
plt.title("Sensitivity of R² to the Spatial Scale: Radius-based RF, Montreal")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
## BIKE CANOPY

import matplotlib.pyplot as plt

# Updated data for the Canopy experiment
experiments = {
    "Separate Means": {
        100: 0.8876,
        250: 0.9024,
        500: 0.9196,
        750: 0.9230,
        1000: 0.9252,
        1500: 0.9249,
        2000: 0.9205,
        3000: 0.9247,
        4000: 0.9184,
        4500: 0.9213,
        5140: 0.9204
    },
    "Weighted Means": {
        0: 0.8345,
        100: 0.8863,
        250: 0.9015,
        500: 0.9236,
        750: 0.9263,
        1000: 0.9282,
        1500: 0.9271,
        2000: 0.9276,
        3000: 0.9249,
        4000: 0.9239,
        4500: 0.9246,
        5140: 0.9276
    }
}

# Define grayscale colors, line styles, and markers for each experiment
colors = {
    "Separate Means": "#000000",   # black
    "Weighted Means": "#666666"    # dark gray
}
line_styles = ['--', '-']
markers = ['o', '^']

# Create the plot
plt.figure(figsize=(10, 6))

for (label, data), ls, marker in zip(experiments.items(), line_styles, markers):
    radii = sorted(data.keys())
    r2_values = [data[r] for r in radii]
    plt.plot(radii, r2_values, linestyle=ls, marker=marker, markersize=6,
             color=colors[label], label=label)

plt.xlabel("Radius (m)")
plt.ylabel("Best R²")
plt.title("Sensitivity of R² to Spatial Scale: Canopy Experiment")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
## SURFACE BIKE

import matplotlib.pyplot as plt

# Updated data for the Surface Temperature experiment
experiments = {
    "Separate Means": {
        0: 0.7736,
        100: 0.7833,
        250: 0.7711,
        500: 0.7873,
        750: 0.7920,
        1000: 0.7865,
        1500: 0.7950,
        2000: 0.7939,
        3000: 0.7969,
        4000: 0.7986,
        4500: 0.7838,
        5140: 0.7555
    },
    "Weighted Means": {
        0: 0.7736,
        100: 0.7814,
        250: 0.7661,
        500: 0.7772,
        750: 0.7873,
        1000: 0.7917,
        1500: 0.7939,
        2000: 0.7936,
        3000: 0.7986,
        4000: 0.7975,
        4500: 0.7970,
        5140: 0.7947
    }
}

# Define grayscale colors, line styles, and markers for each experiment
colors = {
    "Separate Means": "#000000",   # black
    "Weighted Means": "#666666"    # dark gray
}
line_styles = ['--', '-']
markers = ['o', '^']

plt.figure(figsize=(10, 6))

for (label, data), ls, marker in zip(experiments.items(), line_styles, markers):
    radii = sorted(data.keys())
    r2_values = [data[r] for r in radii]
    plt.plot(radii, r2_values, linestyle=ls, marker=marker, markersize=6,
             color=colors[label], label=label)

plt.xlabel("Radius (m)")
plt.ylabel("Best R²")
plt.title("Sensitivity of R² to Spatial Scale: Surface Temperature Experiment")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
## FEATURE IMPORTANCE SEPARATE MEANS RADIUS-BASED APELDOORN SATELLITE

import numpy as np
import matplotlib.pyplot as plt

# Radii in meters (37 values)
radii = np.array([10, 60, 110, 160, 210, 260, 310, 360, 410, 460, 500, 750, 1000,
                  1250, 1500, 1750, 2000, 2250, 2500, 2750, 3000, 3500, 3850, 4200,
                  4550, 4900, 5250, 5600, 5950, 6300, 6650, 7000, 7350, 7700, 8050,
                  8400, 8750])

# Hardcoded feature importance values for Original Segments
original_features = {
    "Segment Predominant Land Use": [0.167080, 0.114028, 0.084344, 0.077455, 0.072590, 0.071316, 0.071010,
                   0.050891, 0.044566, 0.040705, 0.038386, 0.041975, 0.033599, 0.032282,
                   0.033777, 0.032263, 0.035317, 0.037422, 0.042375, 0.047660, 0.049125,
                   0.047266, 0.063787, 0.078538, 0.086754, 0.095393, 0.077937, 0.074456,
                   0.059052, 0.054984, 0.092900, 0.109008, 0.121084, 0.117315, 0.101619,
                   0.095790, 0.093080],
    "Segment Mean Building Elevation": [0.117747, 0.067530, 0.059295, 0.054743, 0.049502, 0.043393, 0.040894,
                   0.037772, 0.037241, 0.032975, 0.034068, 0.024860, 0.022856, 0.024778,
                   0.025461, 0.025497, 0.024601, 0.025506, 0.027019, 0.026730, 0.025659,
                   0.033268, 0.037402, 0.040364, 0.037908, 0.043789, 0.038586, 0.042199,
                   0.045664, 0.053306, 0.064394, 0.074952, 0.081324, 0.075413, 0.066030,
                   0.066906, 0.064174],
    "Segment Mean Population": [0.086424, 0.040376, 0.023675, 0.022294, 0.019832, 0.017917, 0.017423,
                   0.016551, 0.017724, 0.014560, 0.013917, 0.015278, 0.014028, 0.017827,
                   0.019663, 0.017232, 0.017231, 0.018335, 0.017789, 0.018858, 0.018732,
                   0.017500, 0.018015, 0.019395, 0.024799, 0.020641, 0.021708, 0.024050,
                   0.026765, 0.029295, 0.032717, 0.044004, 0.048155, 0.052688, 0.052326,
                   0.044807, 0.042625],
    "Segment Width": [0.121520, 0.055126, 0.033263, 0.024152, 0.020084, 0.016725, 0.015860,
              0.015003, 0.013943, 0.013833, 0.012939, 0.010943, 0.010614, 0.009579,
              0.010699, 0.009473, 0.010068, 0.009771, 0.011204, 0.010118, 0.011667,
              0.013370, 0.013960, 0.013461, 0.013848, 0.014144, 0.015551, 0.017577,
              0.019914, 0.027189, 0.040339, 0.050256, 0.063603, 0.064052, 0.063023,
              0.061996, 0.060721],
    "Segment Predominant Facade": [0.037212, 0.023613, 0.018401, 0.016652, 0.017608, 0.014705, 0.013540,
                    0.012752, 0.012693, 0.011202, 0.009585, 0.009481, 0.011924, 0.012765,
                    0.012931, 0.012121, 0.012739, 0.013557, 0.016535, 0.017109, 0.017643,
                    0.014953, 0.014519, 0.011991, 0.012535, 0.017887, 0.013697, 0.014095,
                    0.016030, 0.017545, 0.015483, 0.020108, 0.026907, 0.027694, 0.019355,
                    0.015859, 0.016635]
}

# Hardcoded feature importance values for Neighbours
neighbors_features = {
    "Neighbour Vehicle Fraction": [0.001890, 0.011544, 0.016985, 0.022807, 0.025085, 0.030800, 0.031700,
                   0.033312, 0.030946, 0.031353, 0.035656, 0.041718, 0.046112, 0.046999,
                   0.052306, 0.070342, 0.063072, 0.039724, 0.033514, 0.027676, 0.024143,
                   0.022306, 0.022869, 0.025051, 0.022493, 0.026072, 0.035472, 0.043395,
                   0.063086, 0.057903, 0.053644, 0.032926, 0.020196, 0.010745, 0.007378,
                   0.005920, 0.005616],
    "Neighbour Water density": [0.001379, 0.014130, 0.022206, 0.030392, 0.034605, 0.035827, 0.035844,
                   0.042730, 0.052162, 0.055347, 0.056689, 0.074035, 0.064596, 0.046565,
                   0.048546, 0.044715, 0.047639, 0.056869, 0.060667, 0.070193, 0.063074,
                   0.061782, 0.064226, 0.057170, 0.064003, 0.069710, 0.063591, 0.049149,
                   0.053179, 0.035319, 0.024666, 0.017360, 0.014122, 0.012991, 0.010824,
                   0.010385, 0.009928],
    "Neighbour Width": [0.005824, 0.038622, 0.045017, 0.046990, 0.049822, 0.045518, 0.050292,
              0.051931, 0.055248, 0.061325, 0.065206, 0.071284, 0.067132, 0.090747,
              0.116289, 0.104548, 0.100774, 0.070354, 0.057895, 0.026856, 0.024215,
              0.036355, 0.045752, 0.043223, 0.041190, 0.033182, 0.027847, 0.029468,
              0.031489, 0.033557, 0.044886, 0.050491, 0.054698, 0.057925, 0.059286,
              0.058944, 0.060802],
    "Neighbour Predominent Facade": [0.000581, 0.020784, 0.047332, 0.049814, 0.055572, 0.060912, 0.069543,
                    0.070736, 0.076995, 0.082805, 0.087302, 0.093904, 0.100409, 0.094503,
                    0.097583, 0.097200, 0.103629, 0.111174, 0.112212, 0.124067, 0.124145,
                    0.121294, 0.118681, 0.094136, 0.060901, 0.063644, 0.051777, 0.052405,
                    0.057613, 0.069741, 0.048948, 0.042257, 0.022487, 0.012859, 0.018522,
                    0.015328, 0.015378],
    "Neighbour Max Building Height": [0.001554, 0.032720, 0.056423, 0.059572, 0.068686, 0.069562, 0.066880,
                   0.066608, 0.067852, 0.070829, 0.067331, 0.073093, 0.069561, 0.072051,
                   0.061034, 0.049326, 0.049199, 0.068566, 0.067520, 0.060122, 0.056374,
                   0.028323, 0.033609, 0.041510, 0.048504, 0.043812, 0.031686, 0.026361,
                   0.032014, 0.036407, 0.035874, 0.031890, 0.027516, 0.028639, 0.026893,
                   0.025788, 0.031755],
    "Neighbour Mean Population": [0.006632, 0.046557, 0.070960, 0.071719, 0.067603, 0.068860, 0.065567,
                   0.062826, 0.059935, 0.057864, 0.060108, 0.062242, 0.080927, 0.110262,
                   0.055059, 0.052912, 0.062438, 0.063052, 0.068953, 0.098490, 0.110848,
                   0.103570, 0.095487, 0.079226, 0.072204, 0.088310, 0.120047, 0.131643,
                   0.120678, 0.102394, 0.089617, 0.054365, 0.044904, 0.042635, 0.040132,
                   0.040765, 0.040367]
}

# We'll define a small set of black & gray colors, plus a small set of line styles and markers.
colors = ['#000000', '#222222', '#444444', '#555555', '#666666']
line_styles = ['-', '--', '-.', ':', (0, (3, 1, 1, 1))]
markers = ['o', '^', 's', 'd', 'v', 'P']

plt.style.use('default')  # keep the default white background

###############################################################################
# Plot 1: Original Segments (mix of black and shades of gray)
###############################################################################
plt.figure(figsize=(10, 6))

feat_list_orig = list(original_features.items())  # easier iteration
for i, (feature, values) in enumerate(feat_list_orig):
    c = colors[i % len(colors)]
    ls = line_styles[i % len(line_styles)]
    mk = markers[i % len(markers)]
    plt.plot(radii, values, color=c, linestyle=ls, marker=mk, markersize=5,
             label=feature)

plt.xlabel("Radius (m)")
plt.ylabel("Feature Importance")
plt.title("Original Segments Feature Importances Sensitivity Separate Means")
plt.grid(True)
plt.legend()
plt.show()

###############################################################################
# Plot 2: Neighbours (again mix black/gray, line styles, markers)
###############################################################################
plt.figure(figsize=(10, 6))

feat_list_neigh = list(neighbors_features.items())
for i, (feature, values) in enumerate(feat_list_neigh):
    c = colors[i % len(colors)]
    ls = line_styles[i % len(line_styles)]
    mk = markers[i % len(markers)]
    plt.plot(radii, values, color=c, linestyle=ls, marker=mk, markersize=5,
             label=feature)

plt.xlabel("Radius (m)")
plt.ylabel("Feature Importance")
plt.title("Neighbourhood Feature Importances Sensitivity Separate Means")
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
## FEATURE IMPORTANCE WEIGHTED MEANS RADIUS-BASED APELDOORN SATELLITE

import matplotlib.pyplot as plt
import numpy as np

# Radii (37 values)
radii = [10, 60, 110, 160, 210, 260, 310, 360, 410, 460, 500, 750, 1000,
         1250, 1500, 1750, 2000, 2250, 2500, 2750, 3000, 3500, 3850, 4200,
         4550, 4900, 5250, 5600, 5950, 6300, 6650, 7000, 7350, 7700, 8050,
         8400, 8750]

# --------------------------
# ORIGINAL SEGMENTS FEATURES
# --------------------------
# original_Veg_densit (from block 1)
original_Veg_densit = [0.174160, 0.102120, 0.065818, 0.056556, 0.054545, 0.052544, 0.046701,
                         0.046344, 0.044863, 0.042110, 0.042120, 0.038089, 0.034416, 0.034268,
                         0.033706, 0.034524, 0.032975, 0.036274, 0.037301, 0.039580, 0.039317,
                         0.041902, 0.042868, 0.042023, 0.047654, 0.041982, 0.037688, 0.033487,
                         0.033037, 0.031941, 0.033662, 0.033548, 0.033146, 0.032827, 0.032661,
                         0.032063, 0.032196]

# original_Pred_Facad (from block 2)
original_Pred_Facad = [0.032790, 0.021689, 0.017252, 0.014478, 0.014107, 0.014601, 0.013226,
                         0.012618, 0.011470, 0.010092, 0.009529, 0.008944, 0.011532, 0.010920,
                         0.010487, 0.008512, 0.009913, 0.012886, 0.014370, 0.015541, 0.016173,
                         0.011660, 0.012845, 0.012062, 0.011931, 0.011349, 0.012116, 0.011530,
                         0.011377, 0.010163, 0.011217, 0.011936, 0.011035, 0.011397, 0.011374,
                         0.011682, 0.012817]

# original_Width (from block 2)
original_Width = [0.128506, 0.055654, 0.033240, 0.024941, 0.020141, 0.017368, 0.015943,
                  0.014363, 0.014446, 0.013712, 0.012285, 0.011532, 0.010826, 0.010111,
                  0.010469, 0.009220, 0.009326, 0.010563, 0.010825, 0.010902, 0.011305,
                  0.011471, 0.011303, 0.011293, 0.010383, 0.009829, 0.009438, 0.009151,
                  0.009387, 0.009198, 0.009161, 0.009217, 0.009435, 0.009239, 0.009118,
                  0.009161, 0.009312]

# original_Pred_LandU (from block 2)
original_Pred_LandU = [0.157703, 0.112801, 0.087914, 0.078209, 0.074912, 0.075112, 0.073836,
                        0.055584, 0.048105, 0.043116, 0.042965, 0.047387, 0.035533, 0.032806,
                        0.034528, 0.031849, 0.037004, 0.040935, 0.040213, 0.047355, 0.040591,
                        0.048526, 0.050735, 0.051295, 0.043887, 0.042324, 0.041706, 0.035678,
                        0.035808, 0.031506, 0.030800, 0.029198, 0.031186, 0.033384, 0.031217,
                        0.031962, 0.030483]

# original_Mean_Bld_E (from block 3)
original_Mean_Bld_E = [0.114090, 0.069992, 0.057980, 0.056860, 0.050979, 0.042981, 0.041511,
                         0.038066, 0.035038, 0.033385, 0.031626, 0.027276, 0.024544, 0.024286,
                         0.023608, 0.022488, 0.022781, 0.024650, 0.023429, 0.024263, 0.024117,
                         0.027970, 0.030217, 0.028175, 0.027943, 0.027347, 0.026112, 0.025417,
                         0.024153, 0.023769, 0.021959, 0.022254, 0.022136, 0.022958, 0.021580,
                         0.022198, 0.022701]

# --------------------------
# NEIGHBOURS FEATURES (weighted_avg_ columns)
# --------------------------
# weighted_avg_Veg_dens_neighbors
weighted_avg_Veg_dens_neighbors = [0.001291, 0.013698, 0.022525, 0.028728, 0.031969, 0.036102,
                                   0.037158, 0.039984, 0.047209, 0.054701, 0.056335, 0.068227,
                                   0.057274, 0.049516, 0.047555, 0.042677, 0.048869, 0.055052,
                                   0.061096, 0.066716, 0.062213, 0.057160, 0.064922, 0.063364,
                                   0.065595, 0.067210, 0.063944, 0.062247, 0.064323, 0.063279,
                                   0.064392, 0.068289, 0.065346, 0.068209, 0.068153, 0.066185,
                                   0.066601]

# weighted_avg_Pred_Facad_neighbors
weighted_avg_Pred_Facad_neighbors = [0.000649, 0.026834, 0.050846, 0.060307, 0.066586, 0.067970,
                                       0.075596, 0.075648, 0.080469, 0.089143, 0.088601, 0.098862,
                                       0.101390, 0.102771, 0.106599, 0.108048, 0.105825, 0.116265,
                                       0.115745, 0.118710, 0.132486, 0.122155, 0.121063, 0.120976,
                                       0.098212, 0.087280, 0.077168, 0.077435, 0.082247, 0.099053,
                                       0.101982, 0.101302, 0.100620, 0.101820, 0.102967, 0.099506,
                                       0.101556]

# weighted_avg_Max_Bld_He_neighbors
weighted_avg_Max_Bld_He_neighbors = [0.001641, 0.034272, 0.054409, 0.058697, 0.066014, 0.068120,
                                     0.068889, 0.068790, 0.068398, 0.072100, 0.071925, 0.079254,
                                     0.073559, 0.073557, 0.052219, 0.053217, 0.053748, 0.060415,
                                     0.066881, 0.055752, 0.048417, 0.042282, 0.052660, 0.088181,
                                     0.120380, 0.122345, 0.118402, 0.108342, 0.101699, 0.078071,
                                     0.064359, 0.057584, 0.055428, 0.054233, 0.062897, 0.062504,
                                     0.063820]

# weighted_avg_Width_neighbors
weighted_avg_Width_neighbors = [0.005832, 0.037982, 0.043286, 0.047607, 0.047659, 0.048017,
                                0.050967, 0.053917, 0.057647, 0.060045, 0.062867, 0.078103,
                                0.074908, 0.098982, 0.123600, 0.105373, 0.105413, 0.077577,
                                0.063760, 0.056974, 0.056793, 0.034177, 0.033607, 0.036137,
                                0.028571, 0.027461, 0.031922, 0.049191, 0.061245, 0.076430,
                                0.092812, 0.108118, 0.100176, 0.098565, 0.092659, 0.090303,
                                0.088107]

# weighted_avg_Mean_Popul_neighbors
weighted_avg_Mean_Popul_neighbors = [0.006436, 0.044845, 0.070284, 0.071273, 0.070794, 0.073347,
                                     0.067080, 0.062811, 0.062981, 0.062330, 0.061780, 0.061936,
                                     0.083241, 0.125459, 0.073985, 0.062705, 0.069859, 0.076191,
                                     0.089972, 0.111221, 0.117184, 0.114620, 0.103304, 0.098422,
                                     0.085508, 0.084411, 0.105957, 0.106283, 0.103859, 0.108124,
                                     0.093975, 0.094750, 0.096183, 0.094113, 0.094891, 0.097597,
                                     0.098058]

# weighted_avg_Water_dens_neighbors
weighted_avg_Water_dens_neighbors = [0.001291, 0.013698, 0.022525, 0.028728, 0.031969, 0.036102,
                                      0.037158, 0.039984, 0.047209, 0.054701, 0.056335, 0.068227,
                                      0.057274, 0.049516, 0.047555, 0.042677, 0.048869, 0.055052,
                                      0.061096, 0.066716, 0.062213, 0.057160, 0.064922, 0.063364,
                                      0.065595, 0.067210, 0.063944, 0.062247, 0.064323, 0.063279,
                                      0.064392, 0.068289, 0.065346, 0.068209, 0.068153, 0.066185,
                                      0.066601]

# --------------------------
# PLOTTING PARAMETERS (for both plots)
# --------------------------
# Define a set of colors (black plus various shades of gray), line styles, and markers.
colors_list = ['#000000', '#222222', '#444444', '#555555', '#666666']
line_styles = ['-', '--', '-.', ':', (0, (3, 1, 1, 1))]
markers = ['o', '^', 's', 'd', 'v', 'P']

# Use default style (white background, standard axes)
plt.style.use('default')

# --------------------------
# PLOT 1: ORIGINAL SEGMENTS
# --------------------------
plt.figure(figsize=(10, 6))
orig_features = {
    "Segemnt Vegetation Density": original_Veg_densit,
    "Segemnt Predominant Facade": original_Pred_Facad,
    "Segemnt Width": original_Width,
    "Segemnt Predominant Land Use": original_Pred_LandU,
    "Segemnt Mean Building Elevation": original_Mean_Bld_E
}

for i, (label, values) in enumerate(orig_features.items()):
    color = colors_list[i % len(colors_list)]
    ls = line_styles[i % len(line_styles)]
    mk = markers[i % len(markers)]
    plt.plot(radii, values, color=color, linestyle=ls, marker=mk, markersize=5, label=label)

plt.xlabel("Radius (m)")
plt.ylabel("Feature Importance")
plt.title("Original Segments Feature Importances Sensitivity Weighted Means")
plt.grid(True)
plt.legend()
plt.show()

# --------------------------
# PLOT 2: NEIGHBOURS
# --------------------------
plt.figure(figsize=(10, 6))
neigh_features = {
    "Neighbour Vegetation Density": weighted_avg_Veg_dens_neighbors,
    "Neighbour Predominant Facade": weighted_avg_Pred_Facad_neighbors,
    "Neighbour Max Building Height": weighted_avg_Max_Bld_He_neighbors,
    "Neighbour Width": weighted_avg_Width_neighbors,
    "Neighbour Mean Population": weighted_avg_Mean_Popul_neighbors,
    "Neighbour Water density": weighted_avg_Water_dens_neighbors
}

for i, (label, values) in enumerate(neigh_features.items()):
    color = colors_list[i % len(colors_list)]
    ls = line_styles[i % len(line_styles)]
    mk = markers[i % len(markers)]
    plt.plot(radii, values, color=color, linestyle=ls, marker=mk, markersize=5, label=label)

plt.xlabel("Radius (m)")
plt.ylabel("Feature Importance")
plt.title("Neighbourhood Feature Importances Sensitivity Weighted Means")
plt.grid(True)
plt.legend()
plt.show()
